## Eir v1.7.2 - Automated Bond Valence Sum Analysis

## Overview

Eir performs automated Bond Valence Sum (BVS) analysis on multiple crystal structures from:
- **ICSD CIF files** (downloaded from ICSD database)
- **GSAS-II refinement outputs** (.lst files)

The code uses proper crystallographic symmetry operations via pymatgen to correctly handle all space groups, particularly high-symmetry cases (R-3m, P63/mmc) common in layered oxides.

### 1. Config File Support
- Load oxidation states from `config.json` or Python dictionary
- Enables "Run All" batch processing without manual input
- Automatic fallback to interactive mode if no config provided

### 2. 3D Visualisation
- Automatic visualisation of problematic sites (BVS deviation > 30%)
- Uses py3Dmol for interactive bond visualisation in Jupyter
- Helps diagnose coordination issues (missing bonds, wrong oxidation state)

### 3. Smart Gap Detection
- Adaptive thresholding for Jahn-Teller distorted sites (Mn³⁺, Cu²⁺)
- Considers both absolute gap size and relative gap ratio
- More robust for highly distorted coordination environments

## Key Features

- Automatic coordination number (CN) detection using adaptive gap analysis
- Mixed/partial occupancy support (e.g., Cr₀.₈Ti₀.₂ on same site)
- Bond Valence Sum (BVS) calculation using Gagné & Hawthorne (2015) parameters
- Global Instability Index (GII) calculation for structural stability assessment
- Batch processing of multiple structures with summary CSV output
- Interactive 3D visualisation of problematic sites

## Input Files

### Option 1: ICSD CIF Files
- Place CIF files in a folder
- Filenames should contain CollCode (e.g., `YourCustomFileName_CollCode116507.cif`)
- Script will extract CollCode from filename

### Option 2: GSAS-II Files
- Requires both `.lst` file (refinement output) and corresponding `.cif` file
- Place in same directory with matching base names

## How to Use

### Quick Start (with config file)
```python
# Create config.json in your CIF directory:
{
  "Na": 1,
  "Ni": 2,
  "Mn": 4,
  "O": -2
}

# Then run all cells - walks away for coffee
```

### Interactive Mode (no config)
```python
# Just run all cells
# Script will prompt you for oxidation states
# Useful when analyzing unfamiliar materials
```

### With 3D Visualisation
```python
# Problematic sites (|deviation| > 30%) are automatically visualised
# Look for:
#   - Missing bonds (coordination too low)
#   - Wrong oxidation state assignment
#   - Jahn-Teller distortion
```

## Output Columns

- **File**: Input filename
- **CollCode/Phase**: ICSD CollCode or phase name
- **Space_Group**: Crystallographic space group
- **Element**: Chemical element analyzed
- **Site_Label**: Crystallographic site label
- **CN**: Coordination number (from adaptive gap analysis)
- **BVS**: Bond Valence Sum
- **Formal_Charge**: Expected oxidation state
- **Deviation**: BVS - Formal_Charge
- **GII**: Global Instability Index (whole structure)

## Interpreting Results

### Coordination Numbers (CN)
- **CN = 6**: Octahedral coordination (expected for most transition metals in layered oxides)
- **CN = 4**: Tetrahedral coordination
- **CN = 1-3**: Likely indicates structural problems or missing bonds

### Bond Valence Sum (BVS)
- Should approximately equal the formal oxidation state
- **Deviation < ±0.2 v.u.**: Likely well-bonded
- **Deviation > ±0.3 v.u.**: Over-bonded (negative) or under-bonded (positive)
- **Deviation > ±0.5 v.u.**: Severe problem - check 3D visualisation

### Global Instability Index (GII) - Traditional interpretation
- **GII < 0.20 v.u.**: Stable, well-ordered structure
- **0.20 < GII < 0.40 v.u.**: Moderate strain, possibly acceptable
- **GII > 0.40 v.u.**: Significant structural instability
- **GII > 0.60 v.u.**: Severe problems - synthesis likely difficult

**Note:** The traditional 0.2 v.u. threshold is system-dependent (see Miller & Rondinelli, APL Mater. 2023)

## Common Issues

### Problem: All structures show CN = 1-2 instead of 6
- **Solution**: Already fixed in v1.0 using proper pymatgen symmetry operations

### Problem: "No bond valence parameters found"
- **Cause**: Element/oxidation state combination not in Gagné & Hawthorne database
- **Solution**: Check oxidation state assignment or add parameters manually

### Problem: Very high GII (>1.0) for literature structures
- **Possible causes:**
  - Mixed-phase sample (XRD refinement artefact)
  - Metastable structure
  - Genuine severe strain (synthesis should be difficult)
  
### Problem: High deviation but coordination looks correct
- **Solution**: Check 3D visualisation - may be Jahn-Teller distortion or wrong oxidation state

## Technical Details

### Adaptive Gap Detection (New in v1.1)
- Considers both absolute gap size (default 0.05 Å minimum)
- Also considers relative gap ratio (gap/bond length)
- For Jahn-Teller ions (Mn³⁺, Cu²⁺), uses more lenient thresholds
- Prevents over-splitting of distorted but chemically valid coordination spheres

### 3D Visualisation (New in v1.1)
- Automatically triggers for sites with |deviation| > 30%
- Shows central atom, coordinating atoms, and bonds
- Bond colours indicate bond valence contribution
- Interactive rotation and zooming in Jupyter

## Citation

If using this code, please cite:
- **BVS parameters:** Gagné, O.C. & Hawthorne, F.C. (2015) *Acta Cryst. B* **71**, 562-578
- **Structural validation:** Brown, I.D. (2002) *The Chemical Bond in Inorganic Chemistry: The Bond Valence Model*

## Version History
- **v1.7.2 (May 2026)**: Anti-site disorder fix (minority species no longer EXPUNGE'd when element is majority elsewhere); dataset-level multi-entry source flagging
- **v1.7.0 (May 2026)**: Data integrity tier system (ACCEPT/FLAG/EXPUNGE) for BVS reliability classification
- **v1.6.1 (Apr 2026)**: Fractional valence R0 interpolation for mixed-valence cathodes
- **v1.1 (05 Dec 2025)**: Config file support, 3D visualisation, adaptive gap detection, consistent naming
- **v1.0 (03 Dec 2025)**: Complete rewrite with pymatgen for proper crystallographic symmetry
- **v0.4**: Attempted supercell approach
- **v0.3**: ICSD CIF compatibility (failed - low CN)
- **v0.2**: Mixed occupancy support
- **v0.1**: Original GSAS-II only version

In [ ]:
import sys
print(f"Current Python: {sys.executable}")
print(f"Installing py3Dmol to this environment...\n")

!{sys.executable} -m pip install py3Dmol

print(f"\n✓ Installation complete - restart kernel to use py3Dmol")

# Once run, go to Kernel → Restart Kernel (or Restart & Clear Output)
# Then change this cell from Code to Markdown to prevent wasting time reinstalling the library every time.

In [ ]:
# CELL 3: HELPER FUNCTIONS (EIR v1.7)
# ============================================================================
# Run this cell BEFORE Cell 4 (the main Eir code)
#
# VERSION HISTORY:
# - v1.7 (May 2026): Data integrity tier system (ACCEPT/FLAG/EXPUNGE),
#                     minority species detection (I3), implausible BVS detection (I6),
#                     mixed TM occupancy flagging, CIF valence metadata columns,
#                     structure-level tier summary, version string fix (was "v1.5")
# - v1.6.1 (Apr 2026): Fractional valence interpolation for mixed-valence cathodes
# - v1.5 (Mar 2026): Type 6 flagging fix, GII per-label iteration, Type 5 occupancy fix,
#                     formula element parsing fix, version string consolidation
# - v1.4.2 (Feb 2026): Per-atom valence iteration, valence_source tracking
# - v1.4 (29 Jan 2026): Oxidation state extraction from CIF, Type 7 error flags,
#                       anion parameter (u) extraction, enhanced structural parameters
# - v1.2: Temperature and pressure extraction from CIF files
# - v1.1: Config file support, 3D visualisation, adaptive gap detection

import os
import json
import re
import math

# ============================================================================
# ANION DETECTION
# ============================================================================
# Known anions that can form ionic bonds with metals
KNOWN_ANIONS = {
    # Chalcogens (Group 16)
    'O': -2, 'S': -2, 'Se': -2, 'Te': -2,
    # Halogens (Group 17)
    'F': -1, 'Cl': -1, 'Br': -1, 'I': -1,
    # Other common anions
    'N': -3, 'P': -3, 'As': -3, 'H': -1
}

def is_anion(element):
    """Check if an element is a known anion."""
    return element in KNOWN_ANIONS

def get_anion_charge(element):
    """Get typical charge for an anion."""
    return KNOWN_ANIONS.get(element, 0)

# ============================================================================
# OXIDATION STATE HANDLING (NEW IN v1.4)
# ============================================================================

# Elements that commonly exhibit mixed valence in spinels
VARIABLE_VALENCE_ELEMENTS = {
    'Fe', 'Mn', 'Co', 'Cu', 'Cr', 'V', 'Ni', 'Ti', 'Ce', 'Eu', 'Ru', 'Rh', 'Ir'
}

# ============================================================================
# DATA INTEGRITY TIER SYSTEM (NEW IN v1.7)
# ============================================================================
# Classifies each site (and hence each structure) into one of three tiers
# based on the reliability of the oxidation state assignment and BVS result.
#
# ACCEPT: Valence assignment is unambiguous (integer formal states, or
#         single-element sites with charge-balance-determined non-integer states).
# FLAG:   Two or more TM elements share a crystallographic site AND at least one
#         has a non-integer assigned valence. The valence partitioning is an
#         assumption made by the depositor and cannot be verified from diffraction.
# EXPUNGE: BVS result is demonstrably meaningless (implausible coordinates or
#          minority species on a site dominated by a different cation).
#
# See: Eir 1.7 Specification (NRM, May 2026)
# ============================================================================

TRANSITION_METALS = {
    'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
    'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd',
    'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg',
    'La', 'Ce', 'Pr', 'Nd', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu',
}

# Thresholds for EXPUNGE tier
EXPUNGE_BVS_HIGH = 10.0       # BVS > 10 v.u. is physically impossible
EXPUNGE_BVS_LOW = 0.2         # BVS < 0.2 v.u. for expected valence > 1
MINORITY_OCC_THRESHOLD = 0.05  # Occupancy below which a species is "minority"
MAJORITY_OCC_THRESHOLD = 0.90  # Occupancy above which another species dominates


def is_transition_metal(element):
    """Check if an element is a transition metal (d-block or lanthanide)."""
    return element in TRANSITION_METALS


def is_valence_integer(valence, tolerance=0.05):
    """
    Check if a valence is effectively integer.
    
    Tolerance of 0.05 matches the symbol_number_agreement threshold,
    so 3.99 (clearly Mn4+ rounded) is treated as integer while 3.313
    (genuine charge-balance average) is not.
    """
    if valence is None:
        return True  # No valence = no non-integer flag
    return abs(valence - round(valence)) < tolerance


def assign_site_integrity_tier(element, bvs_value, expected_valence, occupancy,
                                is_mixed_tm_site, has_noninteger_tm_on_site,
                                is_minority_on_majority_site):
    """
    Assign an integrity tier to a single site.
    
    Priority order (highest to lowest severity):
        1. EXPUNGE (I6): implausible BVS
        2. EXPUNGE (I3): minority species on majority-occupied site
        3. FLAG: mixed TM occupancy with non-integer valence partitioning
        4. ACCEPT: everything else
    
    Args:
        element: Element symbol for this site entry
        bvs_value: Calculated BVS (float or None)
        expected_valence: Expected valence from CIF (float or None)
        occupancy: Site occupancy (float, 0-1)
        is_mixed_tm_site: True if this TM shares a Wyckoff position with another TM
        has_noninteger_tm_on_site: True if any TM on this shared site has non-integer valence
        is_minority_on_majority_site: True if occupancy < 0.05 and another element
                                      on the same position has occupancy > 0.90
    
    Returns:
        tuple: (tier, reason)
            tier: 'ACCEPT', 'FLAG', or 'EXPUNGE'
            reason: str or None (reason for FLAG/EXPUNGE)
    """
    # --- EXPUNGE checks (highest priority) ---
    
    # I6: Implausible BVS
    if bvs_value is not None:
        if bvs_value > EXPUNGE_BVS_HIGH:
            return 'EXPUNGE', 'implausible_bvs'
        # Low BVS check: catches broken coordinates.
        # For valence >= 2: BVS < 0.2 is impossible for any reasonable structure.
        # For valence == 1 (alkali metals): BVS < 0.2 means <20% of expected,
        # which is also broken (legitimate Na+ in a large cavity is still ~0.7+).
        if expected_valence is not None and abs(expected_valence) >= 1 and bvs_value < EXPUNGE_BVS_LOW:
            return 'EXPUNGE', 'implausible_bvs'
    
    # I3: Minority species
    if is_minority_on_majority_site:
        return 'EXPUNGE', 'minority_species'
    
    # --- FLAG check ---
    if is_mixed_tm_site and has_noninteger_tm_on_site:
        return 'FLAG', 'mixed_occupancy_noninteger_valence'
    
    # --- Default ---
    return 'ACCEPT', None


def assign_structure_tier(site_tiers):
    """
    Assign an overall tier to a structure based on its worst site tier.
    
    Args:
        site_tiers: list of tier strings ('ACCEPT', 'FLAG', 'EXPUNGE')
    
    Returns:
        str: worst tier across all sites
    """
    tier_severity = {'ACCEPT': 0, 'FLAG': 1, 'EXPUNGE': 2}
    if not site_tiers:
        return 'ACCEPT'
    worst = max(site_tiers, key=lambda t: tier_severity.get(t, 0))
    return worst


def assess_flag_severity(bvs_value, expected_valence, validation_threshold=10.0):
    """
    For FLAG-tier sites, assess whether the depositor's valence partitioning
    produces a BVS that is consistent with the expected valence.
    
    This is the pragmatic position: we can't verify the partitioning from
    diffraction, but we can test whether it produces a physically reasonable
    BVS. A FLAG site where BVS validates is "unverifiable but plausible";
    one where BVS fails is "unverifiable and suspicious".
    
    Args:
        bvs_value: Calculated BVS (float or None)
        expected_valence: Expected valence from CIF (float or None)
        validation_threshold: Percentage deviation threshold (default 10%)
    
    Returns:
        str: 'consistent', 'inconsistent', or 'indeterminate'
    """
    if bvs_value is None or expected_valence is None or expected_valence == 0:
        return 'indeterminate'
    
    pct_dev = abs(bvs_value - abs(expected_valence)) / abs(expected_valence) * 100
    
    if pct_dev <= validation_threshold:
        return 'consistent'
    else:
        return 'inconsistent'


# Plausible integer oxidation states for TM elements in oxides
# Used by the "what if" analysis for FLAG-tier sites
PLAUSIBLE_OXIDATION_STATES = {
    'Cr': [3],
    'Mn': [2, 3, 4],
    'Fe': [2, 3],
    'Co': [2, 3, 4],
    'Ni': [2, 3],
    'Ti': [3, 4],
    'V':  [3, 4, 5],
    'Cu': [1, 2],
    'Mg': [2],
    'Al': [3],
    'Zn': [2],
    'Sc': [3],
    'Zr': [4],
    'Nb': [3, 5],
    'Mo': [4, 6],
    'Ru': [3, 4],
    'Rh': [3, 4],
    'Ir': [3, 4],
    'Ce': [3, 4],
    'Eu': [2, 3],
}


def whatif_analysis(element, bvs_value, expected_valence, anion='O'):
    """
    For FLAG-tier sites, test alternative integer oxidation states and report
    which gives the best BVS match.
    
    This implements the pragmatic position: even if we can't verify the
    depositor's valence partitioning, we can ask "does any plausible integer
    state produce a reasonable BVS?" If yes, the structure is probably fine
    and the CIF metadata just needs correcting. If no, the problem may be
    in the coordinates, not just the valence assignment.
    
    Args:
        element: Element symbol
        bvs_value: Calculated BVS (float)
        expected_valence: CIF-assigned expected valence (float)
        anion: Anion element (default 'O')
    
    Returns:
        dict with:
            best_integer_valence: int or None
            best_integer_deviation: float or None (percentage)
            candidates_tested: str (e.g. "Mn: 2(+108%), 3(+5%), 4(-28%)")
            depositor_deviation: float or None (percentage for CIF-assigned valence)
    """
    result = {
        'best_integer_valence': None,
        'best_integer_deviation': None,
        'candidates_tested': None,
        'depositor_deviation': None,
    }
    
    if bvs_value is None or element is None:
        return result
    
    # Depositor's deviation
    if expected_valence is not None and expected_valence != 0:
        result['depositor_deviation'] = round(
            (bvs_value - abs(expected_valence)) / abs(expected_valence) * 100, 1
        )
    
    candidates = PLAUSIBLE_OXIDATION_STATES.get(element, [])
    if not candidates:
        return result
    
    tested = []
    best_val = None
    best_dev = None
    
    for candidate_v in candidates:
        if candidate_v == 0:
            continue
        dev_pct = (bvs_value - candidate_v) / candidate_v * 100
        tested.append(f"{candidate_v}({dev_pct:+.0f}%)")
        
        abs_dev = abs(dev_pct)
        if best_dev is None or abs_dev < best_dev:
            best_dev = abs_dev
            best_val = candidate_v
    
    result['best_integer_valence'] = best_val
    result['best_integer_deviation'] = round(best_dev, 1) if best_dev is not None else None
    result['candidates_tested'] = f"{element}: {', '.join(tested)}" if tested else None
    
    return result


def check_minority_species(site_label, per_atom_valences, site_occupancies_data):
    """
    Check if a site label represents a minority species on a majority-occupied site.
    
    A minority species is defined as: occupancy < 0.05 AND another element on the
    same Wyckoff position has occupancy > 0.90.
    
    This requires matching sites by fractional coordinates (since Wyckoff symbols
    are not always reliably present in ICSD CIFs).
    
    Args:
        site_label: The site label to check (e.g. 'Ti1')
        per_atom_valences: dict from extract_per_atom_valences()
        site_occupancies_data: dict from extract_site_occupancies()
    
    Returns:
        bool: True if this is a minority species on a majority-occupied site
    """
    if site_label not in per_atom_valences:
        return False
    
    this_occ = per_atom_valences[site_label].get('occupancy', 1.0)
    if this_occ >= MINORITY_OCC_THRESHOLD:
        return False
    
    # This site has low occupancy. Check if another element at the same
    # fractional coordinates has high occupancy.
    this_site_data = site_occupancies_data.get(site_label, {})
    this_frac = (this_site_data.get('fract_x'), this_site_data.get('fract_y'), this_site_data.get('fract_z'))
    
    if any(v is None for v in this_frac):
        return False
    
    for other_label, other_data in site_occupancies_data.items():
        if other_label == site_label:
            continue
        other_frac = (other_data.get('fract_x'), other_data.get('fract_y'), other_data.get('fract_z'))
        if any(v is None for v in other_frac):
            continue
        
        # Match by fractional coordinates (within tolerance for rounding)
        if all(abs(a - b) < 0.001 for a, b in zip(this_frac, other_frac)):
            other_occ = other_data.get('occupancy', 1.0)
            if other_occ > MAJORITY_OCC_THRESHOLD:
                # Anti-site check: if this minority element is a majority
                # species on a DIFFERENT crystallographic site, this is
                # anti-site exchange (e.g. Na/Ti swapping between layers),
                # not a trace dopant. Anti-site disorder produces large but
                # physically meaningful BVS deviations — do not EXPUNGE.
                this_element = per_atom_valences.get(site_label, {}).get('element')
                is_antisite = False
                if this_element:
                    for check_label, check_pav in per_atom_valences.items():
                        if check_label == site_label:
                            continue
                        if check_pav.get('element') == this_element:
                            if check_pav.get('occupancy', 0) > MAJORITY_OCC_THRESHOLD:
                                is_antisite = True
                                break
                if not is_antisite:
                    return True
                # else: anti-site exchange — fall through to return False
    
    return False


def check_mixed_tm_site(site_label, per_atom_valences, site_occupancies_data):
    """
    Check if a site label is part of a mixed TM occupancy site, and whether
    any TM on that site has a non-integer valence.
    
    Sites are matched by fractional coordinates.
    
    Args:
        site_label: The site label to check
        per_atom_valences: dict from extract_per_atom_valences()
        site_occupancies_data: dict from extract_site_occupancies()
    
    Returns:
        tuple: (is_mixed_tm, has_noninteger_tm)
    """
    if site_label not in per_atom_valences:
        return False, False
    
    this_elem = per_atom_valences[site_label].get('element')
    if not is_transition_metal(this_elem):
        # Not a TM site -- cannot be FLAG
        return False, False
    
    this_site_data = site_occupancies_data.get(site_label, {})
    this_frac = (this_site_data.get('fract_x'), this_site_data.get('fract_y'), this_site_data.get('fract_z'))
    
    if any(v is None for v in this_frac):
        return False, False
    
    # Find all other elements at the same position
    tm_labels_at_position = [site_label]
    
    for other_label, other_data in site_occupancies_data.items():
        if other_label == site_label:
            continue
        other_frac = (other_data.get('fract_x'), other_data.get('fract_y'), other_data.get('fract_z'))
        if any(v is None for v in other_frac):
            continue
        
        if all(abs(a - b) < 0.001 for a, b in zip(this_frac, other_frac)):
            other_elem = other_data.get('element')
            if is_transition_metal(other_elem):
                tm_labels_at_position.append(other_label)
    
    is_mixed_tm = len(tm_labels_at_position) > 1
    
    if not is_mixed_tm:
        return False, False
    
    # Check if any TM on this shared position has non-integer valence
    has_noninteger = False
    for label in tm_labels_at_position:
        if label in per_atom_valences:
            val = per_atom_valences[label].get('explicit_valence')
            if val is not None and not is_valence_integer(val):
                has_noninteger = True
                break
    
    return True, has_noninteger

def flag_same_source_entries(df):
    """
    Dataset-level pass: identify multiple ICSD entries deposited from the
    same source paper.
    
    When a single paper produces multiple ICSD entries (e.g. different
    refinement methods, different samples, phase-separated components),
    the database stores them with equal apparent authority. This function
    groups entries by source paper and flags groups with >1 structure,
    so downstream analysis can identify redundant or subsidiary depositions.
    
    Adds columns:
        source_group: identifier string (first_author|year|journal)
        source_group_size: number of ICSD entries from that paper
        source_group_codes: comma-separated list of ICSD codes in the group
        flag_multi_entry_source: True if >1 entry from the same paper
    
    Args:
        df: pandas DataFrame from Eir batch processing (site-level rows)
    
    Returns:
        df: DataFrame with new columns added
    """
    # Build structure-level lookup: one row per ICSD code
    struct_cols = ['icsd_code', 'authors', 'year', 'journal']
    available = [c for c in struct_cols if c in df.columns]
    if 'icsd_code' not in available or 'authors' not in available:
        # Not enough metadata to group — add empty columns and return
        df['source_group'] = None
        df['source_group_size'] = 1
        df['source_group_codes'] = df['icsd_code'] if 'icsd_code' in df.columns else None
        df['flag_multi_entry_source'] = False
        return df
    
    other_cols = [c for c in available if c != 'icsd_code']
    struct_df = df.groupby('icsd_code')[other_cols].first().reset_index()
    
    # Build source group key from first author surname + year + journal
    def make_source_key(row):
        authors = row.get('authors', '') or ''
        year = row.get('year', '') or ''
        journal = row.get('journal', '') or ''
        # Extract first author surname (before first comma or space)
        first_author = authors.split(',')[0].split(' ')[0].strip().lower()
        # Normalise journal (lowercase, strip whitespace)
        journal_norm = journal.strip().lower()[:40]  # truncate for matching
        return f"{first_author}|{year}|{journal_norm}"
    
    struct_df['source_group'] = struct_df.apply(make_source_key, axis=1)
    
    # Count entries per source group
    group_counts = struct_df.groupby('source_group')['icsd_code'].agg(['count', list]).reset_index()
    group_counts.columns = ['source_group', 'source_group_size', 'group_codes_list']
    group_counts['source_group_codes'] = group_counts['group_codes_list'].apply(
        lambda x: ','.join(str(c) for c in sorted(x))
    )
    group_counts['flag_multi_entry_source'] = group_counts['source_group_size'] > 1
    group_counts = group_counts.drop(columns=['group_codes_list'])
    
    # Merge back to structure level, then to site level
    struct_df = struct_df.merge(group_counts, on='source_group', how='left')
    
    # Map to original df via icsd_code
    merge_cols = ['icsd_code', 'source_group', 'source_group_size', 
                  'source_group_codes', 'flag_multi_entry_source']
    df = df.merge(struct_df[merge_cols], on='icsd_code', how='left')
    
    return df

def extract_cif_oxidation_states(cif_path):
    """
    Extract oxidation states from CIF file's _atom_type_oxidation_number loop.

    ICSD CIFs typically contain this information in a loop like:
    loop_
    _atom_type_symbol
    _atom_type_oxidation_number
    Mg2+ 2
    Al3+ 3
    O2- -2

    For mixed-valence compounds, ICSD may give a non-integer value, e.g.:
    Mn4+ 3.5     (LiMn2O4 — symbol rounded up, true value in second column)

    v1.6 change: non-integer values are preserved as floats rather than truncated.
    A third return value (type_symbol -> oxidation_number map) is added so that
    extract_per_atom_valences can override symbol-parsed integers with the true
    fractional value.

    Returns:
        tuple: (oxidation_states_dict, quality, error_message, type_map)
        - oxidation_states_dict: {element: charge} e.g., {'Li': 1, 'Mn': 3.5, 'O': -2}
        - quality: 'explicit' | 'inferred' | 'missing' | 'ambiguous'
        - error_message: None or description of issue
        - type_map: {type_symbol: oxidation_number} e.g., {'Mn4+': 3.5}
    """
    oxidation_states = {}
    type_map = {}
    quality = 'missing'
    error_message = None

    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
    except Exception as e:
        return {}, 'missing', f"Could not read CIF: {e}", {}

    in_atom_type_loop = False
    columns = []
    symbol_col = None
    oxidation_col = None

    for line in lines:
        line_stripped = line.strip()

        if line_stripped == 'loop_':
            in_atom_type_loop = False
            columns = []
            symbol_col = None
            oxidation_col = None
            continue

        if line_stripped.startswith('_atom_type_'):
            columns.append(line_stripped)
            col_idx = len(columns) - 1
            if line_stripped == '_atom_type_symbol':
                symbol_col = col_idx
            elif line_stripped == '_atom_type_oxidation_number':
                oxidation_col = col_idx
                in_atom_type_loop = True
            continue

        if in_atom_type_loop and columns and not line_stripped.startswith('_'):
            if not line_stripped or line_stripped.startswith('#') or line_stripped.startswith('loop_'):
                in_atom_type_loop = False
                continue

            parts = line_stripped.split()

            if symbol_col is not None and symbol_col < len(parts):
                symbol_raw = parts[symbol_col]
                element = re.sub(r'[0-9+-]+', '', symbol_raw)
                element = element.capitalize() if len(element) > 1 else element.upper()

                if oxidation_col is not None and oxidation_col < len(parts):
                    ox_str = parts[oxidation_col]
                    if ox_str not in ['?', '.']:
                        try:
                            # v1.6: preserve fractional values (e.g. 3.5 for Mn in LiMn2O4)
                            ox_float = float(ox_str)
                            if ox_float == int(ox_float):
                                oxidation = int(ox_float)
                            else:
                                oxidation = ox_float

                            type_map[symbol_raw] = oxidation

                            if element in oxidation_states:
                                if oxidation_states[element] != oxidation:
                                    quality = 'ambiguous'
                                    error_message = f"Multiple oxidation states for {element}"
                            else:
                                oxidation_states[element] = oxidation
                                if quality != 'ambiguous':
                                    quality = 'explicit'
                        except ValueError:
                            pass

    # Fallback: infer from atom_type_symbol pattern if loop values missing
    if not oxidation_states:
        for line in lines:
            line_stripped = line.strip()
            matches = re.findall(r'([A-Z][a-z]?)(\d+)([+-])', line_stripped)
            for elem, charge, sign in matches:
                charge_val = int(charge) * (1 if sign == '+' else -1)
                if elem not in oxidation_states:
                    oxidation_states[elem] = charge_val
                    quality = 'inferred'

    if not oxidation_states:
        quality = 'missing'
        error_message = "No oxidation state information found in CIF"

    return oxidation_states, quality, error_message, type_map


def parse_chemical_formula(formula):
    """
    Parse chemical formula string into element counts.
    
    Handles formats like:
    - "Mg1 Al2 O4" (space-separated)
    - "MgAl2O4" (no spaces)
    - "Fe3O4"
    - "Li0.5Mn1.5O4"
    
    Returns:
        dict: {element: count} e.g., {'Mg': 1.0, 'Al': 2.0, 'O': 4.0}
    """
    if not formula:
        return {}
    
    # Clean up formula
    formula = formula.strip().replace("'", "").replace('"', '')
    
    # Pattern: Element followed by optional number (including decimals)
    pattern = r'([A-Z][a-z]?)(\d*\.?\d*)'
    
    elements = {}
    for match in re.finditer(pattern, formula):
        element = match.group(1)
        count_str = match.group(2)
        
        if element:  # Skip empty matches
            count = float(count_str) if count_str else 1.0
            elements[element] = elements.get(element, 0) + count
    
    return elements


def detect_mixed_valence_requirement(formula, oxidation_states):
    """
    Detect if a formula requires mixed valence to achieve charge balance.
    
    For example:
    - Fe3O4 with Fe3+ and O2-: 3(+3) + 4(-2) = +1 != 0 -> needs Fe2+/Fe3+ mix
    - MgAl2O4 with Mg2+, Al3+, O2-: +2 + 2(+3) + 4(-2) = 0 -> OK
    
    Returns:
        dict with:
        - requires_mixed_valence: bool
        - charge_balance_error: float (should be 0 for valid structure)
        - problematic_elements: list of elements that might need mixed valence
        - suggested_average_charges: dict of {element: required_average_charge}
    """
    result = {
        'requires_mixed_valence': False,
        'charge_balance_error': 0.0,
        'problematic_elements': [],
        'suggested_average_charges': {}
    }
    
    elements = parse_chemical_formula(formula)
    if not elements or not oxidation_states:
        return result
    
    # Separate cations and anions
    anion_elements = {e for e in elements if e in KNOWN_ANIONS}
    cation_elements = {e for e in elements if e not in KNOWN_ANIONS}
    
    # Calculate total anion charge
    anion_charge = sum(
        elements.get(e, 0) * KNOWN_ANIONS.get(e, 0)
        for e in anion_elements
    )
    
    # Calculate total cation charge based on provided oxidation states
    cation_charge = 0
    for elem in cation_elements:
        if elem in oxidation_states:
            cation_charge += elements.get(elem, 0) * oxidation_states[elem]
    
    # Check charge balance
    total_charge = cation_charge + anion_charge
    result['charge_balance_error'] = total_charge
    
    if abs(total_charge) > 0.01:  # Not balanced
        result['requires_mixed_valence'] = True
        
        # Identify which elements might need mixed valence
        for elem in cation_elements:
            if elem in VARIABLE_VALENCE_ELEMENTS:
                result['problematic_elements'].append(elem)
                
                # Calculate what average charge would be needed
                # If we remove this element's contribution and solve for it
                other_cation_charge = sum(
                    elements.get(e, 0) * oxidation_states.get(e, 0)
                    for e in cation_elements if e != elem
                )
                
                # Required charge from this element to balance
                required_charge = -anion_charge - other_cation_charge
                elem_count = elements.get(elem, 1)
                if elem_count > 0:
                    required_avg = required_charge / elem_count
                    result['suggested_average_charges'][elem] = round(required_avg, 2)
    
    return result


def flag_oxidation_state_issues(cif_path, formula):
    """
    Comprehensive check for oxidation state issues (Type 7 errors).
    
    Returns:
        dict with:
        - has_type7_error: bool
        - error_code: str or None
        - error_message: str or None
        - oxidation_states: dict from CIF
        - quality: str
        - mixed_valence_info: dict
    """
    result = {
        'has_type7_error': False,
        'error_code': None,
        'error_message': None,
        'oxidation_states': {},
        'quality': 'unknown',
        'mixed_valence_info': {}
    }
    
    # Extract oxidation states from CIF
    ox_states, quality, error_msg, _ = extract_cif_oxidation_states(cif_path)
    result['oxidation_states'] = ox_states
    result['quality'] = quality
    
    # Check for missing oxidation states
    if quality == 'missing':
        result['has_type7_error'] = True
        result['error_code'] = 'TYPE7_NO_OXIDATION_STATES'
        result['error_message'] = error_msg or "CIF lacks oxidation state information"
        return result
    
    # Check for ambiguous (multiple) oxidation states
    if quality == 'ambiguous':
        result['has_type7_error'] = True
        result['error_code'] = 'TYPE7_EXPLICIT_MIXED_VALENCE'
        result['error_message'] = error_msg or "CIF specifies multiple oxidation states for same element"
        return result
    
    # Check if formula requires mixed valence for charge balance
    mixed_valence_info = detect_mixed_valence_requirement(formula, ox_states)
    result['mixed_valence_info'] = mixed_valence_info
    
    if mixed_valence_info['requires_mixed_valence']:
        result['has_type7_error'] = True
        
        if abs(mixed_valence_info['charge_balance_error']) > 0.5:
            result['error_code'] = 'TYPE7_CHARGE_IMBALANCE'
            result['error_message'] = (
                f"Charge imbalance of {mixed_valence_info['charge_balance_error']:+.1f} "
                f"cannot be resolved with given oxidation states"
            )
        else:
            result['error_code'] = 'TYPE7_REQUIRES_MIXED_VALENCE'
            problematic = ', '.join(mixed_valence_info['problematic_elements'])
            result['error_message'] = (
                f"Charge balance requires mixed valence for: {problematic}"
            )
    
    return result

def flag_all_error_types(cif_path, formula, site_data, spinel_results=None):
    """
    Flag all detectable error types (2-7) for a structure.
    Based on taxonomy from Sherwood (2026) Dalton Trans.
    
    Types:
        2: Missing BVS parameters
        3: Vacancy disorder / nonstoichiometric
        4: Heterovalent disorder ΔV ≥ 2
        5: Homovalent / site disorder
        6A: Pre-modern techniques (pre-1980)
        7: Mixed valence (handled separately)
    
    Returns:
        dict with flag_type2 through flag_type6 and error descriptions
    """
    flags = {
        'flag_type2': False,
        'flag_type3': False,
        'flag_type4': False,
        'flag_type5': False,
        'flag_type6': False,
        'flag_type8': False,
        'error_types_detected': [],
        'error_type_details': None,
    }
    
    details = []
    
    # ========================================================================
    # TYPE 2: Missing BVS parameters
    # ========================================================================
    if site_data:
        for site in site_data:
            if site.get('bvs') is None and site.get('element') is not None:
                elem = site.get('element')
                valence = site.get('valence')
                if elem and valence:
                    # Check if parameters exist
                    R0, B = get_bv_parameters_fractional(elem, valence, 'O')
                    if R0 is None:
                        flags['flag_type2'] = True
                        if 'TYPE2' not in flags['error_types_detected']:
                            flags['error_types_detected'].append('TYPE2')
                            details.append(f"T2:No BVS params {elem}{valence:+g}-O")
    
    # ========================================================================
    # TYPE 3: Vacancy disorder / nonstoichiometric
    # ========================================================================
    if formula:
        try:
            parsed = parse_chemical_formula(formula)
            for elem, count in parsed.items():
                if count != int(count) and abs(count - round(count)) > 0.05:
                    flags['flag_type3'] = True
                    if 'TYPE3' not in flags['error_types_detected']:
                        flags['error_types_detected'].append('TYPE3')
                        details.append(f"T3:Nonstoich {elem}={count:.2f}")
                    break
        except:
            pass
    
    # ========================================================================
    # TYPE 4: Heterovalent disorder ΔV ≥ 2
    # ========================================================================
    if spinel_results:
        # Check heterovalent_disorder flag from SpinelAnalyzer
        if spinel_results.get('heterovalent_disorder') == True:
            # Calculate max ΔV from site occupancies if available
            delta_v = spinel_results.get('max_valence_difference', 0)
            if delta_v >= 2:
                flags['flag_type4'] = True
                if 'TYPE4' not in flags['error_types_detected']:
                    flags['error_types_detected'].append('TYPE4')
                    details.append(f"T4:Heterovalent ΔV={delta_v}")
    
    # ========================================================================
    # TYPE 5: Significant site disorder (sub-unity total occupancy = vacancies)
    # ========================================================================
    if site_data:
        for site in site_data:
            occ_str = site.get('occupancy_string', '')
            if occ_str and occ_str != 'unknown':
                try:
                    # Parse "Fe:0.800,Ni:0.200" format → sum individual occupancies
                    total_occ = sum(
                        float(part.split(':')[1]) 
                        for part in occ_str.split(',') 
                        if ':' in part
                    )
                    if total_occ < 0.90:
                        flags['flag_type5'] = True
                        if 'TYPE5' not in flags['error_types_detected']:
                            flags['error_types_detected'].append('TYPE5')
                            elem = site.get('element', '?')
                            details.append(f"T5:Disorder {elem} total_occ={total_occ:.2f}")
                        break
                except (ValueError, IndexError):
                    pass
    
    # Also check spinel classification for random disorder
    if spinel_results:
        classification = spinel_results.get('spinel_classification', '')
        if classification == 'random':
            if not flags['flag_type5']:  # Don't double-flag
                flags['flag_type5'] = True
                if 'TYPE5' not in flags['error_types_detected']:
                    flags['error_types_detected'].append('TYPE5')
                    details.append("T5:Random cation disorder")
    
    # ========================================================================
    # TYPE 6A: Pre-modern techniques (pre-1980)
    # ========================================================================
    try:
        with open(cif_path, 'r', errors='ignore') as f:
            content = f.read()
        
        year = None
        # Try multiple year fields
        for pattern in [r'_citation_year\s+(\d{4})', 
                        r'_journal_year\s+(\d{4})',
                        r"primary\s+'[^']+'\s+(\d{4})"]:
            match = re.search(pattern, content)
            if match:
                year = int(match.group(1))
                break
        
        if year and year < 1980:
            flags['flag_type6'] = True
            if 'TYPE6' not in flags['error_types_detected']:
                flags['error_types_detected'].append('TYPE6')
                details.append(f"T6:Pre-modern ({year})")
    except:
        pass
    
    # ========================================================================
    # TYPE 8: Shannon radii violation (>7% deviation for pure-occupancy sites)
    # Literature basis: <5% = accurate (ML radii papers), 7% catches errors
    # while allowing natural variation from polyhedral distortion
    # ========================================================================
    SHANNON_DEVIATION_THRESHOLD = 7.0  # percent
    
    if site_data:
        type8_sites = []
        for site in site_data:
            # Only flag pure-occupancy sites (single cation, no mixing)
            has_mixed = site.get('has_mixed_occupancy', False)
            n_elements = site.get('n_elements_at_site', 1)
            
            if not has_mixed and n_elements == 1:
                shannon_diff = site.get('shannon_diff')
                if shannon_diff is not None and abs(shannon_diff) > SHANNON_DEVIATION_THRESHOLD:
                    elem = site.get('element', '?')
                    type8_sites.append(f"{elem}:{shannon_diff:+.1f}%")
        
        if type8_sites:
            flags['flag_type8'] = True
            if 'TYPE8' not in flags['error_types_detected']:
                flags['error_types_detected'].append('TYPE8')
                details.append(f"T8:Shannon violation {','.join(type8_sites)}")
    
    # ========================================================================
    # Compile results
    # ========================================================================
    
    # ========================================================================
    # Compile results
    # ========================================================================
    flags['error_types_detected'] = ','.join(flags['error_types_detected']) if flags['error_types_detected'] else None
    flags['error_type_details'] = '; '.join(details) if details else None
    
    return flags

def parse_atom_type_with_valence(atom_type_raw):
    """
    Parse atom type symbol to extract element AND explicit valence.
    
    Examples:
        "Fe3+" -> ('Fe', 3)
        "Fe2+" -> ('Fe', 2)
        "O2-"  -> ('O', -2)
        "Mn"   -> ('Mn', None)  # No explicit valence
    
    Returns:
        tuple: (element_symbol, explicit_valence or None)
    """
    if not atom_type_raw:
        return None, None
    
    atom_type_raw = atom_type_raw.strip()
    
    # Pattern: Element + number + charge sign (Fe3+, O2-, Mn2+)
    match = re.match(r'^([A-Z][a-z]?)(\d+)?([+-])$', atom_type_raw)
    
    if match:
        element = match.group(1)
        number = match.group(2)
        sign = match.group(3)
        
        if number and sign:
            valence = int(number) if sign == '+' else -int(number)
            return element, valence
        elif sign:
            valence = 1 if sign == '+' else -1
            return element, valence
    
    # Alternative: charge before number (Fe+3, O-2)
    match = re.match(r'^([A-Z][a-z]?)([+-])(\d+)$', atom_type_raw)
    if match:
        element = match.group(1)
        sign = match.group(2)
        number = match.group(3)
        valence = int(number) if sign == '+' else -int(number)
        return element, valence
    
    # No explicit valence - just extract element
    element = re.sub(r'[0-9+-]+$', '', atom_type_raw)
    element = element.capitalize() if len(element) > 1 else element.upper()
    
    return element, None


def extract_per_atom_valences(cif_path):
    """
    Extract explicit valences from each atom row in the CIF.

    For mixed-valence CIFs like jacobsite:
        Fe2 Fe2+ 16 d 0.625 0.625 0.625 . 0.375
        Fe3 Fe3+ 16 d 0.625 0.625 0.625 . 0.56

    v1.6 change: when the _atom_type loop contains a more precise (fractional)
    oxidation number than the type symbol suggests — as in LiMn2O4 where the
    symbol "Mn4+" labels a value of 3.5 — the fractional value is preferred.

    Returns:
        dict: {site_label: {'element': str, 'explicit_valence': int|float|None,
                           'occupancy': float, 'type_symbol': str}}
    """
    result = {}

    # v1.6: Read the _atom_type loop first so we have the true valence map
    try:
        _, _, _, type_map = extract_cif_oxidation_states(cif_path)
    except Exception:
        type_map = {}

    try:
        with open(cif_path, 'r', errors='ignore') as f:
            content = f.read()
    except:
        return result

    lines = content.split('\n')

    in_atom_loop = False
    columns = []
    label_col = type_col = occ_col = None

    for line in lines:
        line_stripped = line.strip()

        if line_stripped.startswith('loop_'):
            in_atom_loop = False
            columns = []
            label_col = type_col = occ_col = None
            continue

        if line_stripped.startswith('_atom_site_'):
            col_idx = len(columns)
            columns.append(line_stripped)

            if line_stripped == '_atom_site_label':
                label_col = col_idx
            elif line_stripped == '_atom_site_type_symbol':
                type_col = col_idx
            elif line_stripped == '_atom_site_occupancy':
                occ_col = col_idx

            in_atom_loop = True
            continue

        if in_atom_loop and columns and not line_stripped.startswith('_'):
            if not line_stripped or line_stripped.startswith('#') or line_stripped.startswith('loop_'):
                in_atom_loop = False
                continue

            parts = line_stripped.split()

            label = parts[label_col] if label_col is not None and label_col < len(parts) else None

            type_symbol = None
            element, explicit_valence = None, None
            if type_col is not None and type_col < len(parts):
                type_symbol = parts[type_col]
                element, explicit_valence = parse_atom_type_with_valence(type_symbol)

                # v1.6: if the _atom_type loop has a more precise value, use it.
                # This is the key fix for LiMn2O4 "Mn4+" where the true value is 3.5.
                if type_symbol in type_map:
                    loop_valence = type_map[type_symbol]
                    # Prefer the loop value whenever it differs from the symbol-parsed one
                    # (handles both fractional overrides and any other mismatches)
                    if explicit_valence is None or loop_valence != explicit_valence:
                        explicit_valence = loop_valence

            occupancy = 1.0
            if occ_col is not None and occ_col < len(parts):
                try:
                    occ_str = parts[occ_col]
                    if occ_str not in ['?', '.']:
                        occupancy = float(occ_str.split('(')[0])
                except:
                    pass

            if label and element:
                result[label] = {
                    'element': element,
                    'explicit_valence': explicit_valence,
                    'occupancy': occupancy,
                    'type_symbol': type_symbol,
                }

    return result

def get_oxidation_states_for_analysis(cif_path, formula, config_states=None):
    """
    Get oxidation states with priority: config > CIF explicit > CIF inferred > None.
    
    This is the main entry point for oxidation state determination in Eir 1.4.
    
    Args:
        cif_path: Path to CIF file
        formula: Chemical formula string
        config_states: Dict of {element: charge} from config.json (highest priority)
    
    Returns:
        tuple: (oxidation_states, source, type7_flag_result)
        - oxidation_states: dict or None
        - source: 'config' | 'cif_explicit' | 'cif_inferred' | None
        - type7_flag_result: dict from flag_oxidation_state_issues
    """
    # Always check for Type 7 issues
    type7_result = flag_oxidation_state_issues(cif_path, formula)
    
    # Priority 1: Config file (user override)
    if config_states:
        return config_states, 'config', type7_result
    
    # Priority 2: CIF explicit oxidation states
    if type7_result['quality'] == 'explicit':
        return type7_result['oxidation_states'], 'cif_explicit', type7_result
    
    # Priority 3: CIF inferred oxidation states
    if type7_result['quality'] == 'inferred':
        return type7_result['oxidation_states'], 'cif_inferred', type7_result
    
    # No oxidation states available
    return None, None, type7_result


# ============================================================================
# ENHANCED STRUCTURAL PARAMETER EXTRACTION (NEW IN v1.4)
# ============================================================================

def extract_lattice_parameters(cif_path):
    """
    Extract unit cell parameters from CIF file.
    
    Returns:
        dict with: cell_a, cell_b, cell_c, cell_alpha, cell_beta, cell_gamma, cell_volume
    """
    result = {
        'cell_a': None, 'cell_a_esd': None,
        'cell_b': None, 'cell_b_esd': None,
        'cell_c': None, 'cell_c_esd': None,
        'cell_alpha': None, 'cell_beta': None, 'cell_gamma': None,
        'cell_volume': None, 'cell_volume_esd': None,
    }
    
    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
    except:
        return result
    
    patterns = {
        'cell_a': r'_cell_length_a\s+(\d+\.?\d*)\(?(\d*)\)?',
        'cell_b': r'_cell_length_b\s+(\d+\.?\d*)\(?(\d*)\)?',
        'cell_c': r'_cell_length_c\s+(\d+\.?\d*)\(?(\d*)\)?',
        'cell_alpha': r'_cell_angle_alpha\s+(\d+\.?\d*)',
        'cell_beta': r'_cell_angle_beta\s+(\d+\.?\d*)',
        'cell_gamma': r'_cell_angle_gamma\s+(\d+\.?\d*)',
        'cell_volume': r'_cell_volume\s+(\d+\.?\d*)\(?(\d*)\)?',
    }
    
    for key, pattern in patterns.items():
        match = re.search(pattern, content)
        if match:
            result[key] = float(match.group(1))
            if key in ['cell_a', 'cell_b', 'cell_c', 'cell_volume']:
                if match.lastindex >= 2 and match.group(2):
                    val_str = match.group(1)
                    esd_digits = match.group(2)
                    if '.' in val_str:
                        decimal_places = len(val_str.split('.')[1])
                        result[f'{key}_esd'] = int(esd_digits) * (10 ** -decimal_places)
    
    return result


def extract_space_group(cif_path):
    """
    Extract space group from CIF with multiple tag variants.
    
    Returns:
        tuple: (space_group_symbol, space_group_number)
    """
    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
    except:
        return None, None
    
    space_group = None
    space_group_number = None
    
    # Try multiple tag variants
    sg_patterns = [
        r"_symmetry_space_group_name_H-M\s+['\"]?([^'\"\n]+)",
        r"_space_group_name_H-M_alt\s+['\"]?([^'\"\n]+)",
        r"_space_group\.name_H-M_alt\s+['\"]?([^'\"\n]+)",
    ]
    
    for pattern in sg_patterns:
        match = re.search(pattern, content)
        if match:
            space_group = match.group(1).strip().strip("'\"")
            break
    
    num_patterns = [
        r"_symmetry_Int_Tables_number\s+(\d+)",
        r"_space_group_IT_number\s+(\d+)",
    ]
    
    for pattern in num_patterns:
        match = re.search(pattern, content)
        if match:
            space_group_number = int(match.group(1))
            break
    
    return space_group, space_group_number


def extract_anion_parameter(cif_path, formula=None):
    """
    Extract anion positional parameters from a spinel CIF file.
    
    v1.7.3: Generalised anion parameter framework for all spinel space groups.
    
    Cubic Fd-3m:     32e (x, x, x)        -> 1 free parameter (u)
    Ordered P4332:   24e (x, 0, 0) + 8c   -> 2 sites, 2 parameters
    Tetragonal I41/amd: 16h (0, y, z)      -> 2 free parameters
    Orthorhombic Fddd:  32h (x, y, z)      -> 3 free parameters
    Orthorhombic Imma:  8h (0, y, z)       -> 2 free parameters
    
    New metrics:
        u_eff:   Effective anion parameter (comparable across all space groups)
        delta_u: Anion sublattice anisotropy (0 for cubic, >0 for distorted)
    
    For Fd-3m, detects origin choice and standardises to origin 2.
    
    Returns:
        dict with: u_parameter, u_parameter_raw, u_deviation, origin_choice,
                   anion_x, anion_y, anion_z, anion_wyckoff, anion_multiplicity,
                   u_eff, delta_u, anion_site_count, anion_x_2/y_2/z_2/wyckoff_2
    """
    result = {
        'u_parameter': None,           # Cubic Fd-3m only: standardised to origin 2
        'u_parameter_raw': None,       # As extracted from CIF (before origin correction)
        'u_deviation': None,           # Deviation from ideal (0.25)
        'origin_choice': None,         # 1 or 2 (detected, cubic only)
        'anion_x': None,              # Primary anion site fractional x
        'anion_y': None,              # Primary anion site fractional y
        'anion_z': None,              # Primary anion site fractional z
        'anion_wyckoff': None,         # Full Wyckoff symbol e.g. "32e", "16h"
        'anion_multiplicity': None,    # Just the number e.g. 32
        'anion_wyckoff_letter': None,  # Just the letter e.g. "e"
        # v1.7.3: Generalised anion parameters
        'u_eff': None,                 # Effective u (all space groups)
        'delta_u': None,              # Anion sublattice anisotropy
        'anion_site_count': None,      # Number of distinct anion Wyckoff sites
        'anion_x_2': None,            # Second anion site x (P4332 only)
        'anion_y_2': None,            # Second anion site y
        'anion_z_2': None,            # Second anion site z
        'anion_wyckoff_2': None,       # Second anion Wyckoff symbol
    }
    
    # Determine anion elements
    anion_elements = {'O', 'S', 'Se', 'Te', 'F', 'Cl', 'Br', 'I', 'N'}
    if formula:
        formula_elements = set(re.findall(r'[A-Z][a-z]?', formula))
        if formula_elements & {'S', 'Se', 'Te'}:
            anion_elements = {'S', 'Se', 'Te'}
        elif 'O' in formula_elements:
            anion_elements = {'O'}
    
    try:
        with open(cif_path, 'r', errors='ignore') as f:
            content = f.read()
    except:
        return result
    
    lines = content.split('\n')
    
    # Parse atom_site loop
    in_atom_loop = False
    columns = []
    label_col = type_col = x_col = y_col = z_col = None
    wyckoff_col = multiplicity_col = None
    
    for line in lines:
        line_stripped = line.strip()
        
        if line_stripped.startswith('loop_'):
            in_atom_loop = False
            columns = []
            label_col = type_col = x_col = y_col = z_col = None
            wyckoff_col = multiplicity_col = None
            continue
        
        if line_stripped.startswith('_atom_site_'):
            col_idx = len(columns)
            columns.append(line_stripped)
            
            if line_stripped == '_atom_site_label':
                label_col = col_idx
            elif line_stripped == '_atom_site_type_symbol':
                type_col = col_idx
            elif line_stripped == '_atom_site_fract_x':
                x_col = col_idx
            elif line_stripped == '_atom_site_fract_y':
                y_col = col_idx
            elif line_stripped == '_atom_site_fract_z':
                z_col = col_idx
            elif line_stripped == '_atom_site_Wyckoff_symbol':
                wyckoff_col = col_idx
            elif line_stripped == '_atom_site_symmetry_multiplicity':
                multiplicity_col = col_idx
            
            in_atom_loop = True
            continue
        
        if in_atom_loop and columns and not line_stripped.startswith('_'):
            if not line_stripped or line_stripped.startswith('#') or line_stripped.startswith('loop_'):
                in_atom_loop = False
                continue
            
            parts = line_stripped.split()
            required_cols = [c for c in [label_col, type_col, x_col, y_col, z_col] if c is not None]
            if not required_cols or len(parts) < max(required_cols) + 1:
                continue
            
            # Get element type
            elem = None
            if type_col is not None and type_col < len(parts):
                elem_raw = parts[type_col]
                elem = re.sub(r'[0-9+-]+', '', elem_raw)
                elem = elem.capitalize() if len(elem) > 1 else elem.upper()
            elif label_col is not None and label_col < len(parts):
                label = parts[label_col]
                elem_match = re.match(r'([A-Z][a-z]?)', label)
                elem = elem_match.group(1) if elem_match else None
            
            if elem not in anion_elements:
                continue
            
            # ---- Found an anion site ----
            
            def parse_coord(val_str):
                """Parse coordinate value, handling ESDs in parentheses."""
                if not val_str or val_str in ['?', '.']:
                    return None
                match = re.match(r'^([+-]?\d*\.?\d+)', val_str)
                if match:
                    try:
                        return float(match.group(1))
                    except:
                        return None
                return None
            
            this_x = parse_coord(parts[x_col]) if x_col is not None and x_col < len(parts) else None
            this_y = parse_coord(parts[y_col]) if y_col is not None and y_col < len(parts) else None
            this_z = parse_coord(parts[z_col]) if z_col is not None and z_col < len(parts) else None
            
            # Extract Wyckoff info for this site
            this_wyckoff_letter = parts[wyckoff_col] if wyckoff_col is not None and wyckoff_col < len(parts) else None
            this_multiplicity = None
            if multiplicity_col is not None and multiplicity_col < len(parts):
                try:
                    this_multiplicity = int(parts[multiplicity_col])
                except:
                    pass
            
            # Build full Wyckoff symbol
            if this_multiplicity is not None and this_wyckoff_letter is not None:
                this_wyckoff = f"{this_multiplicity}{this_wyckoff_letter}"
            elif this_wyckoff_letter is not None:
                # Fallback multiplicity lookup (common spinel space groups)
                wyckoff_multiplicities = {
                    'a': 8, 'b': 8, 'c': 16, 'd': 16, 'e': 32, 'f': 48,
                    'g': 48, 'h': 96, 'i': 96, 'j': 192
                }
                mult = wyckoff_multiplicities.get(this_wyckoff_letter.lower())
                if mult:
                    this_wyckoff = f"{mult}{this_wyckoff_letter}"
                    this_multiplicity = mult
                else:
                    this_wyckoff = this_wyckoff_letter
            else:
                this_wyckoff = None
            
            # Store to primary or secondary anion site
            if result['anion_x'] is None:
                # First anion site
                result['anion_x'] = this_x
                result['anion_y'] = this_y
                result['anion_z'] = this_z
                result['anion_wyckoff'] = this_wyckoff
                result['anion_wyckoff_letter'] = this_wyckoff_letter
                result['anion_multiplicity'] = this_multiplicity
                result['anion_site_count'] = 1
            elif result['anion_x_2'] is None:
                # Second anion site (e.g. P4332: 24e + 8c)
                result['anion_x_2'] = this_x
                result['anion_y_2'] = this_y
                result['anion_z_2'] = this_z
                result['anion_wyckoff_2'] = this_wyckoff
                result['anion_site_count'] = 2
            else:
                break  # Two anion sites found, stop
            continue  # Keep looking for additional anion sites
    
    # =========================================================================
    # Generalised anion parameter calculation (v1.7.3)
    # =========================================================================
    ax = result['anion_x']
    ay = result['anion_y']
    az = result['anion_z']
    wy = result.get('anion_wyckoff', '') or ''
    
    if ax is not None:
        
        # ----- Cubic Fd-3m: 32e (x, x, x) -> single u parameter -----
        if '32e' in wy or (ay is not None and az is not None 
                           and abs(ax - ay) < 0.001 and abs(ax - az) < 0.001
                           and '16h' not in wy and '8h' not in wy):
            u_raw = ax
            result['u_parameter_raw'] = u_raw
            
            # Detect origin choice based on u value
            # Origin 1 (S): u ~ 0.375 (range 0.35-0.42)
            # Origin 2 (Z): u ~ 0.250 (range 0.23-0.30)
            if u_raw > 0.32:
                result['origin_choice'] = 1
                u_standardised = u_raw - 0.125
            else:
                result['origin_choice'] = 2
                u_standardised = u_raw
            
            result['u_parameter'] = round(u_standardised, 4)
            result['u_deviation'] = round(u_standardised - 0.25, 4)
            result['u_eff'] = result['u_parameter']
            result['delta_u'] = 0.0
        
        # ----- Tetragonal I41/amd: 16h (0, y, z) -> two parameters -----
        elif '16h' in wy or '8h' in wy:
            if ay is not None and az is not None:
                result['u_eff'] = round((ay + az) / 2, 4)
                result['delta_u'] = round(abs(ay - az), 4)
                # u_parameter left as None (cubic u not defined for tetragonal)
                # u_deviation computed from u_eff for cross-system comparison
                result['u_deviation'] = round(result['u_eff'] - 0.25, 4)
        
        # ----- Ordered cubic P4332: 24e + 8c (two distinct anion sites) -----
        elif '24e' in wy or '8c' in wy:
            ax2 = result.get('anion_x_2')
            ay2 = result.get('anion_y_2')
            az2 = result.get('anion_z_2')
            wy2 = result.get('anion_wyckoff_2', '') or ''
            
            # 8c site: coords (x, x, x), so u = x
            # 24e site: coords (x, 0, 0), so the free param is x
            u_8c = None
            u_24e = None
            
            if '8c' in wy:
                u_8c = ax
                if ax2 is not None and '24e' in wy2:
                    u_24e = ax2
                elif ax2 is not None:
                    u_24e = ax2  # Assume second site is 24e
            elif '24e' in wy:
                u_24e = ax
                if ax2 is not None and '8c' in wy2:
                    u_8c = ax2
                elif ax2 is not None:
                    u_8c = ax2  # Assume second site is 8c
            
            # Effective u: weighted by multiplicity (24e = 75%, 8c = 25%)
            if u_24e is not None and u_8c is not None:
                result['u_eff'] = round((3 * u_24e + u_8c) / 4, 4)
                result['delta_u'] = round(abs(u_24e - u_8c), 4)
                result['u_deviation'] = round(result['u_eff'] - 0.25, 4)
            elif u_8c is not None:
                # Only 8c found (unusual but handle gracefully)
                result['u_eff'] = round(u_8c, 4)
                result['u_parameter'] = round(u_8c, 4)
                result['u_deviation'] = round(u_8c - 0.25, 4)
            elif u_24e is not None:
                result['u_eff'] = round(u_24e, 4)
                result['u_deviation'] = round(u_24e - 0.25, 4)
        
        # ----- Orthorhombic Fddd: 32h (x, y, z) -> three parameters -----
        elif '32h' in wy:
            if ay is not None and az is not None:
                result['u_eff'] = round((ax + ay + az) / 3, 4)
                result['delta_u'] = round(max(abs(ax - ay), abs(ay - az), abs(ax - az)), 4)
                result['u_deviation'] = round(result['u_eff'] - 0.25, 4)
        
        # ----- Fallback: attempt cubic assumption -----
        else:
            u_raw = ax
            result['u_parameter_raw'] = u_raw
            if u_raw > 0.32:
                result['origin_choice'] = 1
                u_standardised = u_raw - 0.125
            else:
                result['origin_choice'] = 2
                u_standardised = u_raw
            result['u_parameter'] = round(u_standardised, 4)
            result['u_deviation'] = round(u_standardised - 0.25, 4)
            result['u_eff'] = result['u_parameter']
    
    return result


def extract_site_occupancies(cif_path):
    """
    Extract site occupancies with full detail.
    
    Returns:
        dict: {site_label: {element, occupancy, wyckoff, fract_x, fract_y, fract_z}}
    """
    result = {}
    
    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
    except:
        return result
    
    in_atom_loop = False
    columns = []
    col_indices = {}
    
    for line in lines:
        line_stripped = line.strip()
        
        if line_stripped == 'loop_':
            in_atom_loop = False
            columns = []
            col_indices = {}
            continue
        
        if line_stripped.startswith('_atom_site_'):
            columns.append(line_stripped)
            col_idx = len(columns) - 1
            tag_map = {
                '_atom_site_label': 'label',
                '_atom_site_type_symbol': 'type',
                '_atom_site_occupancy': 'occupancy',
                '_atom_site_fract_x': 'fract_x',
                '_atom_site_fract_y': 'fract_y',
                '_atom_site_fract_z': 'fract_z',
                '_atom_site_Wyckoff_symbol': 'wyckoff',
                '_atom_site_symmetry_multiplicity': 'multiplicity',
            }
            for tag, name in tag_map.items():
                if line_stripped == tag:
                    col_indices[name] = col_idx
            in_atom_loop = True
            continue
        
        if in_atom_loop and columns and not line_stripped.startswith('_'):
            if not line_stripped or line_stripped.startswith('#') or line_stripped.startswith('loop_'):
                in_atom_loop = False
                continue
            
            parts = line_stripped.split()
            
            def get_val(name):
                if name in col_indices and col_indices[name] < len(parts):
                    return parts[col_indices[name]]
                return None
            
            def parse_numeric(val_str):
                if not val_str or val_str in ['?', '.']:
                    return None, None
                match = re.match(r'^([+-]?\d+\.?\d*)\((\d+)\)$', val_str)
                if match:
                    val = float(match.group(1))
                    if '.' in match.group(1):
                        dec_places = len(match.group(1).split('.')[1])
                        esd = int(match.group(2)) * (10 ** -dec_places)
                    else:
                        esd = float(match.group(2))
                    return val, esd
                try:
                    return float(val_str), None
                except:
                    return None, None
            
            label = get_val('label')
            if label:
                type_raw = get_val('type')
                elem = None
                if type_raw:
                    elem = re.sub(r'[0-9+-]+', '', type_raw)
                    elem = elem.capitalize() if len(elem) > 1 else elem.upper()
                
                occ, _ = parse_numeric(get_val('occupancy'))
                fx, _ = parse_numeric(get_val('fract_x'))
                fy, _ = parse_numeric(get_val('fract_y'))
                fz, _ = parse_numeric(get_val('fract_z'))
                wyckoff = get_val('wyckoff')
                mult_str = get_val('multiplicity')
                mult = int(mult_str) if mult_str and mult_str.isdigit() else None
                
                result[label] = {
                    'element': elem,
                    'occupancy': occ if occ is not None else 1.0,
                    'wyckoff': wyckoff,
                    'multiplicity': mult,
                    'fract_x': fx, 'fract_y': fy, 'fract_z': fz,
                }
    
    return result


def extract_all_structural_parameters(cif_path, formula=None):
    """
    Master function to extract all structural parameters from a CIF.
    
    Returns:
        dict with all structural parameters
    """
    result = {
        'cif_path': cif_path,
        'formula': formula,
        'space_group': None,
        'space_group_number': None,
        'extraction_warnings': [],
    }
    
    # Space group
    sg, sg_num = extract_space_group(cif_path)
    result['space_group'] = sg
    result['space_group_number'] = sg_num
    
    # Lattice parameters
    lattice = extract_lattice_parameters(cif_path)
    result.update(lattice)
    
    # Anion parameter
    anion = extract_anion_parameter(cif_path, formula)
    result.update(anion)
    
    # Site occupancies
    result['site_occupancies'] = extract_site_occupancies(cif_path)
    
    # Validation warnings (v1.7.3: check u_eff for all space groups)
    u_check = result.get('u_parameter') or result.get('u_eff')
    if u_check is not None:
        if u_check < 0.20 or u_check > 0.30:
            result['extraction_warnings'].append(
                f"Unusual anion parameter: u_eff={u_check:.4f} (expected 0.24-0.27)"
            )
    
    return result


# ============================================================================
# CONFIG FILE SUPPORT
# ============================================================================

def load_oxidation_states_from_config(config_path=None, config_dict=None):
    """
    Load oxidation states from config file or dictionary.
    """
    if config_dict is not None:
        if not isinstance(config_dict, dict):
            print("Error: config_dict must be a dictionary")
            return None
        
        try:
            validated = {str(k): int(v) for k, v in config_dict.items()}
            print(f"Loaded oxidation states from provided dictionary:")
            for elem in sorted(validated.keys()):
                print(f"    {elem}: {validated[elem]:+d}")
            return validated
        except (ValueError, TypeError) as e:
            print(f"Error validating config_dict: {e}")
            return None
    
    if config_path is not None:
        if not os.path.exists(config_path):
            print(f"Config file not found: {config_path}")
            return None
        
        try:
            with open(config_path, 'r') as f:
                data = json.load(f)
            
            validated = {str(k): int(v) for k, v in data.items()}
            print(f"Loaded oxidation states from {config_path}:")
            for elem in sorted(validated.keys()):
                print(f"    {elem}: {validated[elem]:+d}")
            return validated
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON in {config_path}: {e}")
            return None
        except (ValueError, TypeError) as e:
            print(f"Error validating oxidation states in {config_path}: {e}")
            return None
    
    return None


def auto_detect_config_file(directory):
    """Automatically search for config.json in the target directory."""
    config_path = os.path.join(directory, "config.json")
    if os.path.exists(config_path):
        print(f"Found config.json in {directory}")
        return config_path
    return None


# ============================================================================
# 3D VISUALISATION OF PROBLEMATIC SITES
# ============================================================================

def visualize_problematic_site_3d(cif_path, element, site_label, bonds_info, 
                                   deviation, formal_charge, bvs_value, filename=None):
    """Create interactive 3D visualisation showing unit cell with highlighted problematic element."""
    try:
        import py3Dmol
        from IPython.display import display
    except ImportError:
        print("  [3D visualisation unavailable - py3Dmol not installed]")
        return
    
    try:
        view = py3Dmol.view(width=650, height=500)
        
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            cif_string = f.read()
        
        view.addModel(cif_string, 'cif')
        view.replicateUnitCell(2, 2, 2)
        
        view.setStyle({'stick': {'radius': 0.2}, 'sphere': {'scale': 0.35}})
        view.addStyle({'elem': element}, {'sphere': {'color': 'red', 'scale': 0.7}})
        view.addStyle({'elem': 'O'}, {'sphere': {'color': 'blue', 'scale': 0.45}})
        view.addStyle({'elem': 'F'}, {'sphere': {'color': 'lightblue', 'scale': 0.40}})
        view.addStyle({'elem': 'Cl'}, {'sphere': {'color': 'green', 'scale': 0.50}})
        view.addStyle({'elem': 'S'}, {'sphere': {'color': 'yellow', 'scale': 0.50}})
        
        other_metals = ['Na', 'K', 'Li', 'Cr', 'Ti', 'Mn', 'Ni', 'Co', 'Fe', 'V', 'Sb', 'Mg', 
                       'Ca', 'Sr', 'Ba', 'Zn', 'Cd', 'Cu', 'Al']
        for metal in other_metals:
            if metal != element:
                view.addStyle({'elem': metal}, {'sphere': {'color': 'grey', 'scale': 0.4}})
        
        view.zoomTo()
        view.rotate(45, {'x': 1, 'y': 1, 'z': 0})
        
        file_id = ""
        if filename:
            collcode_match = re.search(r'CollCode(\d+)', filename)
            if collcode_match:
                file_id = f" in CollCode{collcode_match.group(1)}"
            else:
                file_id = f" ({filename.replace('.cif', '')})"
        
        print(f"\n  3D VISUALISATION: {element} at site {site_label}{file_id}")
        print(f"  BVS: {bvs_value:.3f}  |  Expected: {formal_charge:+d}  |  Delta: {deviation:+.3f} v.u.")
        
        if bonds_info:
            print("  Coordination Environment:")
            for bond in bonds_info[:6]:
                print(f"    - {bond['count']}x {bond['distance']:.4f} A")
        
        display(view)
        print()
        
    except Exception as e:
        print(f"  [3D visualisation failed: {e}]")


# ============================================================================
# JAHN-TELLER ION DETECTION
# ============================================================================

JAHN_TELLER_IONS = {
    'Cr': [2],
    'Mn': [3],
    'Cu': [2],
    'Ni': [3],
    'Co': [2]
}

def is_jahn_teller_ion(element, valence):
    """Check if an element/valence combination is Jahn-Teller active."""
    return element in JAHN_TELLER_IONS and abs(valence) in JAHN_TELLER_IONS[element]


# ============================================================================
# ADAPTIVE GAP DETECTION FOR COORDINATION
# ============================================================================

def detect_coordination_adaptive(bond_distances, element=None, valence=None, min_gap=0.05):
    """Adaptive coordination number detection with special handling for Jahn-Teller ions."""
    if len(bond_distances) < 2:
        return len(bond_distances)
    
    is_jt = False
    if element and valence:
        is_jt = is_jahn_teller_ion(element, valence)
    
    effective_min_gap = min_gap * 2.0 if is_jt else min_gap
    
    gaps = []
    for i in range(len(bond_distances) - 1):
        gap = bond_distances[i+1] - bond_distances[i]
        if gap >= effective_min_gap:
            gaps.append((gap, i+1))
    
    if not gaps:
        return len(bond_distances)
    
    largest_gap, cn = max(gaps, key=lambda x: x[0])
    return cn


# ============================================================================
# ICSD METADATA EXTRACTION WITH TEMPERATURE & PRESSURE
# ============================================================================

def extract_icsd_metadata(cif_path):
    """
    Extract publication metadata AND measurement conditions from ICSD CIF file.
    """
    metadata = {
        'icsd_code': None,
        'creation_date': None,
        'year': None,
        'journal': None,
        'authors': [],
        'r_factor': None,
        'density': None,
        'measurement_temperature_K': None,
        'measurement_pressure_Pa': None,
        'measurement_pressure_GPa': None,
        'measurement_conditions': 'unknown'
    }
    
    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        
        in_citation_loop = False
        in_author_loop = False
        citation_columns = []
        author_columns = []
        
        for i, line in enumerate(lines):
            line_stripped = line.strip()
            
            if line_stripped.startswith('_database_code_ICSD'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    metadata['icsd_code'] = parts[1]
            
            elif line_stripped.startswith('_audit_creation_date'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    metadata['creation_date'] = parts[1]
            
            elif line_stripped.startswith('_exptl_crystal_density_diffrn'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    try:
                        metadata['density'] = float(parts[1])
                    except ValueError:
                        pass
            
            elif line_stripped.startswith('_diffrn_ambient_temperature'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    try:
                        temp_str = parts[1].split('(')[0].strip()
                        metadata['measurement_temperature_K'] = float(temp_str)
                    except ValueError:
                        pass
            
            elif line_stripped.startswith('_cell_measurement_temperature'):
                parts = line_stripped.split()
                if len(parts) > 1 and metadata['measurement_temperature_K'] is None:
                    try:
                        temp_str = parts[1].split('(')[0].strip()
                        metadata['measurement_temperature_K'] = float(temp_str)
                    except ValueError:
                        pass
            
            elif line_stripped.startswith('_diffrn_ambient_pressure'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    try:
                        pressure_str = parts[1].split('(')[0].strip()
                        pressure_kpa = float(pressure_str)
                        metadata['measurement_pressure_Pa'] = pressure_kpa * 1000
                        metadata['measurement_pressure_GPa'] = pressure_kpa / 1e6
                    except ValueError:
                        pass
            
            elif any(tag in line_stripped for tag in ['_refine_ls_R_factor', '_refine_ls_wR_factor']):
                parts = line_stripped.split()
                if len(parts) > 1:
                    try:
                        r_val = float(parts[1])
                        if metadata['r_factor'] is None or r_val < metadata['r_factor']:
                            metadata['r_factor'] = r_val
                    except ValueError:
                        pass
            
            elif line_stripped.startswith('loop_'):
                in_citation_loop = False
                in_author_loop = False
                citation_columns = []
                author_columns = []
            
            elif line_stripped.startswith('_citation_') and not line_stripped.startswith('_citation_author'):
                in_citation_loop = True
                citation_columns.append(line_stripped)
            
            elif line_stripped.startswith('_citation_author'):
                in_author_loop = True
                in_citation_loop = False
                author_columns.append(line_stripped)
            
            elif in_citation_loop and citation_columns and not line_stripped.startswith('_') and not line_stripped.startswith('#') and line_stripped:
                # Robust extraction handling semicolon-delimited multiline fields
                # (common in ICSD CIFs for long journal names).
                # Collects all field values for the primary citation row,
                # then looks up year and journal by column index.
                import shlex as _shlex
                fields = []
                in_semi = False
                semi_buf = []
                j = i

                while j < len(lines):
                    jline = lines[j].strip()
                    # Stop at next loop_ or column definition (but only after
                    # we've started collecting, to avoid stopping immediately)
                    if j > i and (jline.startswith('loop_') or
                                  (jline.startswith('_') and fields)):
                        break
                    if jline.startswith(';'):
                        if in_semi:
                            fields.append(' '.join(semi_buf))
                            semi_buf = []
                            in_semi = False
                            rest = jline[1:].strip()
                            if rest:
                                try:
                                    fields.extend(_shlex.split(rest))
                                except ValueError:
                                    pass
                        else:
                            in_semi = True
                            after = jline[1:].strip()
                            if after:
                                semi_buf.append(after)
                    elif in_semi:
                        semi_buf.append(jline)
                    elif jline:
                        try:
                            fields.extend(_shlex.split(jline))
                        except ValueError:
                            pass
                    j += 1

                if fields and fields[0] == 'primary':
                    for col_idx, col_name in enumerate(citation_columns):
                        if col_idx < len(fields):
                            if '_citation_year' in col_name:
                                try:
                                    metadata['year'] = int(fields[col_idx])
                                except ValueError:
                                    pass
                            elif '_citation_journal_full' in col_name:
                                metadata['journal'] = fields[col_idx]

                in_citation_loop = False
            
            elif in_author_loop and author_columns and not line_stripped.startswith('_') and not line_stripped.startswith('#') and not line_stripped.startswith('loop_') and line_stripped:
                import shlex
                try:
                    values = shlex.split(line_stripped)
                    for col_idx, col_name in enumerate(author_columns):
                        if '_citation_author_name' in col_name and col_idx < len(values):
                            author = values[col_idx]
                            if author and author not in metadata['authors']:
                                metadata['authors'].append(author)
                except ValueError:
                    pass
            
            elif in_author_loop and (line_stripped.startswith('_') or line_stripped.startswith('loop_')):
                in_author_loop = False
        
        if len(metadata['authors']) > 3:
            metadata['authors'] = metadata['authors'][:3] + ['et al.']
        
        # Classify measurement conditions
        temp_K = metadata['measurement_temperature_K']
        pressure_GPa = metadata['measurement_pressure_GPa']
        
        TEMP_AMBIENT_MAX = 350
        TEMP_HIGH = 500
        PRESSURE_AMBIENT_MAX = 0.0002
        PRESSURE_HIGH = 0.5
        
        if temp_K is not None or pressure_GPa is not None:
            is_high_temp = False
            is_high_pressure = False
            is_ambient_temp = True
            is_ambient_pressure = True
            
            if temp_K is not None:
                if temp_K > TEMP_HIGH:
                    is_high_temp = True
                    is_ambient_temp = False
                elif temp_K > TEMP_AMBIENT_MAX:
                    is_ambient_temp = False
            
            if pressure_GPa is not None:
                if pressure_GPa > PRESSURE_HIGH:
                    is_high_pressure = True
                    is_ambient_pressure = False
                elif pressure_GPa > PRESSURE_AMBIENT_MAX:
                    is_ambient_pressure = False
            
            if is_high_temp and is_high_pressure:
                metadata['measurement_conditions'] = 'high_T_high_P'
            elif is_high_temp:
                metadata['measurement_conditions'] = 'high_T'
            elif is_high_pressure:
                metadata['measurement_conditions'] = 'high_P'
            elif is_ambient_temp and is_ambient_pressure:
                metadata['measurement_conditions'] = 'ambient'
            else:
                metadata['measurement_conditions'] = 'elevated'
        else:
            metadata['measurement_conditions'] = 'unknown'
        
    except Exception as e:
        print(f"  [Warning: Could not extract all metadata: {e}]")
    
    return metadata


# ============================================================================
# GEOMETRY VALIDATION
# ============================================================================

def check_geometry_validity(supercell_neighbors, site_info, element_categories):
    """Check O-O and M-M distances for geometric validity."""
    results = {
        'o_o_distances': [],
        'o_o_min': None,
        'o_o_warnings': [],
        'm_m_distances': [],
        'm_m_min': None,
        'm_m_warnings': [],
        'geometry_flags': []
    }
    
    try:
        central_element = site_info['element']
        central_category = element_categories.get(central_element, {}).get('category', 'unknown')
        
        for neighbor in supercell_neighbors:
            neighbor_elem = neighbor.get('element', '')
            neighbor_cat = element_categories.get(neighbor_elem, {}).get('category', 'unknown')
            distance = neighbor.get('distance', 0)
            
            if central_element == 'O' and neighbor_elem == 'O':
                results['o_o_distances'].append(distance)
            
            elif (central_category in ['alkali_metals', 'alkaline_earth_metals', 'transition_metals_3d', 
                                       'transition_metals_4d', 'transition_metals_5d', 'lanthanides'] and
                  neighbor_cat in ['alkali_metals', 'alkaline_earth_metals', 'transition_metals_3d',
                                   'transition_metals_4d', 'transition_metals_5d', 'lanthanides']):
                results['m_m_distances'].append(distance)
        
        if results['o_o_distances']:
            results['o_o_min'] = min(results['o_o_distances'])
            if results['o_o_min'] < 2.3:
                results['o_o_warnings'].append(f"O-O distance {results['o_o_min']:.3f} A unusually short (< 2.3 A)")
                results['geometry_flags'].append('SHORT_O-O')
            elif results['o_o_min'] < 2.5:
                results['o_o_warnings'].append(f"O-O distance {results['o_o_min']:.3f} A short (face-sharing?)")
        
        if results['m_m_distances']:
            results['m_m_min'] = min(results['m_m_distances'])
            if results['m_m_min'] < 2.5:
                results['m_m_warnings'].append(f"M-M distance {results['m_m_min']:.3f} A unusually short")
                results['geometry_flags'].append('SHORT_M-M')
        
    except Exception as e:
        results['geometry_flags'].append('VALIDATION_ERROR')
        print(f"  [Geometry validation error: {e}]")
    
    return results


# ============================================================================
# MIXED OCCUPANCY DETECTION & ANALYSIS
# ============================================================================

def extract_site_occupancy_info(data_block, site_label):
    """Extract occupancy information for a specific crystallographic site."""
    try:
        labels = data_block.get('_atom_site_label', [])
        types = data_block.get('_atom_site_type_symbol', [])
        occupancies = data_block.get('_atom_site_occupancy', [])
        
        if not labels or not types:
            return None
        
        site_data = []
        for i, label in enumerate(labels):
            if label.strip() == site_label:
                element = types[i].strip().rstrip('+-0123456789')
                element = element.capitalize() if len(element) > 1 else element.upper()
                
                occ = 1.0
                if occupancies and i < len(occupancies):
                    try:
                        occ_str = occupancies[i].strip()
                        occ = float(occ_str.split('(')[0])
                    except (ValueError, AttributeError):
                        occ = 1.0
                
                site_data.append({
                    'element': element,
                    'occupancy': occ
                })
        
        return site_data if site_data else None
    
    except Exception:
        return None


def analyze_mixed_occupancy_at_site(site_coords, site_data, final_corrected_data, block_name, data_block=None):
    """Analyse mixed occupancy at a crystallographic site."""
    result = {
        'has_mixed_occupancy': False,
        'occupancy_string': 'unknown',
        'n_elements_at_site': 1,
        'disorder_type': 'pure',
        'valence_difference': 0.0,
        'max_valence_at_site': None,
        'min_valence_at_site': None
    }
    
    elements = site_data.get('elements', [])
    if not elements:
        return result
    
    valences = []
    
    if block_name in final_corrected_data:
        for coords, site_info in final_corrected_data[block_name]['sites_with_bonds'].items():
            if coords == site_coords:
                confirmed_valences = site_info.get('confirmed_valences', {})
                for elem in elements:
                    val = confirmed_valences.get(elem)
                    if val is not None:
                        valences.append(abs(val))
                break
    
    if not valences:
        for elem in elements:
            elem_data = site_data.get('element_data', {}).get(elem, {})
            val = elem_data.get('valence')
            if val is not None:
                valences.append(abs(val))
    
    result['n_elements_at_site'] = len(elements)
    
    if valences:
        result['max_valence_at_site'] = int(max(valences))
        result['min_valence_at_site'] = int(min(valences))
        result['valence_difference'] = float(max(valences) - min(valences))
    
    if len(elements) > 1:
        result['has_mixed_occupancy'] = True
        
        labels = site_data.get('labels', [])
        actual_occupancies = {}
        
        if data_block is not None and labels:
            for elem in elements:
                found = False
                for site_label in labels:
                    occ_info = extract_site_occupancy_info(data_block, site_label)
                    if occ_info:
                        for entry in occ_info:
                            if entry['element'] == elem:
                                actual_occupancies[elem] = entry['occupancy']
                                found = True
                                break
                    if found:
                        break
                
                if not found and labels:
                    match = re.search(r'(\d+)$', labels[0])
                    if match:
                        suffix = match.group(1)
                        constructed_label = f"{elem}{suffix}"
                        occ_info = extract_site_occupancy_info(data_block, constructed_label)
                        if occ_info:
                            for entry in occ_info:
                                if entry['element'] == elem:
                                    actual_occupancies[elem] = entry['occupancy']
                                    break
        
        occ_parts = []
        if actual_occupancies:
            for elem in elements:
                occ = actual_occupancies.get(elem, 0.0)
                occ_parts.append(f"{elem}:{occ:.3f}")
        else:
            for elem in elements:
                occ = 1.0 / len(elements)
                occ_parts.append(f"{elem}:{occ:.2f}")
        
        result['occupancy_string'] = ','.join(occ_parts)
    else:
        result['occupancy_string'] = f"{elements[0]}:1.00"
    
    if len(elements) == 1:
        result['disorder_type'] = 'pure'
        result['valence_difference'] = 0.0
    elif not valences:
        result['disorder_type'] = 'heterovalent'
        result['valence_difference'] = 0.0
    else:
        if len(set(valences)) == 1:
            result['disorder_type'] = 'homovalent'
            result['valence_difference'] = 0.0
        else:
            result['disorder_type'] = 'heterovalent'
            result['valence_difference'] = float(max(valences) - min(valences))
    
    return result


def calculate_discrepancy_factor(bvs_value, formal_valence):
    """Calculate d1 = (S_i - V_i) from Bosi (2014)."""
    if bvs_value is None or formal_valence is None:
        return None, None, 'unknown'
    
    d1 = bvs_value - abs(formal_valence)
    abs_d1 = abs(d1)
    
    if d1 > 0.05:
        bonding_state = 'overbonded'
    elif d1 < -0.05:
        bonding_state = 'underbonded'
    else:
        bonding_state = 'balanced'
    
    return d1, abs_d1, bonding_state


def assign_confidence_flag(disorder_type, valence_diff, abs_d1, deviation):
    """Assign confidence level for BVS result based on disorder and strain."""
    abs_dev = abs(deviation) if deviation is not None else 0
    abs_d1_val = abs_d1 if abs_d1 is not None else 0
    
    requires_review = False
    bosi_artifact_likely = False
    
    if disorder_type == 'heterovalent' and valence_diff >= 3:
        confidence = 'SPECTROSCOPY_REQUIRED'
        bosi_artifact_likely = True
        requires_review = True
        return confidence, bosi_artifact_likely, requires_review
    
    if abs_dev > 0.30 or abs_d1_val > 0.20:
        requires_review = True
    
    if disorder_type == 'pure':
        if abs_d1_val < 0.10:
            confidence = 'HIGH'
        elif abs_d1_val < 0.20:
            confidence = 'MEDIUM'
        else:
            confidence = 'LOW'
            requires_review = True
    
    elif disorder_type == 'homovalent':
        if abs_d1_val < 0.15:
            confidence = 'MEDIUM'
        elif abs_d1_val < 0.25:
            confidence = 'LOW'
        else:
            confidence = 'LOW'
            requires_review = True
    
    elif disorder_type == 'heterovalent':
        if valence_diff < 2:
            confidence = 'MEDIUM'
        elif valence_diff < 3:
            confidence = 'LOW'
            requires_review = True
        else:
            confidence = 'SPECTROSCOPY_REQUIRED'
            bosi_artifact_likely = True
            requires_review = True
    
    else:
        confidence = 'LOW'
        requires_review = True
    
    return confidence, bosi_artifact_likely, requires_review


# ============================================================================
# INITIALISATION MESSAGE
# ============================================================================

print("=" * 80)
print("Eir v1.7 helper functions loaded successfully")
print("=" * 80)
print("  - Multi-anion support enabled (O, F, Cl, Br, I, S, Se, Te, etc.)")
print("  - Config file support enabled")
print("  - 3D visualisation enabled (if py3Dmol installed)")
print("  - Adaptive gap detection enabled")
print("  - Temperature/Pressure extraction enabled")
print("  - ICSD metadata extraction with conditions classification")
print("  - Geometry validation enabled")
print("  - Mixed occupancy analysis enabled")
print("  NEW IN v1.4:")
print("  - CIF oxidation state extraction")
print("  - Type 7 error flagging (mixed valence detection)")
print("  - Anion parameter (u) extraction")
print("  - Enhanced structural parameter extraction")
print("=" * 80)

In [ ]:
# CELL 3.5: STRUCTURE-TYPE SPECIFIC ANALYSERS
# ============================================================================
# Run this cell AFTER Cell 3 (helper functions) and BEFORE Cell 4 (main workflow)
#
# VERSION HISTORY:
# - v1.6 (Apr 2026): Non-stoichiometric spinel handling (inversion pre-check),
#                     column deduplication, B_site_small_cation_fraction output
# - v1.4 (29 Jan 2026): Added u parameter extraction, cell parameters, 
#                       Type 7 error integration, enhanced output columns
# - v1.0: Initial spinel analyser with inversion parameter calculation
#
# To disable: Don't run this cell or set all analysers to False
# ============================================================================

"""
STRUCTURE-TYPE ANALYZERS v1.6
==============================

Extensible framework for detecting and analyzing specific structure types
beyond generic BVS analysis. Each analyzer is self-contained and can be
toggled on/off independently.

Currently implemented:
- SpinelAnalyzer: Detects normal/inverse/random spinels, calculates lambda,
                  extracts u parameter, cell parameters, distortion metrics
                  v1.6: non-stoichiometric spinels handled correctly

Future planned:
- PerovskiteAnalyzer: Goldschmidt tolerance factor, octahedral tilting
- PyrochloreAnalyzer: A-site/B-site ordering, 48f oxygen disorder
- GarnetAnalyzer: Dodecahedral/octahedral/tetrahedral site analysis
"""

# ============================================================================
# CONFIGURATION
# ============================================================================

ENABLED_ANALYZERS = {
    'spinel': True,       # Fd-3m cubic, P4332 ordered, I41/amd tetragonal
    'perovskite': False,  # Not implemented yet
    'pyrochlore': False,  # Not implemented yet
    'garnet': False,      # Not implemented yet
}

# ============================================================================
# BASE CLASS
# ============================================================================

class StructureAnalyzer:
    """
    Base class for structure-type specific analysis.
    
    Subclasses must implement:
    - is_applicable(): Check if structure matches this type
    - analyze(): Perform structure-specific analysis
    - get_output_columns(): Return list of CSV column names
    """
    
    name = "Generic"
    
    @classmethod
    def is_applicable(cls, space_group, formula, metadata):
        """Check if this analyzer applies to the structure."""
        raise NotImplementedError
    
    @classmethod
    def analyze(cls, cif_block, metadata, calculated_results, final_corrected_data=None, cif_path=None):
        """Perform structure-specific analysis."""
        raise NotImplementedError
    
    @classmethod
    def get_output_columns(cls):
        """Return list of column names this analyzer adds to CSV."""
        raise NotImplementedError

# ============================================================================
# SPINEL ANALYZER (Enhanced for v1.6)
# ============================================================================

class SpinelAnalyzer(StructureAnalyzer):
    """
    Analyzes spinel structures (AB2O4 and derivatives).
    
    Detects:
    - Cubic Fd-3m normal/inverse/random spinels
    - Cubic P4332 ordered binary spinels (e.g., Li2NiGe3O8)
    - Tetragonal I41/amd Jahn-Teller distorted spinels
    
    Calculates:
    - Inversion parameter lambda from site occupancies
    - Spinel type classification (normal/inverse/random)
    - A-site and B-site element identification
    - B-site ordering detection (for binary spinels)
    
    NEW IN v1.4:
    - Anion parameter (u) extraction
    - Cell parameters (a, b, c, volume)
    - c/a ratio for tetragonal systems
    - Tetragonality metrics
    
    NEW IN v1.6:
    - Non-stoichiometric spinel detection (Li-excess etc.)
    - B_site_small_cation_fraction output
    - Column deduplication (removed *_extracted)
    
    Theory:
    -------
    For AB2O4 spinel:
        lambda = 0 (normal):  A on 8a (Td), B2 on 16d (Oh)
        lambda = 1 (inverse): B on 8a (Td), AB on 16d (Oh)
        General:              A(1-lambda)B(lambda)[A(lambda)B(2-lambda)]O4
    
    Anion parameter u:
        u = 0.25: ideal cubic close packing
        u > 0.25: expanded tetrahedral site (normal spinels)
        u < 0.25: compressed tetrahedral site (inverse spinels)

    NEW IN v1.7.3:
    - Reinstated anion_y, anion_z (carry independent info for non-cubic)
    - Generalised anion parameter framework for all spinel space groups
    - u_eff: effective anion parameter (defined for all space groups)
    - delta_u: anion sublattice anisotropy (0 for cubic, >0 for distorted)
    - Multi-site anion extraction (P4332: 24e + 8c both captured)
    - anion_site_count, anion_x_2/y_2/z_2, anion_wyckoff_2 columns added
    """
    
    name = "Spinel"
    
    # Spinel space groups
    CUBIC_SPINEL_SG = [
        'F d -3 m', 'Fd-3m', 'Fd3m', 'F d -3 m :1', 'F d -3 m :2',
        'F d -3 m Z', 'Fd-3mZ',
        'F d -3 m S', 'Fd-3mS'
    ]
    
    CUBIC_ORDERED_SG = [
        'P 43 3 2', 'P4332', 'P 43 3 2 :1', 'P 43 3 2 :2',
        'P 41 3 2', 'P4132',
        'I -4 3 d', 'I-43d'
    ]
    
    TETRAGONAL_SG = [
        'I 41/a m d', 'I41/amd', 'I 41/a m d :2',
        'I 41/a m d Z', 'I41/amdZ',
        'I 41/a m d S', 'I41/amdS',
        'I 4/m', 'I4/m',
        'I -4 2 m', 'I-42m',
        'I -4', 'I-4',
        'P 41 21 2', 'P4121 2', 'P41212',
        'P 43 21 2', 'P4321 2', 'P43212'
    ]
    
    ORTHORHOMBIC_SG = [
        'F d d d', 'Fddd', 'F d d d :1', 'F d d d :2',
        'F d d d Z', 'FdddZ',
        'F d d d S', 'FdddS',
        'I m m a', 'Imma', 'I m m a :1', 'I m m a :2',
        'I 2/m 2/m 2/a', 'I2/m2/m2/a',
        'P m m a', 'Pmma', 'P m m a :1', 'P m m a :2',
        'P 21/m 21/m 2/a', 'P21/m21/m2/a',
        'P n m a', 'Pnma',
        'P b n m', 'Pbnm'
    ]
    
    MONOCLINIC_SG = [
        'C c', 'Cc', 'C 1 c 1', 'C c :1', 'C c :2'
    ]
    
    ALL_SPINEL_SG = (CUBIC_SPINEL_SG + CUBIC_ORDERED_SG + 
                     TETRAGONAL_SG + ORTHORHOMBIC_SG + MONOCLINIC_SG)
    
    @classmethod
    def is_applicable(cls, space_group, formula, metadata):
        """Check if structure is a spinel based on space group."""
        sg_normalized = space_group.strip()
        
        is_spinel = (
            sg_normalized in cls.CUBIC_SPINEL_SG or
            sg_normalized in cls.CUBIC_ORDERED_SG or
            sg_normalized in cls.TETRAGONAL_SG or
            sg_normalized in cls.ORTHORHOMBIC_SG or 
            sg_normalized in cls.MONOCLINIC_SG       
        )
        
        return is_spinel
    
    @classmethod
    def analyze(cls, cif_block, metadata, calculated_results, final_corrected_data=None, cif_path=None):
        """
        Analyze spinel structure and calculate structural parameters.
        
        NEW IN v1.4: If cif_path is provided, extracts additional structural
        parameters including u parameter, cell dimensions, etc.
        """
        
        space_group = metadata.get('space_group', '').strip()
        formula = metadata.get('chemical_formula', '')
        
        # Determine spinel type from space group
        if space_group in cls.CUBIC_SPINEL_SG:
            spinel_type = 'cubic'
        elif space_group in cls.CUBIC_ORDERED_SG:
            spinel_type = 'cubic_ordered'
        elif space_group in cls.TETRAGONAL_SG:
            spinel_type = 'tetragonal'
        elif space_group in cls.ORTHORHOMBIC_SG:
            spinel_type = 'orthorhombic'
        elif space_group in cls.MONOCLINIC_SG:
            spinel_type = 'monoclinic'
        else:
            return cls._null_result()
        
        # =====================================================================
        # NEW IN v1.4: Extract structural parameters from CIF
        # =====================================================================
        structural_params = {}
        if cif_path:
            try:
                # Import from Cell 3 helper functions
                structural_params = extract_all_structural_parameters(cif_path, formula)
            except NameError:
                # Function not available - continue without it
                pass
        
        # Extract site information from calculated_results
        sites = cls._extract_site_info(calculated_results, cif_block)
        
        if not sites:
            result = cls._null_result('No site data available for analysis')
            # Still add structural params if available
            if structural_params:
                result.update(cls._format_structural_params(structural_params))
            return result
        
        # Classify sites by coordination number
        tetrahedral_sites = [s for s in sites if s['cn'] == 4]
        octahedral_sites = [s for s in sites if s['cn'] == 6]
        
        if not tetrahedral_sites and not octahedral_sites:
            result = cls._null_result('No tetrahedral or octahedral sites found')
            if structural_params:
                result.update(cls._format_structural_params(structural_params))
            return result
        
        # Calculate inversion parameter and classify
        result = cls._calculate_inversion_parameter(
            tetrahedral_sites,
            octahedral_sites,
            formula,
            spinel_type
        )
        
        result['spinel_type'] = spinel_type
        
        # Add distortion metrics from CIF block
        if spinel_type in ['tetragonal', 'orthorhombic', 'monoclinic']:
            cell_params = {
                'cell_length_a': cif_block.get('_cell_length_a'),
                'cell_length_b': cif_block.get('_cell_length_b'),
                'cell_length_c': cif_block.get('_cell_length_c')
            }
            distortion_metrics = cls._calculate_distortion_metrics(cell_params, spinel_type)
            result.update(distortion_metrics)
        
        # =====================================================================
        # NEW IN v1.4: Add extracted structural parameters
        # =====================================================================
        if structural_params:
            result.update(cls._format_structural_params(structural_params))
        
        return result
    
    @classmethod
    def _format_structural_params(cls, params):
        """
        Format structural parameters for output.
        
        v1.7.3: Reinstated anion_y, anion_z (independent in non-cubic spinels).
                Added u_eff, delta_u, anion_site_count, second anion site columns.
        """
        u_param = params.get('u_parameter')
        u_raw = params.get('u_parameter_raw')
        # Only emit u_parameter_raw when it actually differs (flags non-standard origin)
        if u_raw is not None and u_param is not None and abs(u_raw - u_param) < 1e-5:
            u_raw = None
 
        return {
            'u_parameter': u_param,
            'u_parameter_raw': u_raw,
            'u_deviation': params.get('u_deviation'),
            'origin_choice': params.get('origin_choice'),
            'anion_x': params.get('anion_x'),
            'anion_y': params.get('anion_y'),
            'anion_z': params.get('anion_z'),
            'anion_wyckoff': params.get('anion_wyckoff'),
            'anion_multiplicity': params.get('anion_multiplicity'),
            'anion_wyckoff_letter': params.get('anion_wyckoff_letter'),
            # v1.7.3: Generalised anion parameters
            'u_eff': params.get('u_eff'),
            'delta_u': params.get('delta_u'),
            'anion_site_count': params.get('anion_site_count'),
            'anion_x_2': params.get('anion_x_2'),
            'anion_y_2': params.get('anion_y_2'),
            'anion_z_2': params.get('anion_z_2'),
            'anion_wyckoff_2': params.get('anion_wyckoff_2'),
            # Cell parameters (unchanged)
            'cell_a': params.get('cell_a'),
            'cell_b': params.get('cell_b'),
            'cell_c': params.get('cell_c'),
            'cell_volume': params.get('cell_volume'),
        }
    
    @classmethod
    def _extract_site_info(cls, calculated_results, cif_block=None):
        """Extract relevant site information from calculated_results."""
        sites = []
        
        for coords, site_data in calculated_results.items():
            elements = site_data.get('elements', [])
            labels = site_data.get('labels', [])
            
            for element in elements:
                elem_data = site_data['element_data'].get(element, {})
                
                occupancy = 1.0
                if cif_block is not None and labels:
                    for label in labels:
                        try:
                            occ_info = extract_site_occupancy_info(cif_block, label)
                            if occ_info:
                                for entry in occ_info:
                                    if entry['element'] == element:
                                        occupancy = entry['occupancy']
                                        break
                        except:
                            pass
                        
                        if occupancy != 1.0:
                            break
                    
                    if occupancy == 1.0 and labels:
                        import re
                        match = re.search(r'(\d+)$', labels[0])
                        if match:
                            suffix = match.group(1)
                            constructed_label = f"{element}{suffix}"
                            try:
                                occ_info = extract_site_occupancy_info(cif_block, constructed_label)
                                if occ_info:
                                    for entry in occ_info:
                                        if entry['element'] == element:
                                            occupancy = entry['occupancy']
                                            break
                            except:
                                pass
                
                site_info = {
                    'element': element,
                    'cn': elem_data.get('cn'),
                    'bvs': elem_data.get('bvs'),
                    'valence': elem_data.get('valence'),
                    'coords': coords,
                    'occupancy': occupancy
                }
                
                if site_info['cn'] is not None:
                    sites.append(site_info)
        
        return sites
    
    @classmethod
    def _calculate_inversion_parameter(cls, tetrahedral_sites, octahedral_sites,
                                      formula, spinel_type):
        """
        Calculate inversion parameter lambda from site occupancies.

        v1.6 change: inversion is only meaningful for stoichiometric AB2X4 spinels.
        Non-stoichiometric compositions (Li-excess spinels like Li4Mn5O12 / Li4Ti5O12,
        oxygen-deficient spinels, etc.) cannot be described by a single scalar
        inversion parameter — site occupancies there are dictated by composition,
        not by A-B cation swap. For those cases we emit a distinct classification
        and leave lambda undefined, reporting the B-site small-cation fraction
        as a separate quantity.
        """

        Td_elements = [s['element'] for s in tetrahedral_sites]
        Oh_elements = [s['element'] for s in octahedral_sites]

        all_elements = set(Td_elements + Oh_elements)
        cation_elements = [e for e in all_elements
                          if e not in ['O', 'F', 'S', 'Cl', 'Br', 'I', 'Se', 'Te', 'N']]

        if not cation_elements:
            return cls._null_result('No cations found in structure')

        A_site_elements = list(set(Td_elements))
        B_site_elements = list(set(Oh_elements))

        is_binary = len(cation_elements) > 2

        # ================================================================
        # v1.6: Stoichiometry pre-check. Inversion is only defined for AB2X4.
        # If ANY element has a fractional count, it's non-stoichiometric.
        # ================================================================
        is_stoichiometric_AB2X4 = True

        if formula:
            try:
                parsed = parse_chemical_formula(formula)
                for elem, count in parsed.items():
                    if abs(count - round(count)) > 0.05:
                        is_stoichiometric_AB2X4 = False
                        break
            except Exception:
                pass

        # Non-stoichiometric: report site distribution without computing lambda
        if formula and not is_stoichiometric_AB2X4:
            small_cations = cls._identify_small_cations(tetrahedral_sites + octahedral_sites)
            small_cation = list(small_cations)[0] if small_cations else None

            B_small_fraction = None
            if small_cation:
                oh_occ = [s['occupancy'] for s in octahedral_sites
                         if s['element'] == small_cation]
                if oh_occ:
                    B_small_fraction = oh_occ[0]

            if B_small_fraction is not None and B_small_fraction > 0.01:
                classification = 'normal_nonstoichiometric'
                notes = (f'Non-stoichiometric spinel (cation:anion != 3:4); '
                        f'{small_cation} occupies {B_small_fraction:.3f} of B-sites '
                        f'by composition, not inversion')
            else:
                classification = 'normal_nonstoichiometric'
                notes = 'Non-stoichiometric spinel (cation:anion != 3:4)'

            B_site_ordering = (spinel_type == 'cubic_ordered' and len(B_site_elements) > 1)

            return {
                'inversion_parameter': None,
                'spinel_classification': classification,
                'A_site_elements': ','.join(sorted(A_site_elements)) if A_site_elements else None,
                'B_site_elements': ','.join(sorted(B_site_elements)) if B_site_elements else None,
                'B_site_ordering': B_site_ordering,
                'B_site_small_cation_fraction': B_small_fraction,
                'analysis_confidence': 'high',
                'analysis_notes': notes,
            }

        # ================================================================
        # Stoichiometric AB2X4 — original logic applies
        # ================================================================
        if not tetrahedral_sites:
            lambda_val = 1.0
            classification = 'inverse'
            confidence = 'high'
            notes = 'All cations on octahedral sites'

        elif not octahedral_sites:
            lambda_val = None
            classification = 'unusual'
            confidence = 'low'
            notes = 'All cations on tetrahedral sites - verify structure type'

        else:
            small_cations = cls._identify_small_cations(tetrahedral_sites + octahedral_sites)

            small_on_Td = any(s['element'] in small_cations for s in tetrahedral_sites)
            small_on_Oh = any(s['element'] in small_cations for s in octahedral_sites)

            if small_on_Td and not small_on_Oh:
                lambda_val = 0.0
                classification = 'normal_binary' if is_binary else 'normal'
                confidence = 'high'
                notes = 'Small cations exclusively on tetrahedral sites'

            elif small_on_Oh and not small_on_Td:
                lambda_val = 1.0
                classification = 'inverse'
                confidence = 'high'
                notes = 'Small cations exclusively on octahedral sites'

            elif small_on_Td and small_on_Oh:
                small_cation = list(small_cations)[0] if small_cations else None

                if small_cation:
                    oh_occupancies = [s['occupancy'] for s in octahedral_sites
                                     if s['element'] == small_cation]

                    if oh_occupancies:
                        lambda_val = 2.0 * oh_occupancies[0]

                        if lambda_val < 0.1:
                            classification = 'normal'
                            confidence = 'high'
                        elif lambda_val > 0.9:
                            classification = 'inverse'
                            confidence = 'high'
                        else:
                            classification = 'random'
                            confidence = 'high'

                        notes = f'{small_cation} occupancy at octahedral sites: {oh_occupancies[0]:.3f}'
                    else:
                        lambda_val = 0.5
                        classification = 'random'
                        confidence = 'low'
                        notes = 'Could not determine occupancies - assumed random'
                else:
                    lambda_val = 0.5
                    classification = 'random'
                    confidence = 'low'
                    notes = 'Could not identify small cation'

            else:
                lambda_val = None
                classification = 'unknown'
                confidence = 'low'
                notes = 'Cannot identify cation distribution pattern'

        B_site_ordering = (spinel_type == 'cubic_ordered' and len(B_site_elements) > 1)

        return {
            'inversion_parameter': lambda_val,
            'spinel_classification': classification,
            'A_site_elements': ','.join(sorted(A_site_elements)) if A_site_elements else None,
            'B_site_elements': ','.join(sorted(B_site_elements)) if B_site_elements else None,
            'B_site_ordering': B_site_ordering,
            'B_site_small_cation_fraction': None,
            'analysis_confidence': confidence,
            'analysis_notes': notes
        }
    
    @classmethod
    def _identify_small_cations(cls, sites):
        """Identify 'small' cations that typically occupy A-sites in normal spinels."""
        
        A_site_priority = {
            'Li': 1,
            'Mg': 2,
            'Zn': 2,
            'Mn': 3,
            'Fe': 3,
            'Co': 3,
            'Ni': 3,
            'Cu': 3,
        }
        
        valences = [abs(s['valence']) for s in sites if s['valence'] is not None]
        if not valences:
            return []
        
        min_valence = min(valences)
        
        min_valence_cations = [
            s['element'] for s in sites 
            if s['valence'] is not None and abs(s['valence']) == min_valence
        ]
        
        if len(set(min_valence_cations)) > 1:
            candidates = list(set(min_valence_cations))
            candidates.sort(key=lambda e: A_site_priority.get(e, 999))
            small_cations = [candidates[0]]
        else:
            small_cations = list(set(min_valence_cations))
        
        Td_elements = set(s['element'] for s in sites if s['cn'] == 4)
        Oh_elements = set(s['element'] for s in sites if s['cn'] == 6)
        
        actual_small_cations = []
        for cation in small_cations:
            if cation in Td_elements:
                actual_small_cations.append(cation)
            elif cation in Oh_elements and len(small_cations) > 1:
                continue
            else:
                actual_small_cations.append(cation)
        
        return actual_small_cations if actual_small_cations else small_cations

    @classmethod
    def _calculate_distortion_metrics(cls, cell_params, spinel_type):
        """Calculate structural distortion from ideal cubic symmetry."""
        if spinel_type not in ['tetragonal', 'orthorhombic', 'monoclinic']:
            return {}
        
        a = cell_params.get('cell_length_a')
        b = cell_params.get('cell_length_b')
        c = cell_params.get('cell_length_c')
        
        if not all([a, b, c]):
            return {}
        
        try:
            a_clean = str(a).split('(')[0] if '(' in str(a) else str(a)
            b_clean = str(b).split('(')[0] if '(' in str(b) else str(b)
            c_clean = str(c).split('(')[0] if '(' in str(c) else str(c)
            
            a, b, c = float(a_clean), float(b_clean), float(c_clean)
        except (ValueError, TypeError, AttributeError):
            return {}
        
        if spinel_type == 'tetragonal':
            c_over_a = c / a
            ideal_c_over_a = 1.414
            tetragonality = abs(c_over_a - ideal_c_over_a)
            
            return {
                'c_over_a': round(c_over_a, 4),
                'tetragonality': round(tetragonality, 4),
                'distortion_type': 'elongated' if c_over_a > ideal_c_over_a else 'compressed'
            }
        
        elif spinel_type in ['orthorhombic', 'monoclinic']:
            mean_cell = (a + b + c) / 3
            max_deviation = max(abs(a - mean_cell), abs(b - mean_cell), abs(c - mean_cell))
            orthorhombicity = max_deviation / mean_cell
            
            return {
                'cell_a': round(a, 4),
                'cell_b': round(b, 4),
                'cell_c': round(c, 4),
                'orthorhombicity': round(orthorhombicity, 4),
                'max_cell_deviation': round(max_deviation, 4)
            }
        
        return {}
    
    @classmethod
    def _null_result(cls, notes='Not a spinel structure'):
        """Return null result when analysis fails or is inapplicable."""
        return {
            'spinel_type': None,
            'inversion_parameter': None,
            'spinel_classification': None,
            'A_site_elements': None,
            'B_site_elements': None,
            'B_site_ordering': None,
            'analysis_confidence': None,
            'analysis_notes': notes,
            # Distortion metrics
            'c_over_a': None,
            'tetragonality': None,
            'distortion_type': None,
            'cell_a': None,
            'cell_b': None,
            'cell_c': None,
            'orthorhombicity': None,
            'max_cell_deviation': None,
            # Structural parameters (v1.7.3: generalised anion framework)
            'u_parameter': None,
            'u_parameter_raw': None,
            'u_deviation': None,
            'origin_choice': None,
            'anion_x': None,
            'anion_y': None,
            'anion_z': None,
            'anion_wyckoff': None,
            'anion_multiplicity': None,
            'anion_wyckoff_letter': None,
            'u_eff': None,
            'delta_u': None,
            'anion_site_count': None,
            'anion_x_2': None,
            'anion_y_2': None,
            'anion_z_2': None,
            'anion_wyckoff_2': None,
            'cell_volume': None,
            'B_site_small_cation_fraction': None,
 
        }
    
    @classmethod
    def get_output_columns(cls):
        """
        Column names added to CSV output.
        
        v1.6: Removed redundant columns (cell_*_extracted, anion_y, anion_z).
              Added B_site_small_cation_fraction, cell_volume.
        """
        return [
            # Core spinel analysis
            'inversion_parameter',
            'spinel_classification',
            'A_site_elements',
            'B_site_elements',
            'B_site_ordering',
            'B_site_small_cation_fraction',
            'analysis_confidence',
            'analysis_notes',
            'spinel_type',
            
            # Distortion metrics
            'c_over_a',
            'tetragonality',
            'distortion_type',
            'cell_a',
            'cell_b',
            'cell_c',
            'cell_volume',
            'orthorhombicity',
            'max_cell_deviation',
            # Structural parameters (v1.7.3: generalised anion framework)
            'u_parameter',              # Cubic Fd-3m only: standardised to origin 2
            'u_parameter_raw',          # Only populated when differs from u_parameter
            'u_deviation',              # From ideal (0.25)
            'origin_choice',            # 1 or 2 (detected, cubic only)
            'anion_x',                  # Primary anion site x
            'anion_y',                  # Primary anion site y
            'anion_z',                  # Primary anion site z
            'anion_wyckoff',            # Full symbol e.g. "32e", "16h", "24e"
            'anion_multiplicity',       # Just the number
            'anion_wyckoff_letter',     # Just the letter
            'u_eff',                    # Effective anion parameter (all space groups)
            'delta_u',                  # Anion sublattice anisotropy
            'anion_site_count',         # Number of anion Wyckoff sites (1 or 2)
            'anion_x_2',               # Second anion site x (P4332)
            'anion_y_2',               # Second anion site y
            'anion_z_2',               # Second anion site z
            'anion_wyckoff_2',          # Second anion Wyckoff symbol
        ]


# ============================================================================
# REGISTRY
# ============================================================================

class StructureAnalyzerRegistry:
    """Global registry for structure analyzers."""
    
    def __init__(self):
        self.analyzers = []
        
    def register(self, analyzer_class):
        """Register a structure analyzer."""
        self.analyzers.append(analyzer_class)
        print(f"  Registered {analyzer_class.name} analyzer")
    
    def find_analyzer(self, space_group, formula, metadata):
        """Find applicable analyzer for a structure."""
        for analyzer in self.analyzers:
            if analyzer.is_applicable(space_group, formula, metadata):
                return analyzer
        return None
    
    def get_all_columns(self):
        """Get all possible output columns from all analyzers."""
        columns = []
        for analyzer in self.analyzers:
            columns.extend(analyzer.get_output_columns())
        return columns


# ============================================================================
# INITIALIZE REGISTRY
# ============================================================================

structure_registry = StructureAnalyzerRegistry()

print(f"\n{'='*80}")
print(f"STRUCTURE ANALYZERS v1.6")
print(f"{'='*80}")

if ENABLED_ANALYZERS.get('spinel', False):
    structure_registry.register(SpinelAnalyzer)

if ENABLED_ANALYZERS.get('perovskite', False):
    print("  Warning: Perovskite analyzer not yet implemented")

if ENABLED_ANALYZERS.get('pyrochlore', False):
    print("  Warning: Pyrochlore analyzer not yet implemented")

if ENABLED_ANALYZERS.get('garnet', False):
    print("  Warning: Garnet analyzer not yet implemented")

if len(structure_registry.analyzers) == 0:
    print("  Warning: No analyzers enabled - structure-type analysis disabled")
else:
    print(f"\n  Active analyzers: {len(structure_registry.analyzers)}")
    print(f"  Additional CSV columns: {len(structure_registry.get_all_columns())}")
    # Show first few columns
    cols = structure_registry.get_all_columns()
    if len(cols) > 10:
        print(f"  Columns: {', '.join(cols[:10])}...")
    else:
        print(f"  Columns: {', '.join(cols)}")

print(f"{'='*80}\n")

In [ ]:
## CELL 4: MAIN EIR FUNCTIONALITY
# ============================================================================
# Run this cell BEFORE Cell 5

"""
Eir Batch Processor v1.7 - Automated Bond Valence Sum Analysis
================================================================

NEW IN v1.7:
- Data integrity tier system: ACCEPT/FLAG/EXPUNGE classification per site and structure
- EXPUNGE (I6): Detects implausible BVS values (>10 v.u. or <0.2 v.u. for valence >1)
- EXPUNGE (I3): Detects minority species (<5% occupancy on majority-occupied sites)
- FLAG: Detects mixed TM occupancy with non-integer valence partitioning
- New output columns: integrity_tier, flag_reason, occupancy, is_minority_species,
  is_mixed_occupancy_tm, cif_valence, formal_valence_from_symbol, symbol_number_agreement,
  structure_tier, n_sites_flagged, n_sites_expunged
- Fixed version banner (was stuck at "v1.5")

PREVIOUS (v1.5/v1.6.1):

NEW IN v1.4:
- CIF oxidation state extraction (automatic from _atom_type_oxidation_number)
- Type 7 error flags for mixed valence detection
- Anion parameter (u) extraction for spinels
- Enhanced structural parameter extraction
- Priority: config.json > CIF explicit > CIF inferred > interactive prompt

NEW IN v1.3:
- Improved handling of framework formers (Ge, Si, B, P)
- Parameter-based bond selection replaces hardcoded metal lists
- Auto-detects ALL cation-anion pairs with available BVS parameters
- Future-proof against new elements/parameter additions

NEW IN v1.2:
- Multi-anion support: Handles O, F, Cl, Br, I, S, Se, Te, etc.
- Auto-detects M-X bonds (not just M-O)
- Merged BVS parameters (Gagne & Hawthorne 2015 + Brown 2020)
- 1,385 M-X parameter pairs vs 168 M-O pairs (8x expansion)
- Temperature/Pressure extraction from CIF files
- Automatic classification of measurement conditions

NEW IN v1.1:
- Config file support (config.json) for unattended batch processing
- Automatic 3D visualisation of problematic sites (|deviation| > 0.3 v.u.)
- Adaptive gap detection for Jahn-Teller active ions
- Enhanced output with deviation column

OUTPUT:
All results saved to ./outputs/ subfolder:
- eir_batch_results_v1.7_YYYYMMDD_HHMMSS.csv
- oxidation_states_used_YYYYMMDD_HHMMSS.json
- eir_batch_failed_v1.7_YYYYMMDD_HHMMSS.csv (if any failures)

CITATIONS:
Gagné & Hawthorne (2015) Acta Cryst. B71, 561-578
Brown (2020) IUCr Bond Valence Parameters
Brown & Altermatt (1985) Acta Cryst. B41, 244-247
Brese & O'Keeffe (1991) Acta Cryst. B47, 192-197
"""

import os
import sys
import json
import re
import math
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# CIF parsing
try:
    from CifFile import ReadCif
except ImportError:
    print("ERROR: PyCifRW not found. Install with: pip install pycifrw")
    sys.exit(1)

# pymatgen for ICSD CIF bond calculation (robust parser + proper crystallography)
try:
    from pymatgen.core import Structure
    from pymatgen.io.cif import CifParser
    PYMATGEN_AVAILABLE = True
    print("✓ pymatgen available - ICSD CIF files supported with proper crystallographic symmetry")
except ImportError:
    PYMATGEN_AVAILABLE = False
    print("⚠ pymatgen not available - ICSD CIFs will fail. Install with: pip install pymatgen")
    print("  pymatgen is REQUIRED for proper handling of:")
    print("   - Partial occupancy (e.g., Na2/3 sites)")
    print("   - Mixed occupancy (e.g., Cr0.8Ti0.2)")
    print("   - All space groups")
    print("   - Robust CIF parsing (handles ICSD formatting quirks)")

# py3Dmol for 3D visualisation
try:
    import py3Dmol
    from IPython.display import display
    PY3DMOL_AVAILABLE = True
    print("✓ py3Dmol available - 3D visualisation enabled for problematic sites")
except ImportError as e:
    PY3DMOL_AVAILABLE = False
    print("⚠ py3Dmol not available - Install with: pip install py3Dmol")
    print(f"   Import error: {e}")
except Exception as e:
    PY3DMOL_AVAILABLE = False
    print("⚠ py3Dmol import failed - Unexpected error")
    print(f"   Error: {e}")

# ============================================================================
# DATABASE MANAGER (From Skuld 1.5 - UNCHANGED)
# ============================================================================

class DatabaseManager:
    """Manages external JSON databases for ionic radii and bond valence parameters"""
    
    def __init__(self, data_dir="./Data"):
        self.data_dir = data_dir
        self.shannon_radii = None
        self.bond_valence_params = None
        self.oxidation_states = None
        self.element_categories = None
        
        if not os.path.exists(self.data_dir):
            print(f"Warning: Data directory '{self.data_dir}' not found!")
            print("Please ensure your JSON database files are in the ./Data folder")
    
    def load_shannon_radii(self):
        """Load Shannon-Prewitt ionic radii from JSON file"""
        if self.shannon_radii is not None:
            return self.shannon_radii
            
        filepath = os.path.join(self.data_dir, "shannon_radii.json")
        
        if not os.path.exists(filepath):
            print(f"Shannon radii file not found at {filepath}")
            return None
        
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
                self.shannon_radii = {}
                for element, valence_dict in data.items():
                    for valence_str, cn_dict in valence_dict.items():
                        valence = int(valence_str)
                        for cn_str, radius in cn_dict.items():
                            cn = int(cn_str)
                            self.shannon_radii[(element, valence, cn)] = radius
                
            print(f"✓ Loaded Shannon radii database with {len(self.shannon_radii)} entries")
        except Exception as e:
            print(f"✗ Error loading Shannon radii: {e}")
            return None
        
        return self.shannon_radii
    
    def load_bond_valence_params(self):
        """Load Gagné & Hawthorne bond valence parameters from JSON file"""
        if self.bond_valence_params is not None:
            return self.bond_valence_params
            
        filepath = os.path.join(self.data_dir, "bvs_parameters.json")
        
        if not os.path.exists(filepath):
            print(f"BVS parameters file not found at {filepath}")
            return None
        
        try:
            import ast
            with open(filepath, 'r') as f:
                data = json.load(f)
                self.bond_valence_params = {}
                for key, params in data.items():
                    # Keys are string representations of tuples: "('Na', 1, 'O')"
                    try:
                        cation, cation_valence, anion = ast.literal_eval(key)
                        self.bond_valence_params[(cation, cation_valence, anion)] = {
                            'R0': params['r0'],  # Note lowercase in JSON
                            'B': params.get('b', 0.37)  # Note lowercase in JSON
                        }
                    except (ValueError, SyntaxError) as e:
                        # Skip malformed keys
                        continue
            
            print(f"✓ Loaded BVS parameters database with {len(self.bond_valence_params)} entries")
        except Exception as e:
            print(f"✗ Error loading BVS parameters: {e}")
            import traceback
            traceback.print_exc()
            return None
        
        return self.bond_valence_params
    
    def load_common_oxidation_states(self):
        """Load common oxidation states from JSON file"""
        if self.oxidation_states is not None:
            return self.oxidation_states
            
        filepath = os.path.join(self.data_dir, "common_oxidation_states.json")
        
        if not os.path.exists(filepath):
            print(f"Oxidation states file not found at {filepath}")
            return None
        
        try:
            with open(filepath, 'r') as f:
                self.oxidation_states = json.load(f)
            
            print(f"✓ Loaded oxidation states database with {len(self.oxidation_states)} elements")
        except Exception as e:
            print(f"✗ Error loading oxidation states: {e}")
            return None
        
        return self.oxidation_states
    
    def load_element_categories(self):
        """Load element categories from JSON file"""
        if self.element_categories is not None:
            return self.element_categories
            
        filepath = os.path.join(self.data_dir, "element_categories.json")
        
        if not os.path.exists(filepath):
            print(f"Element categories file not found at {filepath}")
            return None
        
        try:
            with open(filepath, 'r') as f:
                self.element_categories = json.load(f)
            
            print(f"✓ Loaded element categories database")
        except Exception as e:
            print(f"✗ Error loading element categories: {e}")
            return None
        
        return self.element_categories
    
    def get_ionic_radius(self, element, valence, cn):
        """Get ionic radius for given element, valence, and coordination number"""
        if self.shannon_radii is None:
            self.load_shannon_radii()
        
        if self.shannon_radii is None:
            return None
        
        return self.shannon_radii.get((element, valence, cn))
    
    def get_bv_parameters(self, cation, cation_valence, anion):
        """Get R0 and B parameters for BVS calculation"""
        if self.bond_valence_params is None:
            self.load_bond_valence_params()
        
        if self.bond_valence_params is None:
            return None, None
        
        params = self.bond_valence_params.get((cation, cation_valence, anion))
        if params:
            return params['R0'], params['B']
        return None, None
    
    def get_common_oxidation_states(self, element):
        """Get list of common oxidation states for an element"""
        if self.oxidation_states is None:
            self.load_common_oxidation_states()
        
        if self.oxidation_states is None:
            return []
        
        return self.oxidation_states.get(element, [])
    
    def get_element_category(self, element):
        """Get category for an element (alkali, transition, etc.)"""
        if self.element_categories is None:
            self.load_element_categories()
        
        if self.element_categories is None:
            return "unknown"
        
        for category, elements in self.element_categories.items():
            if element in elements:
                return category
        return "other"

# Initialise global database manager
db_manager = DatabaseManager()
db_manager.load_shannon_radii()
db_manager.load_bond_valence_params()
db_manager.load_common_oxidation_states()
db_manager.load_element_categories()

# ============================================================================
# BVS PARAMETER LOOKUP WITH FRACTIONAL VALENCE SUPPORT (NEW IN v1.6)
# ============================================================================

def get_bv_parameters_fractional(cation, cation_valence, anion):
    """
    Look up BVS parameters (R0, b) for a cation, supporting non-integer valences
    via linear interpolation between adjacent integer states.

    For integer valences this is equivalent to db_manager.get_bv_parameters().
    For fractional valences (e.g. Mn 3.5 in LiMn2O4), parameters are interpolated
    linearly between floor(v) and ceil(v). This is the most defensible treatment
    when the underlying valence is a charge-averaged mean on the diffraction
    timescale — which is the usual assumption for electron-hopping systems at
    ambient temperature.

    v1.6 NEW: enables BVS analysis of mixed-valence cathode materials (LiMn2O4,
    LiNi0.5Mn1.5O4, NCA/NMC, etc.) without the user having to pre-assign a
    nominal integer valence.

    Args:
        cation: Element symbol
        cation_valence: Formal valence (int or float)
        anion: Anion element symbol

    Returns:
        tuple (R0, b) or (None, None) if parameters cannot be determined.
    """
    if cation_valence is None:
        return None, None

    # Integer valence (or float-encoded integer): direct lookup
    if float(cation_valence) == int(cation_valence):
        return db_manager.get_bv_parameters(cation, int(cation_valence), anion)

    # Fractional valence: interpolate
    v_lower = int(math.floor(cation_valence))
    v_upper = int(math.ceil(cation_valence))
    fraction = cation_valence - v_lower

    R0_lower, b_lower = db_manager.get_bv_parameters(cation, v_lower, anion)
    R0_upper, b_upper = db_manager.get_bv_parameters(cation, v_upper, anion)

    if R0_lower is not None and R0_upper is not None:
        R0 = (1.0 - fraction) * R0_lower + fraction * R0_upper
        b  = (1.0 - fraction) * b_lower  + fraction * b_upper
        return R0, b

    # Only one neighbour available — fall back to whichever exists
    if R0_lower is not None:
        return R0_lower, b_lower
    if R0_upper is not None:
        return R0_upper, b_upper

    return None, None

# ============================================================================
# UTILITY FUNCTIONS (From Skuld 1.5 - UNCHANGED)
# ============================================================================

def parse_value_with_error(value_string):
    """
    Parse CIF value strings that may contain errors in parentheses.
    Example: "1.2345(67)" -> 1.2345, 0.0067
    """
    if not value_string or value_string == '.' or value_string == '?':
        return None, None
    
    match = re.match(r'^([+-]?\d+\.?\d*(?:[eE][+-]?\d+)?)\((\d+)\)$', value_string.strip())
    if match:
        value_str, error_str = match.groups()
        value = float(value_str)
        
        if '.' in value_str:
            decimal_places = len(value_str.split('.')[1].split('e')[0].split('E')[0])
        else:
            decimal_places = 0
        
        error = float(error_str) * (10 ** (-decimal_places))
        return value, error
    else:
        try:
            value = float(value_string.strip())
            return value, None
        except ValueError:
            return None, None

def validate_element_combination(cation, anion):
    """Check if cation-anion combination is chemically reasonable"""
    cation_category = db_manager.get_element_category(cation)
    anion_category = db_manager.get_element_category(anion)
    
    if cation_category == "other" and anion_category == "other":
        return False
    
    return True

# ============================================================================
# BOND DISTANCE CALCULATION FOR ICSD CIF FILES
# ============================================================================

def clean_cif_for_gemmi(cif_file_path):
    """
    Pre-process CIF file to remove lines that cause gemmi parsing errors.
    
    Removes _chemical_name_structure_type which often has UTF-8 encoding issues in ICSD files.
    Replaces non-standard reverse rhombohedral symops ('HR' suffix) with standard obverse
    centering, which gemmi requires for space group determination.
    
    Returns cleaned CIF as string for gemmi to parse.
    """
    try:
        import re
        
        # Read file with UTF-8 encoding, replacing errors
        with open(cif_file_path, 'r', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
        
        content = ''.join(lines)
        
        # Fix non-standard reverse rhombohedral centering ('HR' suffix)
        # Some older ICSD CIFs encode symops in the reverse setting, which gemmi
        # cannot match to a tabulated space group. Replace the HM label and
        # regenerate symops from the IT number using standard obverse centering.
        if ' HR' in content:
            sg_tags = ['_space_group_name_H-M_alt', '_symmetry_space_group_name_H-M']
            has_hr = any(tag in content and ' HR' in content for tag in sg_tags)
            
            if has_hr:
                try:
                    import gemmi as _gemmi
                    
                    # Fix HM label
                    content = content.replace(" HR'", " H'")
                    content = content.replace(' HR\r', ' H\r')
                    content = content.replace(' HR\n', ' H\n')
                    
                    # Get IT number and regenerate standard symops
                    it_match = re.search(r'_space_group_IT_number\s+(\d+)', content)
                    if not it_match:
                        it_match = re.search(r'_symmetry_Int_Tables_number\s+(\d+)', content)
                    
                    if it_match:
                        sg = _gemmi.find_spacegroup_by_number(int(it_match.group(1)))
                        if sg:
                            ops = sg.operations()
                            new_loop = 'loop_\n_space_group_symop_id\n_space_group_symop_operation_xyz\n'
                            for i, op in enumerate(ops):
                                new_loop += f"{i+1} '{op.triplet()}'\n"
                            
                            # Replace old symops loop
                            pattern = r"loop_\s*\n_space_group_symop_id\s*\n_space_group_symop_operation_xyz\s*\n((?:\d+\s+'[^']+'\s*[\r\n]+)+)"
                            content = re.sub(pattern, new_loop, content)
                            print(f"  Note: Replaced reverse rhombohedral symops with standard obverse (SG {sg.hm})")
                except Exception as e:
                    print(f"  Warning: Could not fix HR symops: {e}")
        
        # Re-split into lines for structure_type removal
        lines = content.split('\n')
        
        cleaned_lines = []
        i = 0
        while i < len(lines):
            line = lines[i]
            
            # Check if this line contains the problematic tag
            if '_chemical_name_structure_type' in line:
                # Skip this line
                # Also skip the next line if it doesn't start with _ or # (it's the value)
                i += 1
                if i < len(lines):
                    next_line = lines[i].strip()
                    # If next line is a value (not a tag), skip it too
                    if next_line and not next_line.startswith('_') and not next_line.startswith('#') and not next_line.startswith('loop_') and not next_line.startswith('data_'):
                        i += 1
                continue
            
            cleaned_lines.append(line)
            i += 1
        
        cleaned = '\n'.join(cleaned_lines)
        return cleaned
    
    except Exception as e:
        print(f"  Warning: Error cleaning CIF file: {e}")
        return None


def calculate_bonds_from_positions(cif_file_path, max_bond_distance=3.5):
    """
    Calculate bond distances from CIF using proper crystallographic symmetry operations.
    
    Uses gemmi library for robust handling of:
    - Partial occupancy (e.g., Na site 2/3 occupied)  
    - Mixed occupancy (e.g., Cr0.8Ti0.2 on same site)
    - All space groups (R-3m, P63/mmc, C2/m, etc.)
    - Proper symmetry code generation
    
    CRITICAL: ICSD files are small molecule/inorganic structures, not macromolecular.
    Uses SmallStructure, not Structure.
    
    Args:
        cif_file_path: Path to CIF file
        max_bond_distance: Maximum bond distance to consider (Ångströms)
    
    Returns:
        dict with bond data formatted like GSAS-II _geom_bond output, or None if failed
    """
    try:
        import gemmi
    except ImportError:
        print("  ✗ gemmi not available - install with: pip install gemmi")
        print("    gemmi is required for proper crystallographic bond calculation")
        return None
    
    import tempfile
    
    try:
        # Read file with UTF-8 error handling
        cleaned_cif = clean_cif_for_gemmi(cif_file_path)
        if cleaned_cif is None:
            print("  ✗ Could not read CIF file")
            return None
        
        # Write to temporary file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.cif', delete=False, encoding='utf-8') as tmp:
            tmp.write(cleaned_cif)
            tmp_path = tmp.name
        
        try:
            # Read CIF document
            doc = gemmi.cif.read(tmp_path)
            block = doc.sole_block()
            
            # CRITICAL: Use SmallStructure for ICSD (small molecule/inorganic) CIFs
            # NOT Structure (macromolecular)
            small_struct = gemmi.make_small_structure_from_block(block)
            
        finally:
            # Clean up temp file
            import os
            try:
                os.unlink(tmp_path)
            except:
                pass
        
        if not small_struct or len(small_struct.sites) == 0:
            print("  ✗ No atomic sites found in CIF")
            return None
        
        # Build bond data in GSAS-II format
        bond_data = {
            'bond_label_1': [],
            'bond_label_2': [],
            'bond_distance': [],
            'bond_symmetry_2': []
        }
        
        bond_count = 0
        atom_type_pairs = set()
        
        # Debug: Print all sites found
        print(f"  Debug: Found {len(small_struct.sites)} sites in asymmetric unit:")
        for site in small_struct.sites:
            frac = site.fract
            print(f"    {site.label}: {site.element.name} at ({frac.x:.4f}, {frac.y:.4f}, {frac.z:.4f}) occ={site.occ:.2f}")
        
        # CRITICAL: Generate full supercell for proper coordination detection
        # Gemmi's neighbor search works with asymmetric units, but we need all atoms
        # Generate 3x3x3 supercell to ensure we find all neighbors within cutoff
        print(f"  Generating 3×3×3 supercell for coordination analysis...")
        
        supercell_atoms = []
        
        # For each site in asymmetric unit
        for site in small_struct.sites:
            elem_type = site.element.name
            site_label = site.label
            
            # Apply all symmetry operations
            for op in small_struct.spacegroup.operations():
                # Transform fractional coordinates
                transformed_frac = op.apply_to_xyz([site.fract.x, site.fract.y, site.fract.z])
                
                # Apply lattice translations for 3x3x3 supercell
                for i in range(-1, 2):  # -1, 0, 1
                    for j in range(-1, 2):
                        for k in range(-1, 2):
                            # Add lattice translation
                            final_frac = [
                                transformed_frac[0] + i,
                                transformed_frac[1] + j,
                                transformed_frac[2] + k
                            ]
                            
                            # Convert to Cartesian
                            cart_pos = small_struct.cell.orthogonalize(gemmi.Fractional(*final_frac))
                            
                            supercell_atoms.append({
                                'label': site_label,
                                'element': elem_type,
                                'position': cart_pos,
                                'frac': final_frac,
                                'original_site': site
                            })
        
        print(f"  Generated {len(supercell_atoms)} atoms in supercell")
        
        # Now find neighbors by simple distance calculation
        for site in small_struct.sites:
            site_orth = site.orth(small_struct.cell)
            elem1 = site.label
            elem1_type = site.element.name
            
            neighbors_for_site = []
            
            # Check distance to every atom in supercell
            for atom_dict in supercell_atoms:
                distance = atom_dict['position'].dist(site_orth)
                
                # Skip self and atoms too close/far
                if distance < 0.01 or distance > max_bond_distance:
                    continue
                
                neighbors_for_site.append({
                    'label': atom_dict['label'],
                    'element': atom_dict['element'],
                    'position': atom_dict['position'],
                    'distance': distance
                })
            
            # Deduplicate by position with 0.05Å tolerance  
            # Looser tolerance handles general positions in R-3m better
            unique_neighbors = []
            seen_positions = set()
            
            for neighbor in neighbors_for_site:
                # Round to 0.05Å (50 pm) - catches numerical precision while keeping distinct atoms
                pos_key = (
                    round(neighbor['position'].x / 0.05) * 0.05,
                    round(neighbor['position'].y / 0.05) * 0.05,
                    round(neighbor['position'].z / 0.05) * 0.05
                )
                if pos_key not in seen_positions:
                    seen_positions.add(pos_key)
                    unique_neighbors.append(neighbor)
            
            print(f"  Debug: Site {site.label} ({site.element.name}) has {len(unique_neighbors)} neighbors in supercell")
            
            for neighbor in unique_neighbors:
                elem2 = neighbor['label']
                elem2_type = neighbor['element']
                distance = neighbor['distance']
                atom_type_pairs.add(tuple(sorted([elem1_type, elem2_type])))
                
                # Generate position key with 0.05Å grid snapping (matches deduplication)
                pos_x = round(neighbor['position'].x / 0.05) * 0.05
                pos_y = round(neighbor['position'].y / 0.05) * 0.05
                pos_z = round(neighbor['position'].z / 0.05) * 0.05
                position_key = f"{pos_x:.2f}_{pos_y:.2f}_{pos_z:.2f}"
                symmetry_code = "supercell"
                
                print(f"    -> {elem2} ({elem2_type}) dist={distance:.4f}")
                
                # Store bond
                bond_data['bond_label_1'].append(elem1)
                bond_data['bond_label_2'].append(elem2)
                bond_data['bond_distance'].append(f"{distance:.5f}")
                bond_data['bond_symmetry_2'].append(f"{symmetry_code}|{position_key}")
                bond_count += 1
        
        print(f"  ✓ Calculated {bond_count} bonds using crystallographic symmetry (gemmi)")
        print(f"  ✓ Bond types found: {sorted(atom_type_pairs)}")
        print(f"  ✓ Space group: {small_struct.spacegroup_hm}")
        
        return bond_data
        
    except Exception as e:
        print(f"  ✗ Error calculating bonds with gemmi: {e}")
        import traceback
        traceback.print_exc()
        return None

def is_bond_data_present(data_block):
    """
    Check if a CIF data block contains pre-calculated bond distance data.
    
    Args:
        data_block: PyCifRW data block dictionary
    
    Returns:
        True if bond data present (GSAS-II style), False if missing (ICSD style)
    """
    required_keys = [
        '_geom_bond_atom_site_label_1',
        '_geom_bond_atom_site_label_2',
        '_geom_bond_distance',
        '_geom_bond_site_symmetry_2'
    ]
    
    return all(key in data_block for key in required_keys)

# ============================================================================
# DIRECTORY SELECTION DIALOG
# ============================================================================

def select_directory():
    """
    Opens a tkinter dialog to select a directory containing CIF files.
    Returns the selected directory path or None if cancelled.
    """
    import tkinter as tk
    from tkinter import filedialog
    
    root = tk.Tk()
    root.overrideredirect(True)
    root.attributes('-topmost', True)
    root.withdraw()
    
    print("\n" + "="*80)
    print("SELECT DIRECTORY CONTAINING CIF FILES")
    print("="*80)
    print("A file dialog will open. Please select the directory containing your CIF files.")
    print("(If the dialog appears behind other windows, check your taskbar)")
    
    directory_path = filedialog.askdirectory(
        title="Select Directory Containing CIF Files",
        mustexist=True
    )
    
    root.destroy()
    
    if not directory_path:
        print("\n✗ No directory selected. Exiting.")
        return None
    
    print(f"\n✓ Selected directory: {directory_path}")
    
    # Validate directory contains CIF files
    cif_files = list(Path(directory_path).glob("*.cif"))
    if not cif_files:
        print(f"\n✗ ERROR: No CIF files found in {directory_path}")
        print("Please select a directory containing .cif files")
        return None
    
    print(f"✓ Found {len(cif_files)} CIF files")
    
    return directory_path

# ============================================================================
# CIF PARSING (From Skuld 1.5 - EXACT PRESERVATION)
# ============================================================================

def parse_cif_file(file_path):
    """
    Parse a CIF file and return the CifFile object.
    For ICSD CIFs without bond data, calculates bonds from atomic positions.
    """
    try:
        cf = ReadCif(file_path)
        
        # Check if bonds need to be calculated (ICSD CIFs)
        for block_name, data_block in cf.items():
            if not is_bond_data_present(data_block):
                print(f"  ⚠ No bond data in CIF - calculating from crystallographic symmetry (ICSD mode)...")
                
                # Calculate bonds using ASE
                bond_data = calculate_bonds_from_positions(file_path)
                
                if bond_data is not None:
                    # Add calculated bonds to the data block
                    data_block['_geom_bond_atom_site_label_1'] = bond_data['bond_label_1']
                    data_block['_geom_bond_atom_site_label_2'] = bond_data['bond_label_2']
                    data_block['_geom_bond_distance'] = bond_data['bond_distance']
                    data_block['_geom_bond_site_symmetry_2'] = bond_data['bond_symmetry_2']
                else:
                    print(f"  ✗ Failed to calculate bonds - structure will be skipped")
        
        return cf
    except Exception as e:
        print(f"  ERROR parsing {os.path.basename(file_path)}: {e}")
        return None

def extract_initial_site_data(cif_object):
    """
    EXACT COPY from Skuld 1.5 (lines 808-938)
    Extracts initial atom site data and all bond information,
    including proper handling of symmetry operations for bonds beyond unit cell.
    """
    if not cif_object:
        return {}

    processed_data = {}

    for block_name, data_block in cif_object.items():
        block_data = {
            'unit_cell': {},
            'atom_sites_data': [],
            'all_bond_pairs_found': set(),
            'sites_with_bonds': {}
        }
        processed_data[block_name] = block_data

        # --- General Data Items ---
        block_data['publ_author_name'] = data_block.get("_publ_contact_author_name")
        block_data['gsasii_version'] = data_block.get("_gsas_GSASII_version")
        block_data['pd_block_id'] = data_block.get("_pd_block_id")

        # --- Atom Site Data Processing ---
        label_to_canonical_coords = {} 
        label_to_original_coord_strs = {}
        label_to_type = {}
        
        try:
            labels = data_block['_atom_site_label']
            types = data_block['_atom_site_type_symbol']
            xs_orig = data_block['_atom_site_fract_x'] 
            ys_orig = data_block['_atom_site_fract_y']
            zs_orig = data_block['_atom_site_fract_z']

            if not (labels and types and xs_orig and ys_orig and zs_orig and \
                    len(labels) == len(types) == len(xs_orig) == len(ys_orig) == len(zs_orig)):
                print(f"  No complete _atom_site_loop data found in block {block_name}. Skipping atom site processing for this block.")
                block_data['atom_sites_data'] = [] 
            else:
                for i in range(len(labels)):
                    label = labels[i].strip()
                    atom_type_raw = types[i].strip().rstrip('+-0123456789')
                    atom_type = atom_type_raw.capitalize() if len(atom_type_raw) > 1 else atom_type_raw.upper()
                    
                    try:
                        x = float(xs_orig[i].split('(')[0].strip())
                        y = float(ys_orig[i].split('(')[0].strip())
                        z = float(zs_orig[i].split('(')[0].strip())
                        coords_tuple = (x, y, z)
                        
                        block_data['atom_sites_data'].append({'label': label, 'type': atom_type, 'coords': coords_tuple})
                        label_to_canonical_coords[label] = coords_tuple 
                        label_to_original_coord_strs[label] = (xs_orig[i], ys_orig[i], zs_orig[i])
                        label_to_type[label] = atom_type 
                    except (ValueError, IndexError):
                        print(f"  Warning: Could not parse coordinates for atom {label}. Skipping.")
                        continue
        except KeyError:
            print(f"  No _atom_site_label or related loop data found in block {block_name}. Skipping atom site processing for this block.")
            block_data['atom_sites_data'] = []

        # --- Bond Distances (CRITICAL: includes symmetry code handling) ---
        all_raw_bonds_per_central_site = {} 
        site_metadata = {} 

        bond_label_1s = data_block.get("_geom_bond_atom_site_label_1")
        bond_label_2s = data_block.get("_geom_bond_atom_site_label_2")
        bond_distance_raw_strs = data_block.get("_geom_bond_distance")
        bond_symmetry_2s = data_block.get("_geom_bond_site_symmetry_2")

        if bond_label_1s and bond_label_2s and bond_distance_raw_strs and bond_symmetry_2s and \
           all(len(lst) == len(bond_label_1s) for lst in [bond_label_2s, bond_distance_raw_strs, bond_symmetry_2s]):
            
            for i in range(len(bond_label_1s)):
                atom1_label = bond_label_1s[i].strip()
                atom2_label = bond_label_2s[i].strip()
                dist_str = bond_distance_raw_strs[i].strip()
                atom2_symmetry_code = bond_symmetry_2s[i].strip()  # CRITICAL: preserves symmetry info

                dist_val, dist_err = parse_value_with_error(dist_str)
                
                if dist_val is None:
                    continue 

                atom1_canonical_coords = label_to_canonical_coords.get(atom1_label)
                atom2_canonical_coords = label_to_canonical_coords.get(atom2_label) 

                atom1_type = label_to_type.get(atom1_label)
                atom2_type = label_to_type.get(atom2_label)

                # CRITICAL FIX for ICSD CIFs: If type not found in label_to_type,
                # extract element from label (e.g., "Na1" -> "Na", "Cr1" -> "Cr")
                if not atom1_type and atom1_label:
                    # Extract element symbol from label (strip trailing digits)
                    import re
                    match = re.match(r'^([A-Z][a-z]?)', atom1_label)
                    if match:
                        atom1_type = match.group(1)
                
                if not atom2_type and atom2_label:
                    import re
                    match = re.match(r'^([A-Z][a-z]?)', atom2_label)
                    if match:
                        atom2_type = match.group(1)

                if not atom1_type or not atom2_type:
                    continue 
                
                # Validation
                if not validate_element_combination(atom1_type, atom2_type):
                    print(f"  Warning: Unusual element combination {atom1_type}-{atom2_type}, but including in analysis")
                
                block_data['all_bond_pairs_found'].add(tuple(sorted((atom1_type, atom2_type))))

                central_atom_coords = atom1_canonical_coords
                central_atom_element = atom1_type
                
                if central_atom_coords:
                    if central_atom_coords not in all_raw_bonds_per_central_site:
                        all_raw_bonds_per_central_site[central_atom_coords] = []
                        site_metadata[central_atom_coords] = {
                            'elements': set(),
                            'labels_at_site': set(),
                        }

                    site_metadata[central_atom_coords]['elements'].add(central_atom_element)
                    site_metadata[central_atom_coords]['labels_at_site'].add(atom1_label)
                    
                    # CRITICAL: Store complete bond tuple including symmetry code
                    all_raw_bonds_per_central_site[central_atom_coords].append(
                        (atom1_label, atom2_label, atom1_type, atom2_type, atom2_symmetry_code, dist_val, dist_err)
                    )
        else:
            print(f"  No geometric bond data found or incomplete in block {block_name}.")

        block_data['sites_with_bonds'] = {
            coords: {
                'elements': sorted(list(site_metadata[coords]['elements'])),
                'labels': sorted(list(site_metadata[coords]['labels_at_site'])),
                'raw_bonds_list': raw_bonds_list
            } for coords, raw_bonds_list in all_raw_bonds_per_central_site.items()
        }
    
    return processed_data

# ============================================================================
# AUTOMATED BOND SELECTION (v1.2 - Multi-anion support)
# ============================================================================

def has_bvs_parameters_for_pair(cation, anion, valence_dict=None):
    """
    Check if BVS parameters exist for a given cation-anion pair.
    
    Args:
        cation: Cation element symbol (e.g., 'Ge', 'Si', 'Ti')
        anion: Anion element symbol (e.g., 'O', 'F', 'S')
        valence_dict: Optional dict of {element: valence} to check specific oxidation states
    
    Returns:
        bool: True if parameters exist for this pair
    """
    # Check common oxidation states for the cation
    if valence_dict and cation in valence_dict:
        # User specified oxidation state
        cation_valences = [abs(valence_dict[cation])]
    else:
        # Try common oxidation states from database
        cation_valences = db_manager.get_common_oxidation_states(cation)
        if not cation_valences:
            # If not in database, try common defaults
            common_defaults = {
                'Ge': [4, 2], 'Si': [4], 'B': [3], 'P': [5, 3],
                'As': [5, 3], 'Sb': [5, 3], 'Te': [6, 4, 2]
            }
            cation_valences = common_defaults.get(cation, [])
    
    if not cation_valences:
        return False
    
    # CORRECTED: Use bond_valence_params, not bvs_params
    # Also handle the fact that keys are tuples
    for val in cation_valences:
        param_key = (cation, val, anion)
        if param_key in db_manager.bond_valence_params:
            return True
    
    return False

def auto_select_bond_types(processed_data):
    """
    Automatically select cation-anion bond types for analysis based on BVS parameter availability.
    
    EIR v1.3 ENHANCEMENT: Now checks BVS parameter database directly instead of hardcoded metal lists.
    This ensures ALL analyzable elements are included (Ge, Si, B, P, etc.) without manual updates.
    
    Args:
        processed_data: Data structure from previous processing steps
    
    Returns: processed_data with 'selected_bond_types_for_analysis' added
    """
    for block_name, block_content in processed_data.items():
        unique_bond_pairs = sorted(list(block_content['all_bond_pairs_found']))
        
        print(f"  DEBUG: Found {len(unique_bond_pairs)} unique bond pairs: {unique_bond_pairs}")
        
        if not unique_bond_pairs:
            block_content['selected_bond_types_for_analysis'] = {}
            continue
        
        # Auto-select ALL cation-anion bonds where BVS parameters exist
        selected_bond_types_map = {}
        
        for type1, type2 in unique_bond_pairs:
            # Identify which element is the anion
            anion = None
            cation = None
            
            if is_anion(type1) and not is_anion(type2):
                anion = type1
                cation = type2
            elif is_anion(type2) and not is_anion(type1):
                anion = type2
                cation = type1
            else:
                # Neither is a known anion, or both are - skip
                continue
            
            # EIR v1.3: Check if BVS parameters exist for this pair
            has_params = has_bvs_parameters_for_pair(cation, anion)
            
            print(f"  DEBUG: Bond {cation}-{anion}: has_bvs_params={has_params}")
            
            if has_params:
                # Parameters exist - include this bond type for analysis
                selected_bond_types_map[(cation, anion)] = {
                    'central_atom_type': cation,
                    'ligand_type': anion
                }
            else:
                print(f"  WARNING: No BVS parameters for {cation}-{anion}, excluding from analysis")
        
        block_content['selected_bond_types_for_analysis'] = selected_bond_types_map
        
        if selected_bond_types_map:
            print(f"  ✓ Auto-selected {len(selected_bond_types_map)} cation-anion bond types: {list(selected_bond_types_map.keys())}")
        else:
            print(f"  ✗ No cation-anion bonds with BVS parameters found")
    
    return processed_data

# ============================================================================
# AUTOMATED COORDINATION ANALYSIS (Replaces interactive_review_and_correct)
# ============================================================================

def auto_analyze_coordination(processed_data):
    """
    Automatically analyse coordination numbers and bond lengths.
    Uses the EXACT gap analysis logic from Skuld 1.5 but accepts results automatically.
    
    CRITICAL: Preserves lines 1083-1160 from Skuld 1.5 for proper coordination detection.
    """
    final_data = processed_data

    for block_name, block_content in final_data.items():
        
        if not block_content['sites_with_bonds'] or not block_content['selected_bond_types_for_analysis']:
            continue

        for coords, site_info in block_content['sites_with_bonds'].items():
            central_atom_elements_at_site = site_info['elements']
            
            bonds_for_analysis = {}
            for (central_type_selected, ligand_type_selected) in block_content['selected_bond_types_for_analysis'].keys():
                if central_type_selected in central_atom_elements_at_site:
                    filtered_raw_bonds = [
                        b for b in site_info['raw_bonds_list'] 
                        if b[2] == central_type_selected and b[3] == ligand_type_selected
                    ]
                    if filtered_raw_bonds:
                        bonds_for_analysis[(central_type_selected, ligand_type_selected)] = filtered_raw_bonds

            if not bonds_for_analysis:
                continue

            site_info['cn_and_bonds_by_type'] = {}

            for (central_type, ligand_type), raw_bonds_list_for_pair in bonds_for_analysis.items():
                
                # Report site coordinates for validation
                print(f"\n    Analyzing site at {coords}: {central_type}")
                
                # ======================================================================
                # CRITICAL SECTION: EXACT COPY from Skuld 1.5 (lines 1083-1160)
                # This handles symmetry operations and proper coordination detection
                # ======================================================================
                
                # STEP 1: Identify unique geometric positions using (label, position)
                # For ICSD files, symmetry_code now includes |x_y_z position encoding
                unique_geometric_positions_to_shortest_bond = {}
                
                # DEBUG: Show first few bonds to verify position encoding
                if len(raw_bonds_list_for_pair) <= 10:
                    print(f"  DEBUG: Raw bonds for {central_type}-{ligand_type}:")
                    for bond_tuple in raw_bonds_list_for_pair[:10]:
                        print(f"    {bond_tuple[0]}-{bond_tuple[1]}: {bond_tuple[5]:.4f}Å symm='{bond_tuple[4]}'")
                
                # DEBUG: Track unique positions
                position_debug = set()
                
                for bond_tuple in raw_bonds_list_for_pair:
                    ligand_label = bond_tuple[1]
                    ligand_symmetry_code = bond_tuple[4]  # May include |x_y_z position
                    dist_val = bond_tuple[5]
                    dist_err = bond_tuple[6]

                    # Use Cartesian position if available, else use symmetry code
                    if '|' in ligand_symmetry_code:
                        # ICSD mode: extract position part for uniqueness
                        symm_part, pos_part = ligand_symmetry_code.split('|', 1)
                        geo_position_id = (ligand_label, pos_part)  # Use 3D position
                        position_debug.add(pos_part)
                    else:
                        # GSAS-II mode: use symmetry code
                        geo_position_id = (ligand_label, ligand_symmetry_code)
                    
                    if geo_position_id not in unique_geometric_positions_to_shortest_bond or \
                       dist_val < unique_geometric_positions_to_shortest_bond[geo_position_id][0]:  # Compare with OLD distance [0], not error [1]
                        unique_geometric_positions_to_shortest_bond[geo_position_id] = (dist_val, dist_err)
                
                # DEBUG: Show unique positions found
                if position_debug:
                    print(f"  DEBUG: Found {len(position_debug)} unique positions: {sorted(list(position_debug)[:6])}")

                # STEP 2: Create list of all bonds within coordination sphere
                cn_bonds_list = []
                for (dist_val, dist_err) in unique_geometric_positions_to_shortest_bond.values():
                    cn_bonds_list.append((dist_val, dist_err))
                
                sorted_cn_bonds = sorted(cn_bonds_list, key=lambda x: x[0])

                # STEP 3: Gap analysis to find primary coordination sphere
                # Add minimum gap threshold to avoid splitting coordination due to numerical precision
                MIN_GAP_THRESHOLD = 0.05  # Å - ignore gaps smaller than this
                initial_cn = 0
                primary_sphere_cutoff_dist = 0.0 
                
                if sorted_cn_bonds:
                    max_gap = 0
                    cutoff_index = len(sorted_cn_bonds) 
                    
                    if len(sorted_cn_bonds) > 1:
                        for i in range(len(sorted_cn_bonds) - 1):
                            gap = sorted_cn_bonds[i+1][0] - sorted_cn_bonds[i][0]
                            # Only consider gaps larger than threshold
                            if gap > MIN_GAP_THRESHOLD and gap > max_gap:
                                max_gap = gap
                                cutoff_index = i + 1 
                    
                    initial_cn = cutoff_index
                    if cutoff_index > 0:
                        primary_sphere_cutoff_dist = sorted_cn_bonds[cutoff_index - 1][0]

                # STEP 4: Extract bonds within primary sphere
                initial_bonds_for_display = []
                for bond_tuple in sorted_cn_bonds: 
                    dist_val, dist_err = bond_tuple
                    if dist_val <= primary_sphere_cutoff_dist:
                        initial_bonds_for_display.append({'distance': dist_val, 'error': dist_err})
                
                # STEP 5: Count bond multiplicities
                initial_detailed_bond_counts = []
                if initial_bonds_for_display:
                    grouped_bonds_info = []
                    for bond_dict in initial_bonds_for_display:
                        rounded_dist = round(bond_dict['distance'], 5)
                        original_error = bond_dict['error']
                        grouped_bonds_info.append((rounded_dist, original_error, bond_dict['distance']))

                    bond_counts_map = Counter((r_d, o_e) for r_d, o_e, _ in grouped_bonds_info)
                    sorted_unique_keys = sorted(bond_counts_map.keys())

                    for (dist_val_rounded, dist_err_val) in sorted_unique_keys:
                        count = bond_counts_map[(dist_val_rounded, dist_err_val)]
                        original_dist_val = next((orig_d for r_d, e_v, orig_d in grouped_bonds_info 
                                                  if r_d == dist_val_rounded and e_v == dist_err_val), 
                                                 dist_val_rounded)

                        initial_detailed_bond_counts.append({
                            'count': count, 
                            'distance': original_dist_val, 
                            'error': dist_err_val 
                        })

                # ======================================================================
                # END CRITICAL SECTION
                # ======================================================================
                
                # AUTOMATED DECISION: Accept calculated CN and bonds
                confirmed_cn = initial_cn
                confirmed_bond_counts = initial_detailed_bond_counts
                
                # DIAGNOSTIC OUTPUT: Show bond distances for validation
                if initial_detailed_bond_counts:
                    bond_strs = []
                    for b in initial_detailed_bond_counts:
                        bond_strs.append(f"{b['count']}×{b['distance']:.4f}Å")
                    
                    avg_dist = sum(b['distance']*b['count'] for b in initial_detailed_bond_counts) / sum(b['count'] for b in initial_detailed_bond_counts)
                    
                    # VALIDATION: Check against expected values for different bond types
                    warnings = []
                    
                    # Oxide validation
                    if ligand_type == 'O':
                        if central_type == 'Cr' and confirmed_cn == 6:
                            if avg_dist < 1.95 or avg_dist > 2.05:
                                warnings.append(f"⚠️ Cr-O distance {avg_dist:.3f}Å unusual (expect 1.98-2.01Å)")
                        elif central_type == 'Na' and confirmed_cn == 6:
                            if avg_dist < 2.25 or avg_dist > 2.40:
                                warnings.append(f"⚠️ Na-O distance {avg_dist:.3f}Å unusual (expect 2.30-2.35Å)")
                        elif central_type == 'Ni' and confirmed_cn == 6:
                            if avg_dist < 2.00 or avg_dist > 2.15:
                                warnings.append(f"⚠️ Ni-O distance {avg_dist:.3f}Å unusual (expect 2.05-2.10Å)")
                    
                    # Halide validation
                    elif ligand_type == 'Cl':
                        if central_type == 'Na' and confirmed_cn == 6:
                            if avg_dist < 2.75 or avg_dist > 2.90:
                                warnings.append(f"⚠️ Na-Cl distance {avg_dist:.3f}Å unusual (expect 2.80-2.85Å)")
                    
                    elif ligand_type == 'F':
                        if central_type == 'Na' and confirmed_cn == 6:
                            if avg_dist < 2.25 or avg_dist > 2.45:
                                warnings.append(f"⚠️ Na-F distance {avg_dist:.3f}Å unusual (expect 2.30-2.35Å)")
                    
                    # Sulfide validation
                    elif ligand_type == 'S':
                        if central_type == 'Na' and confirmed_cn == 6:
                            if avg_dist < 2.80 or avg_dist > 3.00:
                                warnings.append(f"⚠️ Na-S distance {avg_dist:.3f}Å unusual (expect 2.85-2.95Å)")
                    
                    # Print diagnostic info
                    print(f"    {central_type}-{ligand_type}: CN={confirmed_cn}, bonds=[{', '.join(bond_strs)}], avg={avg_dist:.4f}Å")
                    for warning in warnings:
                        print(f"      {warning}")
                
                site_info['cn_and_bonds_by_type'][(central_type, ligand_type)] = {
                    'confirmed_cn': confirmed_cn,
                    'confirmed_bond_counts': confirmed_bond_counts
                }
    
    return final_data

# ============================================================================
# AUTOMATED VALENCE ASSIGNMENT (Replaces get_valences_interactively)
# ============================================================================

def auto_assign_valences(processed_data, user_oxidation_states=None, per_atom_valences=None):
    """
    Assign oxidation states with priority:
    
    1. Per-atom explicit valence from CIF (e.g., "Fe3+" -> 3) [NEW in v1.4.2]
    2. User-provided oxidation states (config or interactive)
    3. Database common states
    4. Hardcoded fallback defaults
    
    Args:
        processed_data: Data structure from previous processing steps
        user_oxidation_states: Optional dict of {element: valence}
        per_atom_valences: Optional dict from extract_per_atom_valences() [NEW]
    """
    for block_name, block_content in processed_data.items():
        if not block_content['sites_with_bonds'] or not block_content.get('selected_bond_types_for_analysis'):
            continue

        for coords, site_info in block_content['sites_with_bonds'].items():
            if 'cn_and_bonds_by_type' not in site_info:
                continue
            
            relevant_elements = set()
            for (central_type, ligand_type) in site_info['cn_and_bonds_by_type'].keys():
                relevant_elements.add(central_type)
                relevant_elements.add(ligand_type)
            
            if not relevant_elements:
                continue

            site_valences = {}
            valence_sources = {}
            
            for element in relevant_elements:
                valence = None
                source = None
                
                # PRIORITY 1 [NEW]: Per-atom explicit valence from CIF
                if per_atom_valences:
                    site_label = site_info.get('site_label', '')
                    if site_label in per_atom_valences:
                        atom_data = per_atom_valences[site_label]
                        if atom_data['element'] == element and atom_data['explicit_valence'] is not None:
                            valence = atom_data['explicit_valence']
                            source = 'cif_per_atom'
                
                # PRIORITY 2: User-provided oxidation states
                if valence is None and user_oxidation_states and element in user_oxidation_states:
                    valence = user_oxidation_states[element]
                    source = 'user_override'
                
                # PRIORITY 3: Database common states
                if valence is None:
                    common_states = db_manager.get_common_oxidation_states(element)
                    if common_states:
                        valence = common_states[0]
                        source = 'database'
                    else:
                        # PRIORITY 4: Hardcoded fallback
                        default_valences = {
                            'Na': 1, 'K': 1, 'Li': 1,
                            'O': -2, 'F': -1, 'S': -2,
                            'Ni': 2, 'Mn': 3, 'Fe': 3, 'Cr': 3,
                            'Ti': 4, 'Al': 3, 'Mg': 2, 'Sb': 5
                        }
                        valence = default_valences.get(element, 0)
                        source = 'fallback'
                
                site_valences[element] = valence
                valence_sources[element] = source
            
            site_info['confirmed_valences'] = site_valences
            site_info['valence_sources'] = valence_sources
    
    return processed_data

# ============================================================================
# BVS CALCULATIONS (From Skuld 1.5 - EXACT PRESERVATION)
# ============================================================================

def perform_calculations_and_report(processed_data, b_param_default=0.37):
    """
    Perform BVS, Shannon-Prewitt, and other crystallochemical calculations.
    PRESERVES EXACT calculation logic from Skuld 1.5 (lines 1307-1537).
    """
    calculated_results = {}

    for block_name, block_content in processed_data.items():
        
        if not block_content['sites_with_bonds'] or not block_content.get('selected_bond_types_for_analysis'):
            continue
        
        calculated_results[block_name] = {}

        for coords, site_info in block_content['sites_with_bonds'].items():
            if 'cn_and_bonds_by_type' not in site_info:
                continue

            site_results = {
                'elements': site_info['elements'],
                'labels': site_info['labels'],
                'element_data': {}
            }

            for (central_type, ligand_type), bond_analysis_data in site_info['cn_and_bonds_by_type'].items():
                confirmed_cn = bond_analysis_data['confirmed_cn']
                confirmed_bond_counts = bond_analysis_data['confirmed_bond_counts']
                
                central_valence = site_info['confirmed_valences'].get(central_type)
                ligand_valence = site_info['confirmed_valences'].get(ligand_type)

                if central_valence is None or ligand_valence is None:
                    continue
                
                element_calc_data = {'valence': central_valence, 'ligand_valence': ligand_valence, 'cn': confirmed_cn}

                # --- Shannon-Prewitt Ionic Radii Comparison ---
                if confirmed_bond_counts:
                    central_radius = db_manager.get_ionic_radius(central_type, central_valence, confirmed_cn)
                    ligand_radius = db_manager.get_ionic_radius(ligand_type, ligand_valence, confirmed_cn)

                    if central_radius is not None and ligand_radius is not None:
                        expected_distance = central_radius + ligand_radius
                        
                        element_shannon_diffs = []
                        
                        for bond_info in confirmed_bond_counts:
                            count = bond_info['count']
                            observed_distance = bond_info['distance']
                            bond_error = bond_info.get('error', np.nan)
                            percent_diff = ((observed_distance - expected_distance) / expected_distance) * 100
                            element_shannon_diffs.append({
                                'bond_count': count, 
                                'observed_distance': observed_distance, 
                                'expected_distance': expected_distance, 
                                'percent_diff': percent_diff,
                                'error': bond_error 
                            })
                        
                        element_calc_data['shannon_diffs'] = element_shannon_diffs
                        element_calc_data['cation_radius'] = central_radius
                        element_calc_data['anion_radius'] = ligand_radius

                # --- Ionic Potential ---
                if element_calc_data.get('cation_radius') and element_calc_data['cation_radius'] > 0:
                    ionic_potential = float(abs(central_valence)) / element_calc_data['cation_radius']
                    element_calc_data['ionic_potential'] = ionic_potential

                # --- Bond Valence Sum (BVS) Calculation ---
                if confirmed_bond_counts:
                    r0_val, b_val = get_bv_parameters_fractional(central_type, central_valence, ligand_type)
                    
                    if r0_val is not None:
                        bvs_total = 0.0
                        
                        for bond_info in confirmed_bond_counts:
                            count = bond_info['count']
                            distance = bond_info['distance']
                            
                            s = math.exp((r0_val - distance) / b_val)
                            bvs_total += (s * count)
                        
                        if central_valence != 0:
                            bvs_percent_diff = ((bvs_total - abs(central_valence)) / abs(central_valence)) * 100
                            
                            element_calc_data['bvs'] = bvs_total
                            element_calc_data['bvs_percent_diff'] = bvs_percent_diff

                # --- Bond Length Distortion (BLD) ---
                if confirmed_bond_counts and confirmed_cn > 0:
                    total_distance_for_avg = sum(b['distance'] * b['count'] for b in confirmed_bond_counts)
                    total_count_for_avg = sum(b['count'] for b in confirmed_bond_counts)
                    
                    if total_count_for_avg > 0:
                        average_observed_distance = total_distance_for_avg / total_count_for_avg
                        
                        sum_of_squared_deviations = 0.0
                        for bond_info in confirmed_bond_counts:
                            distance = bond_info['distance']
                            count = bond_info['count']
                            sum_of_squared_deviations += count * ((distance - average_observed_distance) / average_observed_distance)**2
                        
                        bld = sum_of_squared_deviations / confirmed_cn
                        element_calc_data['bond_length_distortion'] = bld

                site_results['element_data'][central_type] = element_calc_data
            
            calculated_results[block_name][coords] = site_results
            
    return calculated_results

def calculate_global_instability_index(calculated_results, cif_path=None, site_occupancies=None, per_atom_valences=None):
    """
    Calculate Global Instability Index (GII) and Weighted-Average GII (WAGII).
    
    WAGII weights each contribution by site occupancy, providing a more physically
    meaningful metric for disordered structures where minority species at partially
    occupied sites can disproportionately influence conventional GII.
    
    For fully occupied sites (occ=1.0), WAGII = GII.
    """
    if site_occupancies is None and cif_path is not None:
        site_occupancies = extract_site_occupancies(cif_path)
    
    if site_occupancies is None:
        site_occupancies = {}
    
    gii_results = {}
    
    for block_name, block_results in calculated_results.items():
        total_bvs_deviation = 0.0
        total_formal_valence = 0.0
        total_weighted_bvs_deviation = 0.0
        total_weighted_formal_valence = 0.0
        site_contributions = []
        
        for site_coords, site_data in block_results.items():
            if 'element_data' not in site_data:
                continue
            
            site_labels = site_data.get('labels', [])
            
            # v1.5 FIX: Iterate by LABEL to handle mixed-valence sites correctly
            # (e.g., Fe2/Fe3 with Fe2+/Fe3+ need separate GII contributions)
            for label in site_labels:
                # Resolve element for this label
                element = None
                for elem in site_data['elements']:
                    if label.upper().startswith(elem.upper()):
                        element = elem
                        break
                if element is None and site_data['elements']:
                    element = site_data['elements'][0]
                if element is None:
                    continue
                
                element_data = site_data['element_data'].get(element, {})
                
                if 'bvs' in element_data and element_data['bvs'] is not None:
                    observed_bvs = element_data['bvs']
                    
                    # Use per-atom valence if available (correct for mixed-valence)
                    formal_valence = abs(element_data.get('valence', 0))
                    if per_atom_valences and label in per_atom_valences:
                        pav = per_atom_valences[label]
                        if pav.get('explicit_valence') is not None:
                            formal_valence = abs(pav['explicit_valence'])
                    
                    if formal_valence > 0:
                        bvs_deviation = abs(observed_bvs - formal_valence)
                        total_bvs_deviation += bvs_deviation
                        total_formal_valence += formal_valence
                        
                        # Find occupancy for this label
                        occupancy = 1.0
                        if label in site_occupancies:
                            occ_data = site_occupancies[label]
                            occupancy = occ_data.get('occupancy', 1.0)
                        
                        total_weighted_bvs_deviation += occupancy * bvs_deviation
                        total_weighted_formal_valence += occupancy * formal_valence
                        
                        site_contributions.append({
                            'site': site_coords,
                            'element': element,
                            'label': label,
                            'formal_valence': formal_valence,
                            'observed_bvs': observed_bvs,
                            'bvs_deviation': bvs_deviation,
                            'bvs_percent_diff': element_data.get('bvs_percent_diff', 0),
                            'occupancy': occupancy,
                            'element_category': db_manager.get_element_category(element)
                        })
        
        if total_formal_valence > 0:
            gii = total_bvs_deviation / total_formal_valence
            wagii = total_weighted_bvs_deviation / total_weighted_formal_valence if total_weighted_formal_valence > 0 else gii
            
            if gii < 0.05:
                stability_assessment = "Very Stable"
            elif gii < 0.1:
                stability_assessment = "Stable" 
            elif gii < 0.2:
                stability_assessment = "Moderately Unstable"
            elif gii < 0.3:
                stability_assessment = "Unstable"
            else:
                stability_assessment = "Highly Unstable"
            
            gii_results[block_name] = {
                'gii': gii,
                'wagii': wagii,
                'stability_assessment': stability_assessment,
                'site_contributions': site_contributions,
                'total_sites_analyzed': len(site_contributions)
            }
        else:
            gii_results[block_name] = {
                'gii': None,
                'wagii': None,
                'stability_assessment': 'Cannot Assess',
                'site_contributions': [],
                'total_sites_analyzed': 0
            }
    
    return gii_results

# ============================================================================
# BATCH PROCESSING ENGINE
# ============================================================================

def process_single_cif(file_path, user_oxidation_states=None):
    """
    Process a single CIF file using validated Eir 1.0 logic.
    NEW in v1.2 continued: Mixed occupancy detection, Bosi (2014) strain analysis,
    and automatic 3D visualisation of problematic sites.
    NEW in v1.4: Structure-type specific analysis (spinel, perovskite, etc.)
    
    Args:
        file_path: Path to CIF file
        user_oxidation_states: Optional dict of {element: valence} overrides
                              If provided, these oxidation states will be used instead of database defaults
    
    Returns:
        dict with processing results including success status, metadata, sites data, and GII
    """
    results = {
        'success': False,
        'error': None,
        'metadata': {},
        'sites': [],
        'gii': None,
        'filename': os.path.basename(file_path)
    }
    
    # Parse CIF
    cif_obj = parse_cif_file(file_path)
    if cif_obj is None:
        results['error'] = "Failed to parse CIF file"
        return results
    
    # Process first data block
    block_name = list(cif_obj.keys())[0]
    data_block = cif_obj[block_name]
    
    # Extract metadata
    results['metadata'] = {
        'block_name': block_name,
        'filename': os.path.basename(file_path),
        'chemical_formula': data_block.get("_chemical_formula_sum", "N/A"),
        'space_group': data_block.get("_space_group_name_H-M_alt", 
                                     data_block.get("_symmetry_space_group_name_H-M", "N/A")),
    }
    
    # Extract ICSD publication metadata
    icsd_metadata = extract_icsd_metadata(file_path)
    results['metadata'].update({
        'icsd_code': icsd_metadata.get('icsd_code'),
        'year': icsd_metadata.get('year'),
        'journal': icsd_metadata.get('journal'),
        'authors': ', '.join(icsd_metadata.get('authors', [])) if icsd_metadata.get('authors') else None,
        'r_factor': icsd_metadata.get('r_factor'),
        'density': icsd_metadata.get('density'),
        'measurement_temperature_K': icsd_metadata.get('measurement_temperature_K'),
        'measurement_pressure_Pa': icsd_metadata.get('measurement_pressure_Pa'),
        'measurement_pressure_GPa': icsd_metadata.get('measurement_pressure_GPa'),
        'measurement_conditions': icsd_metadata.get('measurement_conditions')
    })
    
    # Validated Eir 1.0 workflow
    try:
        # Step 1: Extract site data
        initial_extracted_data = extract_initial_site_data(cif_obj)
        
        if not initial_extracted_data:
            results['error'] = "No data extracted from CIF"
            return results
        
        # Step 2: Auto-select bond types
        data_with_selected_bonds = auto_select_bond_types(initial_extracted_data)
        
        if not data_with_selected_bonds[block_name].get('selected_bond_types_for_analysis'):
            results['error'] = "No M-X bonds found"  # Updated to M-X (not just M-O)
            return results
        
        # Step 3: Auto-analyse coordination
        final_corrected_data = auto_analyze_coordination(data_with_selected_bonds)
        
        # Step 3.5 [NEW v1.4.2]: Extract per-atom valences for mixed-valence support
        per_atom_valences = extract_per_atom_valences(file_path)
        
        # Step 3.6 [NEW v1.7]: Extract site occupancies for integrity tier classification
        site_occupancies_data = extract_site_occupancies(file_path)
        
        # Step 4: Auto-assign valences (now with per-atom support)
        final_data_with_valences = auto_assign_valences(
            final_corrected_data, 
            user_oxidation_states,
            per_atom_valences
        )
        
        # Step 5: Perform calculations
        calculated_results = perform_calculations_and_report(final_data_with_valences)
        
        # Step 6: Calculate GII
        gii_results = calculate_global_instability_index(calculated_results, cif_path=file_path, per_atom_valences=per_atom_valences)
        
        # ============================================================================
        # STEP 7: STRUCTURE-TYPE SPECIFIC ANALYSIS (Eir v1.4+)
        # ============================================================================
        structure_specific_results = {}

        if 'structure_registry' in globals():
            analyzer = structure_registry.find_analyzer(
                results['metadata']['space_group'],
                results['metadata']['chemical_formula'],
                results['metadata']
            )
            
            if analyzer:
                print(f"    → Running {analyzer.name} analysis...", end=" ")
                try:
                    structure_specific_results = analyzer.analyze(
                        data_block,
                        results['metadata'],
                        calculated_results.get(block_name, {}),
                        final_corrected_data,
                        cif_path=file_path  # NEW IN v1.4: Pass CIF path for structural extraction
                    )
                    print(f"✓")
                    
                    # Print summary if spinel
                    if analyzer.name == "Spinel" and structure_specific_results.get('spinel_classification'):
                        classification = structure_specific_results['spinel_classification']
                        lambda_val = structure_specific_results.get('inversion_parameter')
                        if lambda_val is not None:
                            print(f"       {classification} spinel (λ={lambda_val:.3f})")
                        else:
                            print(f"       {classification}")
                            
                except Exception as e:
                    print(f"⚠ (failed: {str(e)})")
                    structure_specific_results = {}
        

        # ============================================================================
        # Extract results for export WITH MIXED OCCUPANCY ANALYSIS
        # ============================================================================
        if block_name in calculated_results:
            for coords, site_data in calculated_results[block_name].items():
                labels = site_data.get('labels', ['unknown'])
                elements_at_site = site_data.get('elements', [])
                
                # ================================================================
                # v1.4.2 FIX: Iterate by LABEL, not by element
                # This ensures each site_label (e.g., Fe2, Fe3) gets its own row
                # with the correct element and explicit valence from the CIF
                # ================================================================
                for site_label in labels:
                    
                    # ----------------------------------------------------------
                    # Determine element and valence for this specific label
                    # ----------------------------------------------------------
                    if per_atom_valences and site_label in per_atom_valences:
                        # Best case: found in per_atom_valences with explicit data
                        atom_info = per_atom_valences[site_label]
                        element = atom_info['element']
                        explicit_valence = atom_info['explicit_valence']
                        valence_source = 'cif_per_atom' if explicit_valence is not None else 'cif_inferred'
                    else:
                        # Fallback: try to match label to an element in the site's element list
                        element = None
                        
                        # First try: label starts with element symbol (Fe1 -> Fe)
                        for elem in elements_at_site:
                            if site_label.upper().startswith(elem.upper()):
                                element = elem
                                break
                        
                        # Second try: use first element at site
                        if element is None and elements_at_site:
                            element = elements_at_site[0]
                            print(f"    ⚠ Warning: Could not match label '{site_label}' to element, using {element}")
                        
                        # Last resort: extract from label string
                        if element is None:
                            element = re.sub(r'[0-9]+$', '', site_label)
                            element = element.capitalize() if len(element) > 1 else element.upper()
                        
                        explicit_valence = None
                        valence_source = 'fallback'
                    
                    # Skip if we couldn't determine element
                    if not element:
                        print(f"    ⚠ Skipping label '{site_label}' - could not determine element")
                        continue
                    
                    # ----------------------------------------------------------
                    # Get element_data (BVS, CN, etc.) for this element
                    # ----------------------------------------------------------
                    elem_data = site_data['element_data'].get(element, {})
                    
                    if not elem_data:
                        # Element wasn't analysed (e.g., no bonds found, or anion)
                        # Skip silently for anions, warn for cations
                        if element not in ['O', 'S', 'Se', 'Te', 'F', 'Cl', 'Br', 'I', 'N']:
                            print(f"    ⚠ No element_data for {element} at {site_label}")
                        continue
                    
                    # ----------------------------------------------------------
                    # Override valence with explicit per-atom valence if available
                    # ----------------------------------------------------------
                    elem_data = elem_data.copy()  # Don't modify original
                    
                    if explicit_valence is not None:
                        elem_data['valence'] = explicit_valence
                        elem_data['valence_source'] = valence_source
                        
                        # Recalculate BVS percent diff with correct valence
                        bvs_val = elem_data.get('bvs')
                        if bvs_val is not None and explicit_valence != 0:
                            elem_data['bvs_percent_diff'] = (bvs_val - abs(explicit_valence)) / abs(explicit_valence) * 100
                    else:
                        elem_data['valence_source'] = valence_source
                    
                    # ----------------------------------------------------------
                    # Get bond information for 3D visualisation
                    # ----------------------------------------------------------
                    bond_counts = None
                    full_site_info = None
                    
                    if block_name in final_corrected_data:
                        for site_coords, site_info in final_corrected_data[block_name]['sites_with_bonds'].items():
                            if site_coords == coords:
                                if 'cn_and_bonds_by_type' in site_info:
                                    for (central_type, ligand_type), bond_analysis in site_info['cn_and_bonds_by_type'].items():
                                        if central_type == element:
                                            bond_counts = bond_analysis.get('confirmed_bond_counts', [])
                                            break
                                full_site_info = site_info
                                break
                    
                    # ========================================================================
                    # Mixed occupancy analysis (from helper functions in Cell 3)
                    # ========================================================================
                    mixed_occ_data = analyze_mixed_occupancy_at_site(
                        coords, 
                        site_data,
                        final_corrected_data, 
                        block_name,
                        data_block
                    )
                    
                    # ========================================================================
                    # Calculate deviation and d1 (discrepancy factor from Bosi 2014)
                    # ========================================================================
                    bvs_val = elem_data.get('bvs')
                    valence = elem_data.get('valence')
                    
                    if bvs_val is not None and valence is not None:
                        deviation = bvs_val - abs(valence)
                    else:
                        deviation = None
                    
                    d1, abs_d1, bonding_state = calculate_discrepancy_factor(bvs_val, valence)
                    
                    # ========================================================================
                    # Assign confidence flag based on disorder type and strain
                    # ========================================================================
                    confidence, bosi_likely, needs_review = assign_confidence_flag(
                        mixed_occ_data['disorder_type'],
                        mixed_occ_data['valence_difference'],
                        abs_d1,
                        deviation
                    )
                    
                    # ========================================================================
                    # DATA INTEGRITY TIER ASSIGNMENT (NEW IN v1.7)
                    # ========================================================================
                    # Check minority species (I3) and mixed TM occupancy (FLAG)
                    site_occupancy = per_atom_valences[site_label]['occupancy'] if (per_atom_valences and site_label in per_atom_valences) else 1.0
                    is_minority = check_minority_species(site_label, per_atom_valences or {}, site_occupancies_data or {})
                    is_mixed_tm, has_noninteger_tm = check_mixed_tm_site(site_label, per_atom_valences or {}, site_occupancies_data or {})
                    
                    # Get the CIF-assigned valence (before any override) for tier metadata
                    cif_valence = None
                    formal_valence_from_symbol = None
                    symbol_number_agreement = None
                    
                    if per_atom_valences and site_label in per_atom_valences:
                        pav_info = per_atom_valences[site_label]
                        cif_valence = pav_info.get('explicit_valence')
                        # Parse integer from type_symbol for comparison
                        if pav_info.get('type_symbol'):
                            _, symbol_val = parse_atom_type_with_valence(pav_info['type_symbol'])
                            if symbol_val is not None:
                                formal_valence_from_symbol = int(symbol_val) if float(symbol_val) == int(symbol_val) else symbol_val
                            # Check agreement between symbol and oxidation_number
                            if cif_valence is not None and symbol_val is not None:
                                symbol_number_agreement = abs(cif_valence - symbol_val) < 0.05
                    
                    integrity_tier, flag_reason = assign_site_integrity_tier(
                        element=element,
                        bvs_value=bvs_val,
                        expected_valence=valence,
                        occupancy=site_occupancy,
                        is_mixed_tm_site=is_mixed_tm,
                        has_noninteger_tm_on_site=has_noninteger_tm,
                        is_minority_on_majority_site=is_minority
                    )
                    
                    # ========================================================================
                    # FLAG SEVERITY & "WHAT IF" ANALYSIS (v1.7)
                    # ========================================================================
                    flag_severity = None
                    whatif_result = {
                        'best_integer_valence': None,
                        'best_integer_deviation': None,
                        'candidates_tested': None,
                        'depositor_deviation': None,
                    }
                    
                    if integrity_tier == 'FLAG':
                        # Assess whether the depositor's partitioning produces a valid BVS
                        flag_severity = assess_flag_severity(bvs_val, valence)
                        # Test alternative integer oxidation states
                        whatif_result = whatif_analysis(element, bvs_val, valence)
                    
                    # ========================================================================
                    # Build site result with ALL columns
                    # ========================================================================
                    site_result = {
                        # Original columns
                        'coords': coords,
                        'element': element,
                        'site_label': site_label,
                        'valence': elem_data.get('valence'),
                        'valence_source': elem_data.get('valence_source', 'unknown'),  # NEW in v1.4.2
                        'cn': elem_data.get('cn'),
                        'bvs': round(elem_data.get('bvs'), 4) if elem_data.get('bvs') is not None else None,
                        'bvs_percent_diff': round(elem_data.get('bvs_percent_diff'), 2) if elem_data.get('bvs_percent_diff') is not None else None,
                        'deviation': round(deviation, 4) if deviation is not None else None,
                        
                        # Mixed occupancy columns
                        'has_mixed_occupancy': mixed_occ_data['has_mixed_occupancy'],
                        'occupancy_string': mixed_occ_data['occupancy_string'],
                        'n_elements_at_site': mixed_occ_data['n_elements_at_site'],
                        'disorder_type': mixed_occ_data['disorder_type'],
                        'valence_difference': round(mixed_occ_data['valence_difference'], 1) if mixed_occ_data['valence_difference'] else 0.0,
                        'max_valence_at_site': mixed_occ_data['max_valence_at_site'],
                        'min_valence_at_site': mixed_occ_data['min_valence_at_site'],
                        
                        # Bosi (2014) strain indicators
                        'discrepancy_d1': round(d1, 4) if d1 is not None else None,
                        'abs_discrepancy_d1': round(abs_d1, 4) if abs_d1 is not None else None,
                        'bonding_state': bonding_state,
                        
                        # Confidence & flagging
                        'confidence_flag': confidence,
                        'bosi_artifact_likely': bosi_likely,
                        'requires_manual_review': needs_review,
                        
                        # Data integrity tier (NEW in v1.7)
                        'integrity_tier': integrity_tier,
                        'flag_reason': flag_reason,
                        'flag_severity': flag_severity,
                        'occupancy': round(site_occupancy, 4),
                        'is_minority_species': is_minority,
                        'is_mixed_occupancy_tm': is_mixed_tm,
                        'site_delta_v': round(mixed_occ_data['valence_difference'], 1) if mixed_occ_data['valence_difference'] else 0.0,
                        'cif_valence': cif_valence,
                        'formal_valence_from_symbol': formal_valence_from_symbol,
                        'symbol_number_agreement': symbol_number_agreement,
                        # "What if" analysis (FLAG-tier sites only)
                        'best_integer_valence': whatif_result['best_integer_valence'],
                        'best_integer_deviation': whatif_result['best_integer_deviation'],
                        'candidates_tested': whatif_result['candidates_tested'],
                        'depositor_deviation': whatif_result['depositor_deviation'],
                        
                        # Original columns
                        'ionic_potential': round(elem_data.get('ionic_potential'), 3) if elem_data.get('ionic_potential') is not None else None,
                        'cation_radius': round(elem_data.get('cation_radius'), 3) if elem_data.get('cation_radius') is not None else None,
                        'bld': round(elem_data.get('bond_length_distortion'), 6) if elem_data.get('bond_length_distortion') is not None else None,
                    }
                    
                    # Add Shannon-Prewitt data if available (unchanged)
                    if 'shannon_diffs' in elem_data and elem_data['shannon_diffs']:
                        avg_shannon_diff = np.mean([s['percent_diff'] for s in elem_data['shannon_diffs']])
                        site_result['shannon_diff'] = round(avg_shannon_diff, 2)
                        site_result['avg_bond_length'] = round(np.mean([s['observed_distance'] for s in elem_data['shannon_diffs']]), 4)

                    # ========================================================================
                    # ADD STRUCTURE-SPECIFIC RESULTS TO THIS SITE
                    # ========================================================================
                    if structure_specific_results:
                        site_result.update(structure_specific_results)
                    
                    # ========================================================================
                    # 3D visualisation for problematic sites
                    # ========================================================================
                    if deviation is not None and abs(deviation) > 0.30 and PY3DMOL_AVAILABLE and bond_counts:
                        print(f"\n    ⚠️  Significant BVS deviation detected for {element} site!")
                        print(f"        BVS: {bvs_val:.3f}  │  Expected: {valence:+g}  │  Δ: {deviation:+.3f} v.u.")
                        
                        # NEW: Print diagnostic information
                        if d1 is not None:
                            print(f"        d1: {d1:+.3f} v.u. ({bonding_state})")
                        print(f"        Disorder: {mixed_occ_data['disorder_type']}", end="")
                        if mixed_occ_data['valence_difference'] > 0:
                            print(f" (ΔV={mixed_occ_data['valence_difference']:.0f})")
                        else:
                            print()
                        print(f"        Occupancy: {mixed_occ_data['occupancy_string']}")
                        print(f"        Confidence: {confidence}")
                        if bosi_likely:
                            print(f"        ⚠️  Bosi artifact likely - spectroscopy required!")
                        
                        print(f"    [Generating 3D visualisation...]")
                        
                        try:
                            visualize_problematic_site_3d(
                                cif_path=file_path,
                                element=element,
                                site_label=site_label,
                                bonds_info=bond_counts,
                                deviation=deviation,
                                formal_charge=valence,
                                bvs_value=bvs_val,
                                filename=os.path.basename(file_path)
                            )
                        except Exception as viz_error:
                            print(f"    [3D visualisation failed: {viz_error}]")
                    
                    elif deviation is not None and abs(deviation) > 0.30 and not PY3DMOL_AVAILABLE:
                        print(f"\n    ⚠️  Significant deviation ({deviation:+.3f} v.u.)")
                        if d1 is not None:
                            print(f"        d1: {d1:+.3f} v.u. ({bonding_state})")
                        print(f"        Disorder: {mixed_occ_data['disorder_type']}")
                        print(f"        Confidence: {confidence}")
                        print(f"        Install py3Dmol for 3D visualisation")
                    
                    results['sites'].append(site_result)
        
        # Add GII (unchanged)
            if block_name in gii_results:
                results['gii'] = gii_results[block_name].get('gii')
                results['wagii'] = gii_results[block_name].get('wagii')
                results['stability_assessment'] = gii_results[block_name].get('stability_assessment')
        
        results['success'] = True
        
    except Exception as e:
        results['error'] = f"Processing error: {str(e)}"
        import traceback
        print(f"  Detailed error: {traceback.format_exc()}")
    
    return results

# ============================================================================
# INTERACTIVE OXIDATION STATE ASSIGNMENT
# ============================================================================

def collect_unique_elements_from_cifs(directory_path):
    """
    Pre-scan all CIF files to identify unique elements present.
    This allows asking for oxidation states once before batch processing.
    
    Args:
        directory_path: Path to directory containing CIF files
    
    Returns:
        set of element symbols found across all CIFs
    """
    cif_files = list(Path(directory_path).glob("*.cif"))
    unique_elements = set()
    
    print("\n" + "="*80)
    print("PRE-SCANNING CIF FILES TO IDENTIFY ELEMENTS")
    print("="*80)
    
    for cif_file in cif_files:
        try:
            cf = ReadCif(str(cif_file))
            
            for block_name, data_block in cf.items():
                # Get element types from atom site data
                types = data_block.get('_atom_site_type_symbol', [])
                
                for atom_type_raw in types:
                    # Clean up: "Na1" → "Na", "O2-" → "O"
                    atom_type = atom_type_raw.strip().rstrip('+-0123456789')
                    atom_type = atom_type.capitalize() if len(atom_type) > 1 else atom_type.upper()
                    unique_elements.add(atom_type)
        
        except Exception as e:
            continue  # Skip problematic files in pre-scan
    
    print(f"✓ Found {len(unique_elements)} unique elements: {sorted(unique_elements)}")
    return unique_elements


def prompt_for_oxidation_states(elements):
    """
    Interactive prompt for oxidation state assignment.
    Shows smart defaults based on oxide chemistry, allows user override.
    
    Args:
        elements: set of element symbols
    
    Returns:
        dict mapping element → oxidation state {e.g., 'Ti': 4, 'Na': 1}
    """
    # Smart defaults for oxide battery materials AND halides/sulfides
    OXIDE_DEFAULTS = {
        # Alkali metals
        'Li': 1, 'Na': 1, 'K': 1, 'Rb': 1, 'Cs': 1,
        # Alkaline earth
        'Mg': 2, 'Ca': 2, 'Sr': 2, 'Ba': 2,
        # Transition metals (oxide battery cathodes)
        'Ti': 4,   # Ti⁴⁺ most common in oxides
        'V': 5,    # V⁵⁺ in V₂O₅
        'Cr': 3,   # Cr³⁺ in chromites
        'Mn': 4,   # Mn⁴⁺ in MnO₂, battery cathodes (can also be 2, 3)
        'Fe': 3,   # Fe³⁺ slightly more common (can also be 2)
        'Co': 3,   # Co³⁺ in LiCoO₂
        'Ni': 2,   # Ni²⁺ in most cathodes
        'Cu': 2,   # Cu²⁺ most common
        'Zn': 2,   # Zn²⁺ only
        'Cd': 2,   # Cd²⁺ only
        # Other common
        'Al': 3, 'Sb': 5, 'Sn': 4, 'Zr': 4, 'Nb': 5, 'Mo': 6, 'W': 6,
        # Anions (oxides, halides, chalcogenides)
        'O': -2, 'F': -1, 'Cl': -1, 'Br': -1, 'I': -1,
        'S': -2, 'Se': -2, 'Te': -2,
        'N': -3, 'P': -3, 'As': -3, 'H': -1,
    }
    
    print("\n" + "="*80)
    print("OXIDATION STATE ASSIGNMENT")
    print("="*80)
    print("Please confirm oxidation states for elements found in your CIF files.")
    print("Defaults are shown in [brackets] - press ENTER to accept, or type new value.")
    print("="*80 + "\n")
    
    oxidation_states = {}
    
    # Sort: anions last, then alphabetical
    anions = sorted([e for e in elements if e in ['O', 'F', 'S', 'Cl', 'Br', 'I']])
    cations = sorted([e for e in elements if e not in anions])
    sorted_elements = cations + anions
    
    for element in sorted_elements:
        # Get default suggestion
        default = OXIDE_DEFAULTS.get(element)
        
        if default is not None:
            prompt_text = f"  {element:>2s} oxidation state [{default:+d}]: "
        else:
            # Check database
            common_states = db_manager.get_common_oxidation_states(element)
            if common_states:
                # For elements with multiple common states, show options
                states_str = ', '.join([f"{s:+d}" for s in common_states])
                default = common_states[0]  # Suggest first
                prompt_text = f"  {element:>2s} oxidation state (common: {states_str}) [{default:+d}]: "
            else:
                prompt_text = f"  {element:>2s} oxidation state [?]: "
                default = None
        
        while True:
            try:
                user_input = input(prompt_text).strip()
                
                if user_input == '' and default is not None:
                    # Accept default
                    oxidation_states[element] = default
                    break
                elif user_input != '':
                    # Parse user input
                    valence = int(user_input)
                    oxidation_states[element] = valence
                    break
                else:
                    # No default and no input
                    print("    ERROR: Please enter an oxidation state")
            
            except ValueError:
                print("    ERROR: Please enter a valid integer (e.g., 4, -2)")
    
    # Show summary
    print("\n" + "="*80)
    print("OXIDATION STATES CONFIRMED:")
    print("="*80)
    for element in sorted_elements:
        valence = oxidation_states[element]
        print(f"  {element:>2s}: {valence:+g}")
    print("="*80 + "\n")
    
    # Final confirmation
    confirm = input("Proceed with these oxidation states? [Y/n]: ").strip().lower()
    if confirm and confirm not in ['y', 'yes', '']:
        print("\nAborted by user")
        return None
    
    return oxidation_states

def batch_process_directory(directory_path):
    """
    Process all CIF files in a directory using EXACT Skuld 1.5 logic.
    Includes interactive oxidation state checkpoint for user control.
    Automatically creates /outputs subfolder for results.
    
    NEW IN v1.4: Attempts CIF oxidation state extraction before interactive prompt
    
    Args:
        directory_path: Path to directory containing CIF files
    
    Returns:
        pandas DataFrame with results, or None if failed
    """
    # Create outputs subdirectory
    output_dir = os.path.join(directory_path, "outputs")
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n✓ Output directory: {output_dir}")
    
    cif_files = list(Path(directory_path).glob("*.cif"))
    
    if not cif_files:
        print(f"No CIF files found in {directory_path}")
        return None
    
    # ============================================================================
    # STEP 1: Config file provides OVERRIDES (highest priority)
    # ============================================================================
    config_path = auto_detect_config_file(directory_path)
    override_oxidation_states = None
    
    if config_path:
        override_oxidation_states = load_oxidation_states_from_config(config_path=config_path)
        print(f"✓ Config file loaded - these will override CIF values")
    
    # ============================================================================
    # STEP 2: Scan all CIFs to see what we're dealing with
    # ============================================================================
    print(f"\nScanning {len(cif_files)} CIF files for oxidation states...")
    
    cif_ox_summary = {'explicit': 0, 'inferred': 0, 'ambiguous': 0, 'missing': 0, 'per_atom_explicit': 0}
    elements_needing_fallback = set()
    ambiguous_files = []
    missing_files = []
    per_atom_explicit_files = []  # NEW: Track well-resolved mixed-valence files
    
    for cif_file in cif_files:
        try:
            cf = ReadCif(str(cif_file))
            block_name = list(cf.keys())[0]
            formula = cf[block_name].get("_chemical_formula_sum", "")
            
            # First check for per-atom explicit valences
            per_atom_valences = extract_per_atom_valences(str(cif_file))
            has_per_atom_explicit = any(
                v['explicit_valence'] is not None 
                for v in per_atom_valences.values()
            )
            
            # Check for mixed valence at element level
            element_valences = {}
            for label, data in per_atom_valences.items():
                elem = data['element']
                val = data['explicit_valence']
                if val is not None:
                    if elem not in element_valences:
                        element_valences[elem] = set()
                    element_valences[elem].add(val)
            
            has_mixed_valence = any(len(vals) > 1 for vals in element_valences.values())
            
            # Classify the file
            if has_per_atom_explicit and has_mixed_valence:
                # Mixed valence WITH per-atom resolution - this is GOOD data!
                quality = 'per_atom_explicit'
                cif_ox_summary['per_atom_explicit'] += 1
                per_atom_explicit_files.append(cif_file.name)
                # NO fallback needed - we have explicit valences per atom
            
            elif has_per_atom_explicit:
                # Single valence per element with explicit data
                quality = 'explicit'
                cif_ox_summary['explicit'] += 1
            
            else:
                # Fall back to original flag_oxidation_state_issues logic
                result = flag_oxidation_state_issues(str(cif_file), formula)
                quality = result.get('quality', 'missing')
                cif_ox_summary[quality] = cif_ox_summary.get(quality, 0) + 1
                
                if quality == 'ambiguous':
                    ambiguous_files.append(cif_file.name)
                    parsed = parse_chemical_formula(formula)
                    for elem in parsed.keys():
                        if not is_anion(elem):
                            elements_needing_fallback.add(elem)
                elif quality == 'missing':
                    missing_files.append(cif_file.name)
                    parsed = parse_chemical_formula(formula)
                    for elem in parsed.keys():
                        if not is_anion(elem):
                            elements_needing_fallback.add(elem)
        except Exception as e:
            cif_ox_summary['missing'] += 1
            missing_files.append(cif_file.name)
    
    # Print summary
    print(f"  Explicit oxidation states:    {cif_ox_summary['explicit']} files")
    print(f"  Per-atom explicit (mixed):    {cif_ox_summary['per_atom_explicit']} files")  # NEW
    print(f"  Inferred oxidation states:    {cif_ox_summary['inferred']} files")
    print(f"  Ambiguous (needs fallback):   {cif_ox_summary['ambiguous']} files")
    print(f"  Missing oxidation states:     {cif_ox_summary['missing']} files")
    
    # NEW: Report per-atom explicit files (these are GOOD, not problematic)
    if per_atom_explicit_files:
        print(f"\n  ✓ Per-atom explicit files (mixed valence resolved):")
        for fn in per_atom_explicit_files[:10]:
            print(f"    - {fn}")
        if len(per_atom_explicit_files) > 10:
            print(f"    ... and {len(per_atom_explicit_files) - 10} more")
    
    # List problematic files (only truly ambiguous/missing)
    if ambiguous_files:
        print(f"\n  ⚠ Ambiguous files (will use fallback):")
        for fn in ambiguous_files[:20]:
            print(f"    - {fn}")
        if len(ambiguous_files) > 20:
            print(f"    ... and {len(ambiguous_files) - 20} more")
    
    if missing_files:
        print(f"\n  ⚠ Missing oxidation state files (will use fallback):")
        for fn in missing_files[:20]:
            print(f"    - {fn}")
        if len(missing_files) > 20:
            print(f"    ... and {len(missing_files) - 20} more")
    
    # Export review CSV (only for truly problematic files)
    if ambiguous_files or missing_files or per_atom_explicit_files:
        review_path = os.path.join(output_dir, "oxidation_state_review.csv")
        with open(review_path, 'w') as f:
            f.write("filename,status,notes\n")
            for fn in per_atom_explicit_files:
                f.write(f"{fn},per_atom_explicit,Mixed valence resolved - no action needed\n")
            for fn in ambiguous_files:
                f.write(f"{fn},ambiguous,Needs fallback oxidation states\n")
            for fn in missing_files:
                f.write(f"{fn},missing,No oxidation states found\n")
        print(f"\n  Review list exported to: {review_path}")
    
    # ============================================================================
    # STEP 3: Prompt for FALLBACK values only if needed
    # ============================================================================
    fallback_oxidation_states = {}
    
    if elements_needing_fallback and override_oxidation_states is None:
        print(f"\n{len(elements_needing_fallback)} elements need fallback oxidation states:")
        print(f"  {', '.join(sorted(elements_needing_fallback))}")
        print(f"\nThese will be used ONLY when CIF values are ambiguous/missing.")
        
        fallback_oxidation_states = prompt_for_oxidation_states(elements_needing_fallback)
    
    # ============================================================================
    # STEP 4: Process each file with per-file oxidation state resolution
    # ============================================================================
    all_results = []
    failed_files = []
    
    for cif_file in cif_files:
        # Get formula for THIS file
        try:
            cf = ReadCif(str(cif_file))
            block_name = list(cf.keys())[0]
            this_formula = cf[block_name].get("_chemical_formula_sum", "")
        except:
            this_formula = ""
        
        # NEW v1.4.2: Extract per-atom valences for mixed-valence detection
        per_atom_valences = extract_per_atom_valences(str(cif_file))
        
        # Check if this CIF has per-atom explicit valences (mixed valence resolved)
        has_per_atom_explicit = any(
            v['explicit_valence'] is not None 
            for v in per_atom_valences.values()
        )
        
        # Try CIF first (only pass config override, NOT fallback)
        ox_states, source, type7_result = get_oxidation_states_for_analysis(
            str(cif_file), 
            this_formula,
            override_oxidation_states
        )
        
        # NEW v1.4.2: If we have per-atom explicit valences, override the type7 flag
        # Mixed valence with per-atom resolution is NOT an error - it's good data!
        if has_per_atom_explicit and type7_result:
            if type7_result.get('error_code') == 'TYPE7_EXPLICIT_MIXED_VALENCE':
                type7_result['has_type7_error'] = False
                type7_result['error_code'] = None
                type7_result['quality'] = 'per_atom_explicit'
        
        # If CIF failed AND no per-atom valences, use fallback
        if ox_states is None and not has_per_atom_explicit and fallback_oxidation_states:
            ox_states = fallback_oxidation_states
            source = 'fallback'
        
        # Process with resolved oxidation states
        # NOTE: process_single_cif extracts per_atom_valences internally for BVS calc
        result = process_single_cif(str(cif_file), ox_states)
        
        if result['success']:
            gii_str = f"{result['gii']:.4f}" if result['gii'] is not None else "N/A"
            source_str = f"[{source}]" if source else "[no ox]"
            print(f"✓ {source_str} ({len(result['sites'])} sites, GII={gii_str})")
            
            # ================================================================
            # NEW: Flag all error types (2-6) for this structure
            # ================================================================
            # Get spinel-specific results if available
            spinel_results = {}
            if result['sites']:
                # Spinel columns are in the site data
                first_site = result['sites'][0]
                spinel_results = {
                    'spinel_classification': first_site.get('spinel_classification'),
                    'heterovalent_disorder': first_site.get('heterovalent_disorder'),
                    'inversion_parameter': first_site.get('inversion_parameter'),
                }
            
            error_flags = flag_all_error_types(
                str(cif_file), 
                this_formula, 
                result['sites'],
                spinel_results
            )
            
            # Add each site to results
            # NEW v1.7: Compute structure-level integrity tier
            site_tiers = [site.get('integrity_tier', 'ACCEPT') for site in result['sites']]
            structure_tier = assign_structure_tier(site_tiers)
            n_flagged = sum(1 for t in site_tiers if t == 'FLAG')
            n_expunged = sum(1 for t in site_tiers if t == 'EXPUNGE')
            
            for site in result['sites']:
                row = {
                    **result['metadata'], 
                    **site,
                    'gii': round(result['gii'], 4) if result['gii'] is not None else None,
                    'wagii': round(result['wagii'], 4) if result.get('wagii') is not None else None,
                    # Type 7 flags (oxidation state issues)
                    'flag_type7': type7_result.get('has_type7_error') if type7_result else None,
                    'type7_error_code': type7_result.get('error_code') if type7_result else None,
                    'oxidation_quality': type7_result.get('quality') if type7_result else None,
                    # NEW v1.4.2: Track where valence came from
                    'valence_source': site.get('valence_source', 'unknown'),
                    # NEW v1.4.2: Types 2-6 flags
                    'flag_type2': error_flags.get('flag_type2'),
                    'flag_type3': error_flags.get('flag_type3'),
                    'flag_type4': error_flags.get('flag_type4'),
                    'flag_type5': error_flags.get('flag_type5'),
                    'flag_type6': error_flags.get('flag_type6'),
                    'flag_type8': error_flags.get('flag_type8'),
                    'error_types_detected': error_flags.get('error_types_detected'),
                    'error_type_details': error_flags.get('error_type_details'),
                    # NEW v1.7: Structure-level integrity tier
                    'structure_tier': structure_tier,
                    'n_sites_flagged': n_flagged,
                    'n_sites_expunged': n_expunged,
                }
                all_results.append(row)
        
        else:
            print(f"✗ ({result['error']})")
            failed_files.append({
                'filename': cif_file.name,
                'error': result['error']
            })
    
    # ============================================================================
    # STEP 5: Save results to CSV files
    # ============================================================================
    if all_results:
        df = pd.DataFrame(all_results)
        
        # Add structure-specific columns
        if 'structure_registry' in globals():
            all_possible_columns = structure_registry.get_all_columns()
            for col in all_possible_columns:
                if col not in df.columns:
                    df[col] = None
        
        # NEW IN v1.4: Ensure Type 7 columns exist
        type7_columns = ['flag_type7', 'type7_error_code', 'oxidation_quality']
        for col in type7_columns:
            if col not in df.columns:
                df[col] = None

        # ============================================================================
        # DATASET-LEVEL PASS: Flag multiple entries from same source paper (v1.7.2)
        # ============================================================================
        if 'icsd_code' in df.columns and 'authors' in df.columns:
            df = flag_same_source_entries(df)
            
            # Print summary of multi-entry sources
            multi = df[df['flag_multi_entry_source'] == True]
            if len(multi) > 0:
                multi_groups = multi.groupby('source_group').first().reset_index()
                print(f"\n⚠ Dataset-level check: {len(multi_groups)} source papers have multiple ICSD entries:")
                for _, grp in multi_groups.iterrows():
                    codes = grp['source_group_codes']
                    n = grp['source_group_size']
                    # Get one example formula per code
                    code_list = codes.split(',')
                    examples = []
                    for code in code_list[:4]:
                        formula = df[df['icsd_code'] == code]['chemical_formula'].iloc[0] if len(df[df['icsd_code'] == code]) > 0 else '?'
                        examples.append(f"{code}({formula})")
                    suffix = f" +{len(code_list)-4} more" if len(code_list) > 4 else ""
                    print(f"  {grp['source_group']}: {n} entries — {', '.join(examples)}{suffix}")
                print(f"  Note: multiple entries from one paper may represent different samples,")
                print(f"  refinement methods, or phase-separated components. Check the original")
                print(f"  paper to assess whether entries are independent determinations.")
        
        # Save results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_file = os.path.join(output_dir, f"eir_batch_results_v1.7_{timestamp}.csv")
        df.to_csv(results_file, index=False)
        print(f"\n✓ Results saved to: {results_file}")
        print(f"  Total sites analysed: {len(df)}")
        print(f"  Total columns: {len(df.columns)}")
        
        # ============================================================================
        # PRINT STRUCTURE-TYPE ANALYSIS SUMMARY (Eir v1.4+)
        # ============================================================================
        if 'spinel_classification' in df.columns:
            spinel_count = df['spinel_classification'].notna().sum()
            if spinel_count > 0:
                spinel_structures = df[df['spinel_classification'].notna()]['filename'].nunique()
                print(f"\n  Spinel structures detected: {spinel_structures} files, {spinel_count} sites")
                print(f"    Classifications:")
                for classification in df['spinel_classification'].dropna().unique():
                    count = (df['spinel_classification'] == classification).sum()
                    print(f"      - {classification}: {count} sites")
                
                if 'inversion_parameter' in df.columns:
                    lambda_values = df['inversion_parameter'].dropna()
                    if len(lambda_values) > 0:
                        print(f"    Inversion parameters:")
                        print(f"      - λ = 0 (normal): {(lambda_values == 0.0).sum()} sites")
                        print(f"      - λ = 1 (inverse): {(lambda_values == 1.0).sum()} sites")
                        print(f"      - 0 < λ < 1 (random): {((lambda_values > 0.0) & (lambda_values < 1.0)).sum()} sites")
        
        # Build merged oxidation states dict for logging
        # Priority: override (config) > fallback (interactive prompt) > empty
        effective_oxidation_states = {}
        if fallback_oxidation_states:
            effective_oxidation_states.update(fallback_oxidation_states)
        if override_oxidation_states:
            effective_oxidation_states.update(override_oxidation_states)
        
        # Save oxidation states used for this batch
        oxidation_states_file = os.path.join(output_dir, f"oxidation_states_used_{timestamp}.json")
        with open(oxidation_states_file, 'w') as f:
            import json
            json.dump(effective_oxidation_states, f, indent=2, sort_keys=True)
        print(f"\n✓ Oxidation states saved to: {oxidation_states_file}")
        
        # Save failed files log if any
        if failed_files:
            failed_df = pd.DataFrame(failed_files)
            failed_file = os.path.join(output_dir, f"eir_batch_failed_v1.7_{timestamp}.csv")
            failed_df.to_csv(failed_file, index=False)
            print(f"✓ Failed files log saved to: {failed_file}")
            print(f"  ({len(failed_files)} files failed)")
        
        # Print summary statistics
        print(f"\n{'='*80}")
        print(f"EIR BATCH PROCESSOR v1.7 - Summary Statistics")
        print(f"{'='*80}")
        
        # Show oxidation states used
        print(f"\nOxidation states used:")
        for element in sorted(effective_oxidation_states.keys()):
            valence = effective_oxidation_states[element]
            print(f"  {element:>2s}: {valence:+g}")
        
        # Show GII statistics if available
        if 'gii' in df.columns and df['gii'].notna().any():
            print(f"\nGlobal Instability Index (GII) statistics:")
            gii_by_structure = df.groupby('filename')['gii'].first()
            print(f"  Mean GII: {gii_by_structure.mean():.4f}")
            print(f"  Min GII:  {gii_by_structure.min():.4f} ({gii_by_structure.idxmin()})")
            print(f"  Max GII:  {gii_by_structure.max():.4f} ({gii_by_structure.idxmax()})")
            
            stable_structures = (gii_by_structure < 0.1).sum()
            unstable_structures = (gii_by_structure >= 0.2).sum()
            print(f"  Stable (GII < 0.1): {stable_structures} structures")
            print(f"  Unstable (GII ≥ 0.2): {unstable_structures} structures")
        
        # NEW v1.7: Integrity tier summary
        if 'structure_tier' in df.columns:
            print(f"\nData Integrity Tier Summary (v1.7):")
            structures_by_tier = df.groupby('filename')['structure_tier'].first()
            for tier in ['ACCEPT', 'FLAG', 'EXPUNGE']:
                count = (structures_by_tier == tier).sum()
                print(f"  {tier}: {count} structures")
            
            # FLAG severity
            if 'flag_severity' in df.columns:
                flag_df = df[df['integrity_tier'] == 'FLAG']
                if len(flag_df) > 0:
                    n_con = (flag_df['flag_severity'] == 'consistent').sum()
                    n_inc = (flag_df['flag_severity'] == 'inconsistent').sum()
                    print(f"  FLAG breakdown: {n_con} consistent, {n_inc} inconsistent sites")
            
            # EXPUNGE breakdown
            exp_df = df[df['integrity_tier'] == 'EXPUNGE']
            if len(exp_df) > 0 and 'flag_reason' in df.columns:
                n_i6 = (exp_df['flag_reason'] == 'implausible_bvs').sum()
                n_i3 = (exp_df['flag_reason'] == 'minority_species').sum()
                print(f"  EXPUNGE breakdown: {n_i6} I6 (broken), {n_i3} I3 (minority/WAGII)")
        
        print(f"{'='*80}\n")
        
        return df
    else:
        print("\n✗ No successful analyses")
        if failed_files:
            print(f"All {len(failed_files)} files failed. Check error messages above.")
        return None

# ============================================================================
# CONVENIENCE FUNCTIONS
# ============================================================================

def run_batch_analysis():
    """
    Convenience function for easy use in Jupyter notebooks or scripts.
    Opens directory selection dialog and processes all CIF files.
    
    Usage:
        from Eir_v1_1 import run_batch_analysis
        df = run_batch_analysis()
    
    Returns:
        pandas DataFrame with results, or None if failed/cancelled
    """
    print("\n" + "="*80)
    print("Running Eir v1.7 Batch Analysis...")
    print("="*80)
    
    # Select directory with tkinter dialog
    directory = select_directory()
    
    if directory is None:
        return None
    
    # Process all CIF files
    results_df = batch_process_directory(directory)
    
    if results_df is not None:
        print("\n" + "="*80)
        print("BATCH PROCESSING COMPLETE")
        print("="*80)
        print(f"Successfully analysed: {results_df['filename'].nunique()} structures")
        print(f"Total sites analysed: {len(results_df)}")
        output_dir = os.path.join(directory, "outputs")
        print(f"\nAll results saved to: {output_dir}")
        print("\nNext steps:")
        print("  1. Review the results CSV file")
        print("  2. Check for any failed files in the failed log")
        print("  3. Analyse results with statistical tools or custom scripts")
        
        return results_df
    
    return None

# ============================================================================
# EXECUTE BATCH ANALYSIS (Jupyter notebook mode)
# ============================================================================

# Run the batch analysis and store results in 'df' for Cell 5 to analyse
df = run_batch_analysis()

if df is not None:
    print(f"\n✓ Analysis complete! Results stored in 'df' variable")
    print(f"✓ Run Cell 5 to see detailed analysis and problem site visualisation")
else:
    print(f"\n✗ Analysis cancelled or failed")

In [ ]:
# CELL 5: ENHANCED ANALYSIS WITH MEASUREMENT CONDITIONS
# ============================================================================
# Run this cell AFTER Cell 4 (which creates the 'df' variable)
# v1.2: Analysis includes measurement condition awareness

import pandas as pd
import numpy as np

# Check if df exists
if 'df' not in locals() or df is None:
    print("⚠️ ERROR: No results found!")
    print("Please run Cell 4 first to generate the 'df' DataFrame")
else:
    # ========================================================================
    # PART 0: MEASUREMENT CONDITIONS OVERVIEW
    # ========================================================================
    
    print("="*80)
    print("EIR v1.7 ANALYSIS WITH MEASUREMENT CONDITIONS")
    print("="*80)
    
    if 'measurement_conditions' in df.columns:
        print("\n🌡️  MEASUREMENT CONDITIONS DISTRIBUTION:")
        print("-"*80)
        
        # Get unique structures
        structures = df.groupby('filename')
        
        # Count by condition
        condition_summary = {}
        for filename, struct_data in structures:
            condition = struct_data['measurement_conditions'].iloc[0]
            if condition not in condition_summary:
                condition_summary[condition] = []
            condition_summary[condition].append(filename)
        
        # Print summary
        total_structures = len(df['filename'].unique())
        for condition in sorted(condition_summary.keys()):
            count = len(condition_summary[condition])
            percent = (count / total_structures) * 100
            print(f"  {condition:20s} n={count:3d} ({percent:5.1f}%)")
        
        # Highlight high-pressure structures
        high_p_conditions = [c for c in condition_summary.keys() if 'high_P' in c or 'high_T_high_P' in c]
        if high_p_conditions:
            total_high_p = sum(len(condition_summary[c]) for c in high_p_conditions)
            print(f"\n  ⚠️  High-pressure structures: n={total_high_p}")
            print(f"      These should be analyzed separately from ambient data!")
        
        # Show examples with temperature/pressure
        print("\n📊 SAMPLE STRUCTURES WITH METADATA:")
        print("-"*80)
        
        sample_conditions = ['ambient', 'high_T_high_P', 'high_P', 'unknown']
        for condition in sample_conditions:
            if condition in condition_summary:
                sample_file = condition_summary[condition][0]
                sample_data = df[df['filename'] == sample_file].iloc[0]
                
                temp = sample_data.get('measurement_temperature_K')
                pressure = sample_data.get('measurement_pressure_GPa')
                
                temp_str = f"{temp:.0f} K" if pd.notna(temp) else "not specified"
                pressure_str = f"{pressure:.2f} GPa" if pd.notna(pressure) else "not specified"
                
                print(f"\n  {condition}:")
                print(f"    File: {sample_file}")
                print(f"    Temperature: {temp_str}")
                print(f"    Pressure: {pressure_str}")
    
    else:
        print("\n⚠️  No measurement condition data found")
    
    # ========================================================================
    # PART 1: STATISTICAL OUTLIER DETECTION
    # ========================================================================
    
    print("\n\n" + "="*80)
    print("STATISTICAL OUTLIER ANALYSIS")
    print("="*80)
    
    # Calculate z-scores for key numerical columns
    df_outliers = df.copy()
    
    # For each element, calculate z-scores within that element group
    elements = df['element'].unique()
    
    for element in elements:
        element_mask = df['element'] == element
        element_data = df[element_mask]
        
        if len(element_data) > 1:  # Need at least 2 samples for std
            # BVS z-score
            if 'bvs' in df.columns:
                mean_bvs = element_data['bvs'].mean()
                std_bvs = element_data['bvs'].std()
                if std_bvs > 0:
                    df_outliers.loc[element_mask, 'bvs_zscore'] = (element_data['bvs'] - mean_bvs) / std_bvs
            
            # Deviation z-score
            if 'deviation' in df.columns:
                mean_dev = element_data['deviation'].mean()
                std_dev = element_data['deviation'].std()
                if std_dev > 0:
                    df_outliers.loc[element_mask, 'deviation_zscore'] = (element_data['deviation'] - mean_dev) / std_dev
            
            # Bond length z-score
            if 'avg_bond_length' in df.columns:
                mean_bl = element_data['avg_bond_length'].mean()
                std_bl = element_data['avg_bond_length'].std()
                if std_bl > 0:
                    df_outliers.loc[element_mask, 'bond_length_zscore'] = (element_data['avg_bond_length'] - mean_bl) / std_bl
    
    # Calculate GII z-scores across all structures
    if 'gii' in df.columns:
        mean_gii = df['gii'].mean()
        std_gii = df['gii'].std()
        if std_gii > 0:
            df_outliers['gii_zscore'] = (df['gii'] - mean_gii) / std_gii
    
    # Identify outliers (|z-score| > 2.0 = ~95% confidence)
    outlier_threshold = 2.0
    df_outliers['is_outlier'] = False
    
    for col in ['bvs_zscore', 'deviation_zscore', 'bond_length_zscore']:
        if col in df_outliers.columns:
            df_outliers['is_outlier'] |= (df_outliers[col].abs() > outlier_threshold)
    
    # Report outliers
    outliers = df_outliers[df_outliers['is_outlier'] == True]
    
    if len(outliers) > 0:
        print(f"\nFound {len(outliers)} outlier sites (|z-score| > {outlier_threshold}):")
        print()
        for idx, row in outliers.iterrows():
            print(f"  {row['filename'].replace('YourCustomFileName_', '')}")
            print(f"    Site: {row['element']} (label: {row['site_label']})")
            
            if 'bvs_zscore' in row and not pd.isna(row['bvs_zscore']):
                print(f"    BVS z-score: {row['bvs_zscore']:.2f} (BVS={row['bvs']:.3f}, expected={row['valence']:+.0f})")
            if 'deviation_zscore' in row and not pd.isna(row['deviation_zscore']):
                print(f"    Deviation z-score: {row['deviation_zscore']:.2f} (Δ={row['deviation']:+.3f} v.u.)")
            if 'bond_length_zscore' in row and not pd.isna(row['bond_length_zscore']):
                print(f"    Bond length z-score: {row['bond_length_zscore']:.2f} ({row['avg_bond_length']:.4f} Å)")
            
            # Show measurement conditions if available
            if 'measurement_conditions' in row:
                print(f"    Measurement: {row['measurement_conditions']}")
            
            print()
    else:
        print("\n✓ No statistical outliers detected (all z-scores < 2.0)")
    
    # ========================================================================
    # PART 2: QUALITY VERDICTS
    # ========================================================================
    
    print("\n" + "="*80)
    print("QUALITY ASSESSMENT & VERDICTS")
    print("="*80)
    
    # Get unique structures
    structures = df.groupby('filename')
    
    quality_verdicts = []
    
    for filename, struct_data in structures:
        verdict = {
            'filename': filename,
            'icsd_code': struct_data['block_name'].iloc[0] if 'block_name' in struct_data.columns else None,
            'gii': struct_data['gii'].iloc[0],
            'n_sites': len(struct_data),
            'n_problematic': len(struct_data[struct_data['deviation'].abs() > 0.30]),
            'max_deviation': struct_data['deviation'].abs().max(),
            'conditions': struct_data['measurement_conditions'].iloc[0] if 'measurement_conditions' in struct_data.columns else 'unknown',
            'flags': [],
            'verdict': 'UNKNOWN',
            'quality_tier': 'UNKNOWN'
        }
        
        # BVS Check
        if verdict['max_deviation'] > 0.30:
            verdict['flags'].append('BVS_FAIL')
        elif verdict['max_deviation'] > 0.20:
            verdict['flags'].append('BVS_WARN')
        else:
            verdict['flags'].append('BVS_PASS')
        
        # GII Check
        if verdict['gii'] > 0.40:
            verdict['flags'].append('GII_FAIL')
        elif verdict['gii'] > 0.20:
            verdict['flags'].append('GII_WARN')
        else:
            verdict['flags'].append('GII_PASS')
        
        # Check for statistical outliers
        struct_outliers = df_outliers[df_outliers['filename'] == filename]
        if struct_outliers['is_outlier'].any():
            verdict['flags'].append('STATISTICAL_OUTLIER')
        
        # Check for non-ambient conditions
        if 'high_P' in verdict['conditions'] or 'high_T' in verdict['conditions']:
            verdict['flags'].append('NON_AMBIENT')
        
        # Overall verdict
        if 'NON_AMBIENT' in verdict['flags']:
            verdict['verdict'] = '⚠️ NON-AMBIENT'
            verdict['quality_tier'] = 'Exclude from ambient analysis'
        elif 'BVS_FAIL' in verdict['flags'] or 'GII_FAIL' in verdict['flags']:
            verdict['verdict'] = '⛔ EXCLUDE'
            verdict['quality_tier'] = 'F (Reject)'
        elif 'STATISTICAL_OUTLIER' in verdict['flags']:
            verdict['verdict'] = '⚠️ QUESTIONABLE'
            verdict['quality_tier'] = 'C (Suspicious)'
        elif 'BVS_WARN' in verdict['flags'] or 'GII_WARN' in verdict['flags']:
            verdict['verdict'] = '⚠️ CAUTION'
            verdict['quality_tier'] = 'B (Use with care)'
        else:
            verdict['verdict'] = '✅ INCLUDE'
            verdict['quality_tier'] = 'A (High confidence)'
        
        quality_verdicts.append(verdict)
    
    # Display verdicts
    print()
    for v in sorted(quality_verdicts, key=lambda x: x['gii'], reverse=True):
        print(f"{v['verdict']} {v['filename'].replace('YourCustomFileName_', '')}")
        print(f"  Quality Tier: {v['quality_tier']}")
        print(f"  Conditions: {v['conditions']}")
        print(f"  GII: {v['gii']:.4f}  │  Max |Δ|: {v['max_deviation']:.3f} v.u.")
        print(f"  Problematic sites: {v['n_problematic']}/{v['n_sites']}")
        print(f"  Flags: {', '.join(v['flags'])}")
        print()
    
    # ========================================================================
    # PART 3: COMPARATIVE STATISTICS BY ELEMENT (CONDITION-AWARE)
    # ========================================================================
    
    print("="*80)
    print("COMPARATIVE STATISTICS (By Element, Ambient vs High-P)")
    print("="*80)
    print()
    
    # Separate ambient from non-ambient
    if 'measurement_conditions' in df.columns:
        ambient_df = df[df['measurement_conditions'].isin(['ambient', 'unknown'])]
        high_p_df = df[df['measurement_conditions'].str.contains('high_P', na=False)]
        
        if len(ambient_df) > 0:
            print("AMBIENT STRUCTURES:")
            print("-"*80)
            
            # Group by element
            element_stats = []
            
            for element in sorted(ambient_df['element'].unique()):
                element_data = ambient_df[ambient_df['element'] == element]
                
                # Exclude outliers for "consensus" values
                if element in df_outliers['element'].values:
                    element_outliers = df_outliers[(df_outliers['element'] == element) & (df_outliers['is_outlier'] == True)]
                    element_consensus = element_data[~element_data.index.isin(element_outliers.index)]
                else:
                    element_consensus = element_data
                
                if len(element_consensus) > 0:
                    stats = {
                        'element': element,
                        'valence': element_data['valence'].iloc[0],
                        'n_total': len(element_data),
                        'n_outliers': len(element_data) - len(element_consensus),
                        'bvs_mean': element_consensus['bvs'].mean(),
                        'bvs_std': element_consensus['bvs'].std(),
                        'deviation_mean': element_consensus['deviation'].mean(),
                        'deviation_std': element_consensus['deviation'].std(),
                        'bond_length_mean': element_consensus['avg_bond_length'].mean(),
                        'bond_length_std': element_consensus['avg_bond_length'].std(),
                        'cn_mode': element_consensus['cn'].mode()[0] if len(element_consensus['cn'].mode()) > 0 else None
                    }
                    
                    element_stats.append(stats)
            
            # Display as table
            if element_stats:
                stats_df = pd.DataFrame(element_stats)
                
                print("Element | Valence | n | BVS (consensus) | Deviation | Bond Length (Å) | CN")
                print("-"*80)
                for _, row in stats_df.iterrows():
                    bvs_str = f"{row['bvs_mean']:.3f}±{row['bvs_std']:.3f}" if pd.notna(row['bvs_std']) else f"{row['bvs_mean']:.3f}"
                    dev_str = f"{row['deviation_mean']:+.3f}±{row['deviation_std']:.3f}" if pd.notna(row['deviation_std']) else f"{row['deviation_mean']:+.3f}"
                    bl_str = f"{row['bond_length_mean']:.4f}±{row['bond_length_std']:.4f}" if pd.notna(row['bond_length_std']) else f"{row['bond_length_mean']:.4f}"
                    
                    outlier_note = f" ({row['n_outliers']} outlier)" if row['n_outliers'] > 0 else ""
                    
                    print(f"{row['element']:>7} | {row['valence']:>7.0f} | {row['n_total']:>1}{outlier_note:<12} | {bvs_str:<15} | {dev_str:<9} | {bl_str:<15} | {row['cn_mode']}")
        
        if len(high_p_df) > 0:
            print("\n\nHIGH-PRESSURE STRUCTURES:")
            print("-"*80)
            print("⚠️  These structures should be analyzed separately!")
            print("    BVS parameters are fitted to ambient conditions.\n")
            
            # Group by element
            for element in sorted(high_p_df['element'].unique()):
                element_data = high_p_df[high_p_df['element'] == element]
                
                print(f"\n{element} (n={len(element_data)}):")
                print(f"  Mean deviation: {element_data['deviation'].mean():+.3f} ± {element_data['deviation'].std():.3f} v.u.")
                print(f"  Range: {element_data['deviation'].min():+.3f} to {element_data['deviation'].max():+.3f} v.u.")
                
                # Show pressure range
                if 'measurement_pressure_GPa' in element_data.columns:
                    pressures = element_data['measurement_pressure_GPa'].dropna()
                    if len(pressures) > 0:
                        print(f"  Pressure range: {pressures.min():.1f} - {pressures.max():.1f} GPa")
    
    else:
        # Fall back to original analysis if no condition data
        print("(No measurement condition data - showing all structures combined)")
        
        element_stats = []
        
        for element in sorted(elements):
            element_data = df[df['element'] == element]
            
            # Exclude outliers for "consensus" values
            if element in df_outliers['element'].values:
                element_outliers = df_outliers[(df_outliers['element'] == element) & (df_outliers['is_outlier'] == True)]
                element_consensus = element_data[~element_data.index.isin(element_outliers.index)]
            else:
                element_consensus = element_data
            
            stats = {
                'element': element,
                'valence': element_data['valence'].iloc[0],
                'n_total': len(element_data),
                'n_outliers': len(element_data) - len(element_consensus),
                'bvs_mean': element_consensus['bvs'].mean(),
                'bvs_std': element_consensus['bvs'].std(),
                'deviation_mean': element_consensus['deviation'].mean(),
                'deviation_std': element_consensus['deviation'].std(),
                'bond_length_mean': element_consensus['avg_bond_length'].mean(),
                'bond_length_std': element_consensus['avg_bond_length'].std(),
                'cn_mode': element_consensus['cn'].mode()[0] if len(element_consensus['cn'].mode()) > 0 else None
            }
            
            element_stats.append(stats)
        
        # Display as table
        stats_df = pd.DataFrame(element_stats)
        
        print("Element | Valence | n | BVS (consensus) | Deviation | Bond Length (Å) | CN")
        print("-"*80)
        for _, row in stats_df.iterrows():
            bvs_str = f"{row['bvs_mean']:.3f}±{row['bvs_std']:.3f}" if pd.notna(row['bvs_std']) else f"{row['bvs_mean']:.3f}"
            dev_str = f"{row['deviation_mean']:+.3f}±{row['deviation_std']:.3f}" if pd.notna(row['deviation_std']) else f"{row['deviation_mean']:+.3f}"
            bl_str = f"{row['bond_length_mean']:.4f}±{row['bond_length_std']:.4f}" if pd.notna(row['bond_length_std']) else f"{row['bond_length_mean']:.4f}"
            
            outlier_note = f" ({row['n_outliers']} outlier)" if row['n_outliers'] > 0 else ""
            
            print(f"{row['element']:>7} | {row['valence']:>7.0f} | {row['n_total']:>1}{outlier_note:<12} | {bvs_str:<15} | {dev_str:<9} | {bl_str:<15} | {row['cn_mode']}")
    
    # ========================================================================
    # PART 4: PUBLICATION METADATA SUMMARY
    # ========================================================================
    
    if 'year' in df.columns and df['year'].notna().any():
        print("\n" + "="*80)
        print("PUBLICATION METADATA")
        print("="*80)
        print()
        
        for filename in df['filename'].unique():
            struct = df[df['filename'] == filename].iloc[0]
            
            print(f"{filename.replace('YourCustomFileName_', '')}")
            if pd.notna(struct.get('icsd_code')):
                print(f"  ICSD Code: {struct['icsd_code']}")
            if pd.notna(struct.get('year')):
                print(f"  Year: {struct['year']}")
            if pd.notna(struct.get('journal')):
                print(f"  Journal: {struct['journal']}")
            if pd.notna(struct.get('authors')):
                print(f"  Authors: {struct['authors']}")
            if pd.notna(struct.get('r_factor')):
                print(f"  R-factor: {struct['r_factor']:.4f}")
            if pd.notna(struct.get('density')):
                print(f"  Density: {struct['density']:.2f} g/cm³")
            
            # Show measurement conditions
            if 'measurement_temperature_K' in struct and pd.notna(struct['measurement_temperature_K']):
                print(f"  Temperature: {struct['measurement_temperature_K']:.0f} K")
            if 'measurement_pressure_GPa' in struct and pd.notna(struct['measurement_pressure_GPa']):
                print(f"  Pressure: {struct['measurement_pressure_GPa']:.2f} GPa")
            if 'measurement_conditions' in struct:
                print(f"  Conditions: {struct['measurement_conditions']}")
            
            print()
    
    # ========================================================================
    # PART 5: PROBLEMATIC SITES
    # ========================================================================
    
    print("="*80)
    print("PROBLEMATIC SITES SUMMARY (|Deviation| > 0.30 v.u.)")
    print("="*80)
    
    problematic = df[df['deviation'].abs() > 0.30].sort_values('deviation', key=abs, ascending=False)
    
    if len(problematic) > 0:
        print(f"Found {len(problematic)} sites with significant BVS deviations:")
        for _, row in problematic.iterrows():
            icsd_id = row['block_name'].split('-')[0] if '-' in row['block_name'] else row['block_name']
            print(f"\n  {icsd_id}")
            print(f"    Site: {row['element']} (label: {row['site_label']})")
            print(f"    BVS: {row['bvs']:.3f}  │  Expected: {row['valence']:+.0f}  │  Δ: {row['deviation']:+.3f} v.u.")
            print(f"    CN: {row['cn']}")
            
            # Show measurement conditions
            if 'measurement_conditions' in row:
                print(f"    Conditions: {row['measurement_conditions']}")
            
            # Add z-score if available
            if row['element'] in df_outliers['element'].values:
                row_outlier = df_outliers[df_outliers.index == row.name]
                if not row_outlier.empty and 'deviation_zscore' in row_outlier.columns:
                    zscore = row_outlier['deviation_zscore'].iloc[0]
                    if pd.notna(zscore):
                        print(f"    Statistical: z-score = {zscore:.2f}")
            
            print(f"    [3D visualisation displayed during Cell 4 processing]")
    else:
        print("No sites with |Δ| > 0.30 v.u. found")
    
    print("\n" + "="*80)
    print("EIR v1.7 ENHANCED ANALYSIS COMPLETE")
    print("="*80)
    
    # NEW v1.7: Integrity tier summary
    if 'integrity_tier' in df.columns:
        print("\nDATA INTEGRITY TIER SUMMARY (v1.7):")
        for tier in ['ACCEPT', 'FLAG', 'EXPUNGE']:
            tier_count = len(df[df['integrity_tier'] == tier])
            tier_structures = df[df['integrity_tier'] == tier]['filename'].nunique() if tier_count > 0 else 0
            print(f"  {tier}: {tier_count} sites across {tier_structures} structures")
        
        # FLAG severity breakdown
        if 'flag_severity' in df.columns:
            flag_sites = df[df['integrity_tier'] == 'FLAG']
            if len(flag_sites) > 0:
                n_consistent = (flag_sites['flag_severity'] == 'consistent').sum()
                n_inconsistent = (flag_sites['flag_severity'] == 'inconsistent').sum()
                print(f"\n  FLAG severity breakdown:")
                print(f"    consistent (BVS validates under depositor partitioning): {n_consistent}")
                print(f"    inconsistent (BVS fails under depositor partitioning):  {n_inconsistent}")
        
        # EXPUNGE breakdown (I6 vs I3)
        expunge_sites = df[df['integrity_tier'] == 'EXPUNGE']
        if len(expunge_sites) > 0 and 'flag_reason' in df.columns:
            n_i6 = (expunge_sites['flag_reason'] == 'implausible_bvs').sum()
            n_i3 = (expunge_sites['flag_reason'] == 'minority_species').sum()
            print(f"\n  EXPUNGE breakdown:")
            print(f"    I6 (implausible BVS - exclude entirely): {n_i6}")
            print(f"    I3 (minority species - retain with WAGII weighting): {n_i3}")
        
        # Validation rates by tier (TM cations only)
        tm_mask = df['element'].apply(lambda x: x in TRANSITION_METALS if pd.notna(x) else False)
        print(f"\n  TM validation rates by tier (|dev| <= 10%):")
        for tier in ['ACCEPT', 'FLAG']:
            tier_tm = df[(df['integrity_tier'] == tier) & tm_mask]
            if len(tier_tm) > 0 and 'bvs_percent_diff' in tier_tm.columns:
                validated = tier_tm[tier_tm['bvs_percent_diff'].abs() <= 10]
                rate = len(validated) / len(tier_tm) * 100
                print(f"    {tier}: {rate:.1f}% ({len(validated)}/{len(tier_tm)})")
    
    print("\nKey Outputs:")
    print("  1. Data integrity tiers (ACCEPT/FLAG/EXPUNGE) per site and structure")
    print("  2. FLAG severity: does the depositor's partitioning produce a valid BVS?")
    print("  3. 'What if' analysis: best-matching integer oxidation state for FLAG sites")
    print("  4. Measurement conditions distribution and ambient/high-P separation")
    print("  5. Statistical outlier detection with z-scores")
    print("  6. Consensus statistics by element (outliers excluded)")
    print("  7. Publication metadata with temperature/pressure")
    print("\nRecommended Actions:")
    print("  • I6 EXPUNGE sites: exclude from all statistics (broken coordinates)")
    print("  • I3 EXPUNGE sites: retain with WAGII weighting (minority species)")
    print("  • FLAG consistent: usable with caveat (partitioning unverifiable but BVS validates)")
    print("  • FLAG inconsistent: check 'what if' — does a better integer state exist?")
    print("  • ACCEPT: most reliable BVS validation rates")
    print("  • High-P structures: analyse separately (BVS parameters assume ambient)")
    print("="*80)